# QTrust-PPO: Ablations, Robustness, Scalability, and Confidence-Bound Analysis

This notebook contains the targeted follow-up experiments and consolidated analysis for QTrust-PPO. It loads the completed primary benchmark, verifies its protocol, and evaluates confidence-level and rollback ablations, controlled output noise, VQC depth, and alternative finite-shot confidence bounds.

The executable experiment logic is unchanged from the completed runs. The notebook is distributed without stored execution output; all corresponding result files, figures, and tables are included in this package.


In [ ]:
# Initialize Checkpoint System


from pathlib import Path
from datetime import datetime, timezone
import os
import re
import json
import shutil
import zipfile
import hashlib
import platform

import numpy as np
import pandas as pd



# GLOBAL PATHS


KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORK  = Path("/kaggle/working")

JOURNAL_ROOT = (
    KAGGLE_WORK
    / "QTrust_PPO_Journal"
)

EXP_A_DIR = (
    JOURNAL_ROOT
    / "experiment_A_frozen"
)

EXP_B_DIR = (
    JOURNAL_ROOT
    / "experiment_B_ablations"
)

EXP_C_DIR = (
    JOURNAL_ROOT
    / "experiment_C_noise"
)

EXP_D_DIR = (
    JOURNAL_ROOT
    / "experiment_D_scalability"
)

EXP_E_DIR = (
    JOURNAL_ROOT
    / "experiment_E_confidence"
)

STATS_DIR = (
    JOURNAL_ROOT
    / "statistics"
)

FIGURES_DIR = (
    JOURNAL_ROOT
    / "figures"
)

TABLES_DIR = (
    JOURNAL_ROOT
    / "tables"
)

LOGS_DIR = (
    JOURNAL_ROOT
    / "logs"
)

CHECKPOINT_DIR = (
    KAGGLE_WORK
    / "qtrust_journal_checkpoints"
)


for directory in [
    JOURNAL_ROOT,
    EXP_A_DIR,
    EXP_B_DIR,
    EXP_C_DIR,
    EXP_D_DIR,
    EXP_E_DIR,
    STATS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    LOGS_DIR,
    CHECKPOINT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )






EXPECTED_PROTOCOL_HASH = (
    "3cf4bcb18e1111b3"
)

EXPECTED_RUNS = 120

EXPECTED_SEEDS = [
    11,
    23,
    37,
    53,
    71,
    89,
    107,
    131,
    157,
    181,
]

EXPECTED_ENVIRONMENTS = [
    "CartPole-v1",
    "Acrobot-v1",
    "LunarLander-v3",
]

EXPECTED_METHODS = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]



# CHECK GPU AVAILABILITY


# This only confirms the T4 x2 environment for later cells.


GPU_INFO = []

try:
    import torch

    GPU_COUNT = torch.cuda.device_count()

    for gpu_id in range(GPU_COUNT):
        GPU_INFO.append(
            {
                "id": gpu_id,
                "name": torch.cuda.get_device_name(
                    gpu_id
                ),
            }
        )

except Exception as gpu_error:
    GPU_COUNT = 0
    GPU_INFO = []


print("=" * 80)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 1 / 10")
print("=" * 80)

print(
    "Detected GPUs :",
    GPU_COUNT,
)

for gpu in GPU_INFO:
    print(
        f"  GPU {gpu['id']} : "
        f"{gpu['name']}"
    )

if GPU_COUNT < 2:
    print(
        "\nNOTE: Cell 1 can still run, but later "
        "GPU experiments are designed for T4 x2."
    )



# ROBUST ARTIFACT DISCOVERY

# Supports both:
# A. Kaggle automatically extracted the ZIP
# B. Kaggle exposes the original ZIP file

# It does NOT depend on exact Kaggle directory names.


def valid_results_root(root):
    root = Path(root)

    results_file = (
        root
        / "all_completed_runs.csv"
    )

    protocol_file = (
        root
        / "frozen_protocol.json"
    )

    if not (
        results_file.exists()
        and protocol_file.exists()
    ):
        return None

    try:
        frame = pd.read_csv(
            results_file
        )

        return len(frame)

    except Exception:
        return None


def find_extracted_experiment():
    """
    Search /kaggle/input recursively for an already
    extracted QTrust final-results tree.
    """

    candidates = []

    if not KAGGLE_INPUT.exists():
        return candidates

    for results_file in KAGGLE_INPUT.rglob(
        "all_completed_runs.csv"
    ):

        root = results_file.parent

        count = valid_results_root(
            root
        )

        if count is not None:
            candidates.append(
                (
                    count,
                    root,
                )
            )

    return candidates


def inspect_zip_for_experiment(zip_path):
    """
    Check a ZIP by INTERNAL CONTENTS rather than
    relying on the uploaded filename.
    """

    try:

        if not zipfile.is_zipfile(
            zip_path
        ):
            return None

        with zipfile.ZipFile(
            zip_path,
            "r",
        ) as zf:

            names = zf.namelist()

            results_member = next(
                (
                    name
                    for name in names
                    if name.endswith(
                        "all_completed_runs.csv"
                    )
                ),
                None,
            )

            protocol_member = next(
                (
                    name
                    for name in names
                    if name.endswith(
                        "frozen_protocol.json"
                    )
                ),
                None,
            )

            if (
                results_member is None
                or protocol_member is None
            ):
                return None

            with zf.open(
                results_member
            ) as f:
                frame = pd.read_csv(f)

            return {
                "count":
                    len(frame),

                "results_member":
                    results_member,

                "protocol_member":
                    protocol_member,
            }

    except Exception:
        return None


def find_qtrust_zip():
    candidates = []

    if not KAGGLE_INPUT.exists():
        return candidates

    for path in KAGGLE_INPUT.rglob("*"):

        if not path.is_file():
            continue

        # Usually .zip, but inspect QTrust-named files too.
        name = path.name.lower()

        if (
            path.suffix.lower() != ".zip"
            and "qtrust" not in name
        ):
            continue

        info = inspect_zip_for_experiment(
            path
        )

        if info is not None:
            candidates.append(
                (
                    info["count"],
                    path,
                    info,
                )
            )

    return candidates



# LOAD THE AUTHORITATIVE EXPERIMENT A


extracted_candidates = (
    find_extracted_experiment()
)

complete_extracted = [
    item
    for item in extracted_candidates
    if item[0] == EXPECTED_RUNS
]


SOURCE_MODE = None
SOURCE_PATH = None


if complete_extracted:

    _, SOURCE_RESULT_ROOT = sorted(
        complete_extracted,
        key=lambda item: str(
            item[1]
        ),
    )[0]

    SOURCE_MODE = (
        "Kaggle-mounted extracted dataset"
    )

    SOURCE_PATH = (
        SOURCE_RESULT_ROOT
    )


else:

    zip_candidates = (
        find_qtrust_zip()
    )

    complete_zips = [
        item
        for item in zip_candidates
        if item[0] == EXPECTED_RUNS
    ]

    if not complete_zips:

        print()
        print("=" * 80)
        print("INPUT DIAGNOSTIC")
        print("=" * 80)

        if KAGGLE_INPUT.exists():

            all_paths = list(
                KAGGLE_INPUT.rglob("*")
            )

            for path in all_paths[:150]:
                print(path)

        raise FileNotFoundError(
            "Could not locate the completed "
            "QTrust 120-run artifact in /kaggle/input. "
            "Make sure your private Kaggle dataset is "
            "attached to this notebook."
        )


    _, SOURCE_ZIP, _ = sorted(
        complete_zips,
        key=lambda item: str(
            item[1]
        ),
    )[0]

    SOURCE_MODE = "ZIP archive"

    SOURCE_PATH = SOURCE_ZIP


    
    # Extract original Experiment A only once
    

    RAW_EXTRACT_DIR = (
        EXP_A_DIR
        / "source_artifact"
    )

    if RAW_EXTRACT_DIR.exists():
        shutil.rmtree(
            RAW_EXTRACT_DIR
        )

    RAW_EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        SOURCE_ZIP,
        "r",
    ) as zf:
        zf.extractall(
            RAW_EXTRACT_DIR
        )


    result_matches = [
        path.parent
        for path
        in RAW_EXTRACT_DIR.rglob(
            "all_completed_runs.csv"
        )
        if (
            path.parent
            / "frozen_protocol.json"
        ).exists()
    ]

    if len(result_matches) != 1:
        raise RuntimeError(
            "The uploaded ZIP was extracted, but "
            "a unique QTrust final-results directory "
            "could not be identified."
        )

    SOURCE_RESULT_ROOT = (
        result_matches[0]
    )



# LOAD RESULTS + PROTOCOL


RESULTS_PATH = (
    SOURCE_RESULT_ROOT
    / "all_completed_runs.csv"
)

PROTOCOL_PATH = (
    SOURCE_RESULT_ROOT
    / "frozen_protocol.json"
)


results_A = pd.read_csv(
    RESULTS_PATH
)


with open(
    PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as f:
    protocol_A = json.load(f)



# HARD SCIENTIFIC INTEGRITY CHECKS



# Total completed runs


if len(results_A) != EXPECTED_RUNS:
    raise RuntimeError(
        f"Expected {EXPECTED_RUNS} runs, "
        f"found {len(results_A)}."
    )



# Completion status


if "status" in results_A.columns:

    if not results_A[
        "status"
    ].eq(
        "COMPLETE"
    ).all():

        raise RuntimeError(
            "At least one Experiment-A run "
            "is not COMPLETE."
        )



# Protocol hash


if "protocol_hash" not in results_A.columns:
    raise RuntimeError(
        "protocol_hash column is missing."
    )


protocol_hashes = (
    results_A[
        "protocol_hash"
    ]
    .dropna()
    .astype(str)
    .unique()
)


if len(protocol_hashes) != 1:
    raise RuntimeError(
        f"Expected one frozen protocol hash. "
        f"Found: {protocol_hashes}"
    )


FOUND_PROTOCOL_HASH = (
    protocol_hashes[0]
)


if (
    FOUND_PROTOCOL_HASH
    != EXPECTED_PROTOCOL_HASH
):

    raise RuntimeError(
        "\nAUTHORITATIVE PROTOCOL MISMATCH\n"
        f"Expected : {EXPECTED_PROTOCOL_HASH}\n"
        f"Found    : {FOUND_PROTOCOL_HASH}"
    )



# Exact environments


found_envs = sorted(
    results_A[
        "environment"
    ]
    .unique()
    .tolist()
)


if set(found_envs) != set(
    EXPECTED_ENVIRONMENTS
):
    raise RuntimeError(
        f"Unexpected environments: "
        f"{found_envs}"
    )



# Exact methods


found_methods = sorted(
    results_A[
        "method"
    ]
    .unique()
    .tolist()
)


if set(found_methods) != set(
    EXPECTED_METHODS
):
    raise RuntimeError(
        f"Unexpected methods: "
        f"{found_methods}"
    )



# Exact 10 seeds per environment × method


group_counts = (
    results_A
    .groupby(
        [
            "environment",
            "method",
        ]
    )
    .size()
)


expected_group_index = (
    pd.MultiIndex.from_product(
        [
            EXPECTED_ENVIRONMENTS,
            EXPECTED_METHODS,
        ]
    )
)


checked_counts = (
    group_counts.reindex(
        expected_group_index
    )
)


if (
    checked_counts.isna().any()
    or not checked_counts.eq(
        10
    ).all()
):
    raise RuntimeError(
        "Every environment × method group "
        "must contain exactly 10 runs."
    )



# Exact seed set


for environment in EXPECTED_ENVIRONMENTS:

    for method in EXPECTED_METHODS:

        group = results_A[
            (
                results_A[
                    "environment"
                ]
                == environment
            )
            &
            (
                results_A[
                    "method"
                ]
                == method
            )
        ]

        seeds = sorted(
            group[
                "seed"
            ]
            .astype(int)
            .tolist()
        )

        if seeds != EXPECTED_SEEDS:
            raise RuntimeError(
                f"Seed mismatch for "
                f"{environment} / {method}.\n"
                f"Expected: {EXPECTED_SEEDS}\n"
                f"Found:    {seeds}"
            )





# We preserve the key Experiment-A files separately so later



REFERENCE_DIR = (
    EXP_A_DIR
    / "reference"
)

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    RESULTS_PATH,
    REFERENCE_DIR
    / "all_completed_runs.csv",
)


shutil.copy2(
    PROTOCOL_PATH,
    REFERENCE_DIR
    / "frozen_protocol.json",
)



# Locate and preserve controlled boundary benchmark
# if present.


boundary_candidates = list(
    SOURCE_RESULT_ROOT.parent.rglob(
        "qtrust_boundary_reliability.csv"
    )
)


if boundary_candidates:

    shutil.copy2(
        boundary_candidates[0],
        REFERENCE_DIR
        / "qtrust_boundary_reliability.csv",
    )






manifest = {

    "project":
        "QTrust-PPO Journal Extension",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment_A": {

        "status":
            "FROZEN_COMPLETE",

        "completed_runs":
            EXPECTED_RUNS,

        "protocol_hash":
            FOUND_PROTOCOL_HASH,

        "environments":
            EXPECTED_ENVIRONMENTS,

        "methods":
            EXPECTED_METHODS,

        "seeds":
            EXPECTED_SEEDS,

        "source_mode":
            SOURCE_MODE,

        "source_path":
            str(
                SOURCE_PATH
            ),
    },

    "experiment_B_ablations": {
        "status": "PENDING"
    },

    "experiment_C_noise": {
        "status": "PENDING"
    },

    "experiment_D_scalability": {
        "status": "PENDING"
    },

    "experiment_E_confidence": {
        "status": "PENDING"
    },

    "gpu_environment": {
        "count": GPU_COUNT,
        "devices": GPU_INFO,
    },

    "current_completed_cell": 1,
}


MANIFEST_PATH = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )



# CHECKPOINT FUNCTION

# We will reuse THIS SAME FUNCTION at the end of every cell.

# - fast compression
# - excludes previous checkpoint ZIPs
# - excludes unnecessary Python caches
# - preserves configs, CSVs, JSON, logs, figures, tables,
#   completed experimental outputs, etc.


def save_journal_checkpoint(
    cell_number,
    label,
):
    """
    Create a portable checkpoint after a journal cell.
    """

    cell_number = int(
        cell_number
    )

    safe_label = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(label),
    ).strip("_")


    checkpoint_name = (
        f"QTrust_Journal_Cell"
        f"{cell_number:02d}_"
        f"{safe_label}.zip"
    )


    checkpoint_path = (
        CHECKPOINT_DIR
        / checkpoint_name
    )


    temp_path = (
        CHECKPOINT_DIR
        / (
            checkpoint_name
            + ".tmp"
        )
    )


    if temp_path.exists():
        temp_path.unlink()


    if checkpoint_path.exists():
        checkpoint_path.unlink()


    skip_suffixes = {
        ".pyc",
    }


    skip_parts = {
        "__pycache__",
        ".ipynb_checkpoints",
    }


    with zipfile.ZipFile(
        temp_path,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
        compresslevel=1,
    ) as zf:

        for file in sorted(
            JOURNAL_ROOT.rglob("*")
        ):

            if not file.is_file():
                continue


            if (
                file.suffix.lower()
                in skip_suffixes
            ):
                continue


            if any(
                part in skip_parts
                for part in file.parts
            ):
                continue


            relative = (
                file.relative_to(
                    JOURNAL_ROOT
                )
            )


            zf.write(
                file,
                arcname=str(
                    Path(
                        JOURNAL_ROOT.name
                    )
                    / relative
                ),
            )


    
    # Verify ZIP before publishing it
    

    with zipfile.ZipFile(
        temp_path,
        "r",
    ) as zf:

        bad_file = (
            zf.testzip()
        )


    if bad_file is not None:

        temp_path.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            f"Checkpoint ZIP failed integrity "
            f"test: {bad_file}"
        )


    # Atomic publish
    temp_path.replace(
        checkpoint_path
    )


    
    # Maintain one predictable latest checkpoint
    

    latest_path = (
        KAGGLE_WORK
        / "QTrust_Journal_LATEST.zip"
    )


    shutil.copy2(
        checkpoint_path,
        latest_path,
    )


    size_mb = (
        checkpoint_path.stat().st_size
        / (
            1024
            ** 2
        )
    )


    print()
    print("-" * 80)
    print(
        f"CHECKPOINT SAVED AFTER CELL "
        f"{cell_number}"
    )
    print("-" * 80)

    print(
        "Checkpoint :",
        checkpoint_path,
    )

    print(
        "Latest copy:",
        latest_path,
    )

    print(
        "Size       :",
        f"{size_mb:.2f} MB",
    )

    print("-" * 80)


    return checkpoint_path



# WRITE CELL-1 VERIFICATION REPORT


verification_report = {

    "cell":
        1,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "completed_runs":
        len(
            results_A
        ),

    "environment_method_groups":
        len(
            group_counts
        ),

    "runs_per_group":
        sorted(
            group_counts
            .unique()
            .tolist()
        ),

    "protocol_hash":
        FOUND_PROTOCOL_HASH,

    "seeds":
        EXPECTED_SEEDS,

    "source_mode":
        SOURCE_MODE,

    "gpu_count":
        GPU_COUNT,

    "gpu_info":
        GPU_INFO,
}


with open(
    LOGS_DIR
    / "cell01_verification.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        verification_report,
        f,
        indent=2,
    )






print()
print("=" * 80)
print("EXPERIMENT A VERIFIED")
print("=" * 80)

print(
    "Completed runs :",
    f"{len(results_A)} / "
    f"{EXPECTED_RUNS}"
)

print(
    "Groups         :",
    f"{len(group_counts)} / 12"
)

print(
    "Runs/group     :",
    sorted(
        group_counts
        .unique()
        .tolist()
    )
)

print(
    "Protocol hash  :",
    FOUND_PROTOCOL_HASH
)

print(
    "Source mode    :",
    SOURCE_MODE
)

print(
    "Experiment A   : FROZEN"
)

print()
print(
    "No Experiment-A training will be rerun."
)

print(
    "Experiments B-E will be stored separately."
)

print("=" * 80)






CELL1_CHECKPOINT = (
    save_journal_checkpoint(
        1,
        "Experiment_A_Verified",
    )
)


print()
print(
    "CELL 1 COMPLETE."
)

print(
    "Next: Cell 2 will freeze the journal-extension "
    "experimental matrix and resume/checkpoint rules."
)

In [ ]:
# Freeze Experiments B-E + Run Manifest + Resume Rules


from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import pandas as pd






required_objects = [
    "JOURNAL_ROOT",
    "EXP_A_DIR",
    "EXP_B_DIR",
    "EXP_C_DIR",
    "EXP_D_DIR",
    "EXP_E_DIR",
    "STATS_DIR",
    "FIGURES_DIR",
    "TABLES_DIR",
    "LOGS_DIR",
    "EXPECTED_SEEDS",
    "EXPECTED_PROTOCOL_HASH",
    "results_A",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Cell 1 must be run first. Missing: "
        + ", ".join(missing)
    )


print("=" * 80)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 2 / 10")
print("=" * 80)






# Experiment A:
#   Already complete.
#   120 runs.


# Experiments B-D:

#   Use CartPole-v1 because:

# Experiment A already establishes generality across
#      CartPole, Acrobot and LunarLander.

# CartPole gives a clean setting where the quantum
#      policies actually undergo repeated actor optimization.

# LunarLander's QTrust controller was extremely
#      conservative in Experiment A, while Acrobot quantum
#      methods saturated. Those results remain reported,
#      but they are poor settings for mechanism isolation.

# This avoids duplicating the entire 120-run experiment while
# preserving 10 independent seeds for every NEW RL condition.

# Experiment E:
#   Controlled confidence-bound benchmark.
#   CPU-oriented.




JOURNAL_ENV = "CartPole-v1"

JOURNAL_SEEDS = list(
    EXPECTED_SEEDS
)

if len(JOURNAL_SEEDS) != 10:
    raise RuntimeError(
        "Journal extensions require exactly 10 seeds."
    )






SHARED_RL = {
    "environment":
        JOURNAL_ENV,

    "training_steps":
        2048,

    "rollout_steps":
        32,

    "evaluation_episodes":
        30,

    "gamma":
        0.99,

    "gae_lambda":
        0.95,

    "clip_epsilon":
        0.20,

    "actor_lr":
        3e-4,

    "critic_lr":
        1e-3,

    "entropy_coef":
        0.01,

    "value_coef":
        0.50,

    "max_grad_norm":
        0.50,

    "minibatch_size":
        64,

    "update_epochs":
        1,
}


SHARED_QUANTUM = {
    "qubits":
        4,

    "baseline_layers":
        3,

    "baseline_quantum_parameters":
        24,

    "qtrust_base_shots":
        128,

    "qtrust_schedule":
        [
            32,
            64,
            128,
            256,
            512,
            1024,
            2048,
            4096,
        ],

    "qtrust_delta":
        0.05,

    "confidence_method":
        "exact_one_sided_clopper_pearson",

    "tails_per_stage":
        4,
}



# EXPERIMENT B — QTRUST ABLATIONS


# Full QTrust is ALREADY present in Experiment A:

#     delta = 0.05
#     rollback = True
#     base shots = 128

# Therefore it is NOT rerun.

# We add only three scientifically useful ablations:

# B1: no rollback
# B2: stricter delta = 0.01
# B3: looser delta = 0.10

# 3 variants × 10 seeds = 30 NEW GPU runs.


EXPERIMENT_B = {
    "name":
        "QTrust ablation study",

    "environment":
        JOURNAL_ENV,

    "reference_from_experiment_A":
        {
            "method":
                "QTrust-PPO-128",

            "delta":
                0.05,

            "rollback":
                True,

            "base_shots":
                128,
        },

    "new_variants": [

        {
            "variant_id":
                "B_NoRollback",

            "method":
                "QTrust-PPO-128",

            "delta":
                0.05,

            "rollback":
                False,

            "base_shots":
                128,
        },

        {
            "variant_id":
                "B_Delta_001",

            "method":
                "QTrust-PPO-128",

            "delta":
                0.01,

            "rollback":
                True,

            "base_shots":
                128,
        },

        {
            "variant_id":
                "B_Delta_010",

            "method":
                "QTrust-PPO-128",

            "delta":
                0.10,

            "rollback":
                True,

            "base_shots":
                128,
        },
    ],

    "seeds":
        JOURNAL_SEEDS,
}



# EXPERIMENT C — QUANTUM NOISE ROBUSTNESS


# Clean/noiseless performance already exists in Experiment A.

# We add TWO distinct noise regimes:

# C1: readout noise
# C2: depolarizing gate noise

# For each noise regime:

#   Fixed-QPPO-128
#   Fixed-QPPO-256
#   QTrust-PPO-128

# 2 noise settings × 3 methods × 10 seeds
# = 60 NEW GPU runs.

# No combined-noise grid is added because that would multiply
# cost without answering a separate core research question.


EXPERIMENT_C = {
    "name":
        "Quantum noise robustness",

    "environment":
        JOURNAL_ENV,

    "clean_reference":
        "Experiment A",

    "methods": [
        "Fixed-QPPO-128",
        "Fixed-QPPO-256",
        "QTrust-PPO-128",
    ],

    "noise_conditions": [

        {
            "noise_id":
                "C_Readout_002",

            "noise_type":
                "readout",

            "readout_error":
                0.02,

            "depolarizing_probability":
                0.0,
        },

        {
            "noise_id":
                "C_Depolarizing_0005",

            "noise_type":
                "depolarizing",

            "readout_error":
                0.0,

            "depolarizing_probability":
                0.005,
        },
    ],

    "seeds":
        JOURNAL_SEEDS,
}



# EXPERIMENT D — CIRCUIT SCALABILITY


# Main Experiment A uses:

#     4 qubits
#     3 variational layers

# We keep 4 qubits fixed because CartPole has four state
# variables. This avoids introducing artificial dimension
# reduction or padded observations.

# We vary DEPTH instead:

#     1 layer
#     3 layers  <-- already Experiment A
#     4 layers

# Only QTrust is needed here because this experiment measures
# scalability of the proposed method itself.

# 2 new depths × 10 seeds = 20 NEW GPU runs.


EXPERIMENT_D = {
    "name":
        "VQC depth scalability",

    "environment":
        JOURNAL_ENV,

    "method":
        "QTrust-PPO-128",

    "qubits":
        4,

    "reference_from_experiment_A": {
        "layers":
            3,

        "quantum_parameters":
            24,
    },

    "new_depths": [

        {
            "variant_id":
                "D_Layers_1",

            "layers":
                1,
        },

        {
            "variant_id":
                "D_Layers_4",

            "layers":
                4,
        },
    ],

    "seeds":
        JOURNAL_SEEDS,
}



# EXPERIMENT E — CONFIDENCE-BOUND BENCHMARK


# This is primarily CPU work.

# Compare the controller's original exact method against two
# useful alternatives under the SAME finite-shot schedule and
# SAME error budget.

# We will report:

#   - resolved accuracy
#   - resolved coverage
#   - unresolved fraction
#   - mean shots
#   - error rate among all trials

# Clopper-Pearson remains the proposed method.


EXPERIMENT_E = {
    "name":
        "Confidence-bound comparison",

    "benchmark_type":
        "controlled_near_PPO_boundary",

    "confidence_methods": [

        {
            "method_id":
                "E_ClopperPearson",

            "method":
                "Clopper-Pearson",

            "role":
                "proposed_exact_method",
        },

        {
            "method_id":
                "E_Wilson",

            "method":
                "Wilson",

            "role":
                "comparison",
        },

        {
            "method_id":
                "E_Hoeffding",

            "method":
                "Hoeffding",

            "role":
                "comparison",
        },
    ],

    "delta":
        0.05,

    "tails_per_stage":
        4,

    "shot_schedule":
        [
            32,
            64,
            128,
            256,
            512,
            1024,
            2048,
            4096,
        ],

    "ppo_boundaries": [
        0.8,
        1.2,
    ],

    "random_seed":
        20260920,

    "implementation_note":
        (
            "All methods will use the same underlying "
            "controlled cases and repeated-look allocation "
            "for a fair comparison."
        ),
}



# CREATE EXACT GPU RUN MANIFEST


run_rows = []



# Experiment B


for variant in EXPERIMENT_B[
    "new_variants"
]:

    for seed in JOURNAL_SEEDS:

        run_rows.append(
            {
                "experiment":
                    "B",

                "run_id":
                    (
                        f"B__{variant['variant_id']}"
                        f"__seed{seed}"
                    ),

                "environment":
                    JOURNAL_ENV,

                "method":
                    variant["method"],

                "variant":
                    variant["variant_id"],

                "seed":
                    seed,

                "gpu_required":
                    True,

                "status":
                    "PENDING",

                "delta":
                    variant["delta"],

                "rollback":
                    variant["rollback"],

                "base_shots":
                    variant["base_shots"],

                "noise_type":
                    "none",

                "readout_error":
                    0.0,

                "depolarizing_probability":
                    0.0,

                "qubits":
                    4,

                "layers":
                    3,
            }
        )



# Experiment C


for noise in EXPERIMENT_C[
    "noise_conditions"
]:

    for method in EXPERIMENT_C[
        "methods"
    ]:

        for seed in JOURNAL_SEEDS:

            run_rows.append(
                {
                    "experiment":
                        "C",

                    "run_id":
                        (
                            f"C__{noise['noise_id']}"
                            f"__{method}"
                            f"__seed{seed}"
                        ),

                    "environment":
                        JOURNAL_ENV,

                    "method":
                        method,

                    "variant":
                        noise["noise_id"],

                    "seed":
                        seed,

                    "gpu_required":
                        True,

                    "status":
                        "PENDING",

                    "delta":
                        (
                            0.05
                            if method
                            == "QTrust-PPO-128"
                            else None
                        ),

                    "rollback":
                        (
                            True
                            if method
                            == "QTrust-PPO-128"
                            else None
                        ),

                    "base_shots":
                        (
                            128
                            if method
                            == "QTrust-PPO-128"
                            else None
                        ),

                    "noise_type":
                        noise[
                            "noise_type"
                        ],

                    "readout_error":
                        noise[
                            "readout_error"
                        ],

                    "depolarizing_probability":
                        noise[
                            "depolarizing_probability"
                        ],

                    "qubits":
                        4,

                    "layers":
                        3,
                }
            )



# Experiment D


for variant in EXPERIMENT_D[
    "new_depths"
]:

    for seed in JOURNAL_SEEDS:

        run_rows.append(
            {
                "experiment":
                    "D",

                "run_id":
                    (
                        f"D__{variant['variant_id']}"
                        f"__seed{seed}"
                    ),

                "environment":
                    JOURNAL_ENV,

                "method":
                    "QTrust-PPO-128",

                "variant":
                    variant[
                        "variant_id"
                    ],

                "seed":
                    seed,

                "gpu_required":
                    True,

                "status":
                    "PENDING",

                "delta":
                    0.05,

                "rollback":
                    True,

                "base_shots":
                    128,

                "noise_type":
                    "none",

                "readout_error":
                    0.0,

                "depolarizing_probability":
                    0.0,

                "qubits":
                    4,

                "layers":
                    variant[
                        "layers"
                    ],
            }
        )


RUN_MANIFEST = pd.DataFrame(
    run_rows
)



# HARD MANIFEST CHECKS


expected_B = (
    3
    * 10
)

expected_C = (
    2
    * 3
    * 10
)

expected_D = (
    2
    * 10
)

expected_gpu_runs = (
    expected_B
    + expected_C
    + expected_D
)


actual_B = int(
    (
        RUN_MANIFEST[
            "experiment"
        ]
        == "B"
    ).sum()
)

actual_C = int(
    (
        RUN_MANIFEST[
            "experiment"
        ]
        == "C"
    ).sum()
)

actual_D = int(
    (
        RUN_MANIFEST[
            "experiment"
        ]
        == "D"
    ).sum()
)


if actual_B != expected_B:
    raise RuntimeError(
        f"Experiment B manifest error: "
        f"{actual_B} != {expected_B}"
    )


if actual_C != expected_C:
    raise RuntimeError(
        f"Experiment C manifest error: "
        f"{actual_C} != {expected_C}"
    )


if actual_D != expected_D:
    raise RuntimeError(
        f"Experiment D manifest error: "
        f"{actual_D} != {expected_D}"
    )


if len(
    RUN_MANIFEST
) != expected_gpu_runs:

    raise RuntimeError(
        f"Expected {expected_gpu_runs} new GPU runs, "
        f"found {len(RUN_MANIFEST)}."
    )


if RUN_MANIFEST[
    "run_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate run_id detected."
    )



# GPU ASSIGNMENT

# Alternating assignment ensures both T4s receive work.
# Actual parallel execution is implemented in Cells 4-6.


RUN_MANIFEST[
    "preferred_gpu"
] = [
    index % 2
    for index
    in range(
        len(
            RUN_MANIFEST
        )
    )
]






extension_protocol = {

    "project":
        "QTrust-PPO Journal Extension",

    "parent_experiment_A_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_design": {

        "experiment_A":
            (
                "Frozen existing 120-run study. "
                "Not rerun."
            ),

        "experiment_B":
            EXPERIMENT_B,

        "experiment_C":
            EXPERIMENT_C,

        "experiment_D":
            EXPERIMENT_D,

        "experiment_E":
            EXPERIMENT_E,
    },

    "shared_rl_settings":
        SHARED_RL,

    "shared_quantum_settings":
        SHARED_QUANTUM,

    "inferential_unit":
        "independent training seed",

    "new_RL_seeds":
        JOURNAL_SEEDS,

    "new_gpu_runs":
        expected_gpu_runs,

    "execution": {

        "physical_gpus":
            2,

        "gpu_model":
            "Tesla T4",

        "parallel_strategy":
            (
                "independent run-level parallelism "
                "across GPU 0 and GPU 1"
            ),

        "checkpoint_rule":
            (
                "Each completed run writes its final "
                "summary atomically. Completed runs "
                "are skipped on resume."
            ),

        "cell_checkpoint_rule":
            (
                "A portable QTrust_Journal_LATEST.zip "
                "is created after every notebook cell."
            ),
    },
}



# Stable scientific hash

# Do not include creation timestamp in the hash.


hash_payload = {

    "parent":
        EXPECTED_PROTOCOL_HASH,

    "experiment_B":
        EXPERIMENT_B,

    "experiment_C":
        EXPERIMENT_C,

    "experiment_D":
        EXPERIMENT_D,

    "experiment_E":
        EXPERIMENT_E,

    "shared_rl":
        SHARED_RL,

    "shared_quantum":
        SHARED_QUANTUM,

    "seeds":
        JOURNAL_SEEDS,
}


canonical = json.dumps(
    hash_payload,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
)


JOURNAL_PROTOCOL_HASH = (
    hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    )
    .hexdigest()[:16]
)


extension_protocol[
    "journal_protocol_hash"
] = JOURNAL_PROTOCOL_HASH






PROTOCOL_FILE = (
    JOURNAL_ROOT
    / "journal_extension_protocol.json"
)


with open(
    PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        extension_protocol,
        f,
        indent=2,
    )


RUN_MANIFEST_PATH = (
    JOURNAL_ROOT
    / "journal_run_manifest.csv"
)


RUN_MANIFEST.to_csv(
    RUN_MANIFEST_PATH,
    index=False,
)



# WRITE HUMAN-READABLE PLAN


PLAN_TEXT = f"""
QTrust-PPO JOURNAL EXTENSION
============================

Parent Experiment A
-------------------
Status:
    FROZEN COMPLETE

Runs:
    120

Protocol:
    {EXPECTED_PROTOCOL_HASH}

Experiment B: Ablations
-----------------------
Reference:
    Full QTrust from Experiment A

New:
    No rollback
    delta = 0.01
    delta = 0.10

New GPU runs:
    {expected_B}

Experiment C: Noise Robustness
------------------------------
Methods:
    Fixed-QPPO-128
    Fixed-QPPO-256
    QTrust-PPO-128

Noise conditions:
    Readout error = 0.02
    Depolarizing probability = 0.005

Clean references:
    Experiment A

New GPU runs:
    {expected_C}

Experiment D: Circuit Scalability
---------------------------------
Fixed:
    4 qubits

Reference:
    3 layers from Experiment A

New:
    1 layer
    4 layers

New GPU runs:
    {expected_D}

Experiment E: Confidence Bounds
-------------------------------
Clopper-Pearson
Wilson
Hoeffding

This is a controlled CPU benchmark.

TOTAL NEW GPU RUNS
------------------
{expected_gpu_runs}

Seeds per new RL condition:
10

Journal protocol hash:
{JOURNAL_PROTOCOL_HASH}
"""


(
    JOURNAL_ROOT
    / "JOURNAL_EXPERIMENT_PLAN.txt"
).write_text(
    PLAN_TEXT.strip()
    + "\n",
    encoding="utf-8",
)



# UPDATE MASTER MANIFEST


MANIFEST_PATH = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


if MANIFEST_PATH.exists():

    with open(
        MANIFEST_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        journal_manifest = json.load(
            f
        )

else:

    journal_manifest = {}


journal_manifest[
    "journal_protocol_hash"
] = JOURNAL_PROTOCOL_HASH


journal_manifest[
    "experiment_B_ablations"
] = {
    "status":
        "PLANNED",

    "new_runs":
        expected_B,
}


journal_manifest[
    "experiment_C_noise"
] = {
    "status":
        "PLANNED",

    "new_runs":
        expected_C,
}


journal_manifest[
    "experiment_D_scalability"
] = {
    "status":
        "PLANNED",

    "new_runs":
        expected_D,
}


journal_manifest[
    "experiment_E_confidence"
] = {
    "status":
        "PLANNED",

    "gpu_required":
        False,
}


journal_manifest[
    "total_new_gpu_runs"
] = expected_gpu_runs


journal_manifest[
    "current_completed_cell"
] = 2


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        journal_manifest,
        f,
        indent=2,
    )



# CELL-2 STATUS REPORT


status = {
    "cell":
        2,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "experiment_B_new_runs":
        expected_B,

    "experiment_C_new_runs":
        expected_C,

    "experiment_D_new_runs":
        expected_D,

    "experiment_E":
        "CPU benchmark",

    "total_new_gpu_runs":
        expected_gpu_runs,

    "seeds_per_RL_condition":
        10,
}


with open(
    LOGS_DIR
    / "cell02_experiment_plan.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        status,
        f,
        indent=2,
    )






print()
print("=" * 80)
print("JOURNAL EXPERIMENT MATRIX FROZEN")
print("=" * 80)

print(
    "Experiment A : 120 existing runs "
    "(NO RERUN)"
)

print(
    "Experiment B :",
    expected_B,
    "new GPU runs"
)

print(
    "Experiment C :",
    expected_C,
    "new GPU runs"
)

print(
    "Experiment D :",
    expected_D,
    "new GPU runs"
)

print(
    "Experiment E : CPU controlled benchmark"
)

print("-" * 80)

print(
    "TOTAL NEW GPU RUNS :",
    expected_gpu_runs
)

print(
    "Seeds / condition  :",
    len(
        JOURNAL_SEEDS
    )
)

print(
    "Physical GPUs       : 2 × Tesla T4"
)

print(
    "Parent hash         :",
    EXPECTED_PROTOCOL_HASH
)

print(
    "Journal hash        :",
    JOURNAL_PROTOCOL_HASH
)

print("=" * 80)


print()
print("Run distribution:")

print(
    RUN_MANIFEST
    .groupby(
        [
            "experiment",
            "variant",
            "method",
        ]
    )
    .size()
    .to_string()
)






CELL2_CHECKPOINT = (
    save_journal_checkpoint(
        2,
        "Journal_Experiment_Matrix_Frozen",
    )
)


print()
print("=" * 80)
print("CELL 2 COMPLETE")
print("=" * 80)

print(
    "Experiment matrix is now frozen."
)

print(
    "Do not change B-E settings after seeing results."
)

print()
print(
    "Next: Cell 3 will install/define the exact "
    "reusable VQC + PPO + QTrust training engine "
    "and perform a short dual-T4 smoke test."
)

In [ ]:
# Corrected Version
# Dependencies + Exact Engine + Dual-T4 Smoke Test




from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import socket
import shutil
import gzip
import base64
import hashlib
import subprocess
import importlib
import importlib.util
import importlib.metadata

import numpy as np
import pandas as pd



# REQUIRE CELLS 1 AND 2


required_objects = [
    "JOURNAL_ROOT",
    "LOGS_DIR",
    "SOURCE_RESULT_ROOT",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1 and 2 first. Missing: "
        + ", ".join(missing)
    )


print("=" * 80)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 3 / 10")
print("=" * 80)



# EXACT SOFTWARE CONTRACT FROM EXPERIMENT A


# These are taken from the environment metadata saved with
# the completed 120-run Experiment A.

# We do NOT reinstall PyTorch because Kaggle already supplies
# the CUDA-enabled build and unnecessary replacement could
# disturb the CUDA environment.

EXPECTED_PYTHON_MAJOR_MINOR = (
    3,
    12,
)

EXPECTED_PENNYLANE = (
    "0.45.1"
)

EXPECTED_GYMNASIUM = (
    "1.2.0"
)

# PennyLane Lightning-GPU 0.45.0 has a CPython 3.12 Linux wheel.
EXPECTED_LIGHTNING_GPU = (
    "0.45.0"
)



# BASIC PYTHON CHECK


if (
    sys.version_info.major,
    sys.version_info.minor,
) != EXPECTED_PYTHON_MAJOR_MINOR:

    raise RuntimeError(
        "This journal notebook expects Python 3.12, "
        f"but Kaggle is running "
        f"{sys.version_info.major}."
        f"{sys.version_info.minor}."
    )


print(
    "Python version :",
    sys.version.split()[0]
)



# ROBUST INTERNET / DNS CHECK

# Kaggle occasionally shows Internet=ON while DNS activation
# takes a short time. Instead of letting pip fail repeatedly,
# wait briefly for PyPI DNS to become available.


def wait_for_pypi_dns(
    attempts=12,
    delay_seconds=5,
):

    hosts = [
        "pypi.org",
        "files.pythonhosted.org",
    ]

    for attempt in range(
        1,
        attempts + 1,
    ):

        success = True

        for host in hosts:

            try:
                socket.gethostbyname(
                    host
                )

            except Exception:
                success = False
                break


        if success:

            print(
                "PyPI DNS      : PASS"
            )

            return True


        print(
            f"Waiting for Kaggle Internet/DNS "
            f"({attempt}/{attempts})..."
        )

        time.sleep(
            delay_seconds
        )


    return False



# PACKAGE VERSION HELPERS


def package_version(
    distribution_name,
):

    try:

        return (
            importlib.metadata.version(
                distribution_name
            )
        )

    except (
        importlib.metadata
        .PackageNotFoundError
    ):

        return None


def run_pip(
    packages,
):

    if isinstance(
        packages,
        str,
    ):
        packages = [
            packages
        ]


    if not wait_for_pypi_dns():

        raise RuntimeError(
            "\nKaggle Internet is enabled in the UI, "
            "but this runtime still cannot resolve PyPI.\n"
            "Do NOT rerun Cells 1 or 2.\n"
            "Wait about one minute and rerun ONLY Cell 3."
        )


    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-input",
        "--no-cache-dir",
        "--retries",
        "10",
        "--timeout",
        "60",
        "--index-url",
        "https://pypi.org/simple",
    ] + list(
        packages
    )


    print()
    print(
        "Installing / aligning:"
    )

    for package in packages:

        print(
            "  ",
            package
        )


    subprocess.check_call(
        command
    )


    importlib.invalidate_caches()



# INSTALL ONLY WHAT IS NEEDED


install_requirements = []



# Gymnasium


current_gym = package_version(
    "gymnasium"
)

if (
    current_gym
    != EXPECTED_GYMNASIUM
):

    install_requirements.append(
        f"gymnasium=="
        f"{EXPECTED_GYMNASIUM}"
    )

else:

    print(
        "Gymnasium      :",
        current_gym,
        "✓"
    )



# PennyLane core


current_pl = package_version(
    "pennylane"
)

if (
    current_pl
    != EXPECTED_PENNYLANE
):

    install_requirements.append(
        f"pennylane=="
        f"{EXPECTED_PENNYLANE}"
    )

else:

    print(
        "PennyLane      :",
        current_pl,
        "✓"
    )



# Lightning-GPU plugin


current_lgpu = package_version(
    "pennylane-lightning-gpu"
)

if (
    current_lgpu
    != EXPECTED_LIGHTNING_GPU
):

    install_requirements.append(
        f"pennylane-lightning-gpu=="
        f"{EXPECTED_LIGHTNING_GPU}"
    )

else:

    print(
        "Lightning-GPU  :",
        current_lgpu,
        "✓"
    )



# SciPy


if (
    importlib.util.find_spec(
        "scipy"
    )
    is None
):

    install_requirements.append(
        "scipy"
    )

else:

    print(
        "SciPy          : available ✓"
    )



# Install in ONE pip transaction to avoid dependency churn.


if install_requirements:

    run_pip(
        install_requirements
    )

else:

    print()
    print(
        "Required Python packages already available."
    )



# IMPORT SCIENTIFIC STACK AFTER INSTALLATION


import torch
import gymnasium as gym
import pennylane as qml
import scipy


print()
print("-" * 80)

print(
    "PyTorch        :",
    torch.__version__
)

print(
    "Gymnasium      :",
    gym.__version__
)

print(
    "PennyLane      :",
    qml.__version__
)

print(
    "SciPy          :",
    scipy.__version__
)

print("-" * 80)



# HARD VERSION CHECKS


if (
    qml.__version__
    != EXPECTED_PENNYLANE
):

    raise RuntimeError(
        "PennyLane version mismatch.\n"
        f"Expected: {EXPECTED_PENNYLANE}\n"
        f"Found   : {qml.__version__}"
    )


if (
    gym.__version__
    != EXPECTED_GYMNASIUM
):

    raise RuntimeError(
        "Gymnasium version mismatch.\n"
        f"Expected: {EXPECTED_GYMNASIUM}\n"
        f"Found   : {gym.__version__}"
    )



# VERIFY TWO T4 GPUs


GPU_COUNT = (
    torch.cuda.device_count()
)


if GPU_COUNT != 2:

    raise RuntimeError(
        "Journal experiments are configured "
        "for Kaggle T4 x2.\n"
        f"Detected GPU count: {GPU_COUNT}"
    )


GPU_NAMES = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(
        GPU_COUNT
    )
]


print()
print(
    "Detected GPUs  :",
    GPU_COUNT
)

for i, name in enumerate(
    GPU_NAMES
):

    print(
        f"  GPU {i}: {name}"
    )



# VERIFY LIGHTNING.GPU IN CURRENT PROCESS


try:

    test_device = qml.device(
        "lightning.gpu",
        wires=2,
        shots=None,
    )


    @qml.qnode(
        test_device,
        interface="torch",
        diff_method="parameter-shift",
    )
    def _gpu_probe(
        x,
        theta,
    ):

        qml.RY(
            x,
            wires=0,
        )

        qml.RY(
            theta,
            wires=1,
        )

        qml.CNOT(
            wires=[
                0,
                1,
            ]
        )

        return qml.probs(
            wires=[
                0,
                1,
            ]
        )


    probe_x = torch.tensor(
        0.37,
        dtype=torch.float64,
    )


    probe_theta = torch.tensor(
        0.21,
        dtype=torch.float64,
        requires_grad=True,
    )


    probe_probs = _gpu_probe(
        probe_x,
        probe_theta,
    )


    probe_loss = (
        -torch.log(
            probe_probs[0]
            + 1e-12
        )
    )


    probe_loss.backward()


    if not torch.isfinite(
        probe_probs
    ).all():

        raise RuntimeError(
            "Non-finite GPU quantum probabilities."
        )


    if not torch.isfinite(
        probe_theta.grad
    ):

        raise RuntimeError(
            "Non-finite parameter-shift gradient."
        )


    if abs(
        float(
            probe_probs.sum()
        )
        - 1.0
    ) > 1e-8:

        raise RuntimeError(
            "Quantum probabilities do not sum to 1."
        )


    print()
    print(
        "lightning.gpu : PASS"
    )


except Exception as exc:

    raise RuntimeError(
        "PennyLane is installed, but lightning.gpu "
        "could not run on the T4.\n"
        f"Error: {type(exc).__name__}: {exc}"
    )



# LOAD EXPERIMENT-A ENVIRONMENT METADATA


metadata_path = (
    SOURCE_RESULT_ROOT
    / "environment_metadata.json"
)


if metadata_path.exists():

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as f:

        experiment_A_environment = (
            json.load(
                f
            )
        )

else:

    experiment_A_environment = {}


print()
print(
    "Experiment A environment metadata:"
)

print(
    json.dumps(
        experiment_A_environment,
        indent=2,
    )
    if experiment_A_environment
    else "Not present"
)





# The engine below is a gzip-compressed Base64 representation
# of the verified reusable QTrust-PPO implementation.




# SHA-256 after decoding MUST equal:

# 6c7283af1a0091388b47424ce36672409ab1db6cfd35a2fada9cc6004daca188


ENGINE_GZIP_BASE64 = """
H4sIAG3drmoC/+197XIbN7Lofz7FXKZOFRkNaVJ2fBI6k1qtLMc+68iOJOfePSrV1IgExTkeztAzQ9lKjp/jPtB9sdONzwYGQ1Ifzu7mRrXrSJgG0Gg0Gt2N
RuOr4Oezcl3Vg7dv3wT/UazLPMmCo081y6u0yIOj/CrNWeer4Dkr02s2C+ZlsQzqBQuSdb0oyrROaijH4l8ZgH9aAdyS5fXgIEiXq4zh7wBS5MMO/F2UdV
BUYVAm+axYhsG0WN2EQQ01OrzhWVIn0yypKlYFElwXhcE8ZdlMANY3qzS/UjCv06pWzefr5eomSKogX6miFfQGBfC/1UyV1UU5XVh/DPOcV8vd0uF8nU9xCE
AZAHihvl/dLPOkStdLLIU/dHcsz2+yJGdY/mGZSYx5azPAtEwv19icHuJhUrMroOU0kbDVNF3dDCsgnIa5ZHXS6bw9ODk6Povfnrw5e3P45nX88uD0ZRAF3cfT
+ZPL6eX4WzaGn8vH3c5PB6dnRyfx6dHRcwB4st95fvTLq8Mj+F0iwq7TKet1p+tZMhl1g3QuP2DBMK3i5DpJs+QyY71+wLKKBd3pat3td35+d3B89u6n+OzN
yeHLuK1VDto5Ov4lPnxz/OLVjwDxWyeAn+5hUtZvi4wNrsfdSfBbF0fJ4lm6hL+ehEE34aSWBfufQ1HrYFoWl0Xtq/TUrfRYVXq9zpPyNUw+KwfXjxsVv3Ur
PoGKn/UAXcSXSZrHH9aXaV1xVElpltywEksfy9JklqxwXcTVoqjjarpgs3XGAOD88X749Ek43v823P/mafjNeD8cj/afhPujJ9+GT0bfPb2QLcxYVidQYTQc
fYNYwfJsYHSVLJcC5rvvQlXEAJvl5UyWfyPLp1m6itmqSrMi51/2RwrVKUxdnJWIPRuoYU1hZadTUTxmAzWu9WqG9GOrYrrA8Y4VFdI8vUzq6SKu0l9xnE9V
Q7D8S1jk8bRgczEcVec6ydbMlH8z0hT9FF+VySzOi3KpP33udGZsHlSsjq+y4jLJ4oqxWQ//mQRpXgNVCMv3J7ytohqy/Doti/y8+/bvZy/fHOOCQYDuBVSA
pcgb6HNgIZOGutn+MxAhQ0+pYPVlkq8pFqKRTctI4IQ/BKTRzLOWr3GSZaQjAXSZTN+zfFYhNAiqGatZiXNRweTBAEGys3boS5ZPF8ukfA+QLxJY4gR0XcEq
oa1B9yii6sWy6mGzYfAxKfO4yLObCP+G1e7ODZ2PTkfM3zJ5D+yTX/fg/3GeLNkEZyEMNs9jt9vl/z0sGbBfAFPCKwRJ8KOWwXKicb8JSrYqi9l6ml5mN0Or
BZgf1XOQFzX0GBghZSaoTFKQeL8ghx6VZVH25t13+fu8+JjTfkCgqMY+o7jDmlAAQ4DdYIhD1cOUX4tL2MbSfF4ADHwalgyIxmc1ElOr2hhKwVStkikbOkyG
36ElVl4nfiDBz6yG3RyBBelXnCpTVlUxl4I9/m8YCJkdCUnuErzIrxnsPok17oB0jnNWwETMgU9gauZZkdSP9wPUH4pyKDA5rdkqGE9wWjLANFgVVQVTw4Lj
5PjRq3wecFFQBVUyZ9mNqbIPzFHM6+wmuCzWMOcSbF3h1l8n+cKAPp4Aa60EHGopwCdrIC5wc/BhneQ1MEhZCEUEBnOVwafzwSoNg1V6IZA8LgJOEFU3SKsA
dutpUkKDhon4LwIwQvmQVElZJjeamKCYsAjKJSHkXHwVnMLYatBLrmC/qOpgnTNQlaY14AoKC+N7vxyf20Oe5HFdxADW0/wpetN/AkgEktUUAIWBy6KxVZiz
KywcYCkvNMgBlQMUt0mW/iqIBIQ7H4zDYHzBYSRhY4IXToAYtm7nZ0HqwbxMQc7AvLXTHCvwQtrkKg2+truyWFmIJsFahhakFTNWMQ+igpwK8lFwvPiPosRf
tKLZ4f8GJ0WWFev6r+v5nJX2sjhdJCXMXF0m/8Vw+wQG5VDBvCgDkNMBKtPXSZkCOSof71QTrrOe4/TOOAPhfsT12x6s1WSd1fGc78w3UQaAYt0LmaDqwsLb
pVJWXMWw7i9VNU6NbRUl1UHEz25fT7DxbaoJ7vxUx7evKvYsvk8BXTW2l0WR7dJrXYJuf9uKktvfVcADGVQW8y4k4AA1vuBnZIBHxLQSU4/K4BTYu77FHHYE
x86DZDYjAoBl87BNHAg+MX8rFjAlYmrN35zuVFKouTBlhsakTFGPoAJjrCIpdYjKgwgPBe8PkxWYSGQwvMvtonSI1mKvr6tJ2ujW5fJQzQNte6Ko37cB9ZJQ
oLyHnip2weU6sIFFYd/FQTCwDcrL3EYJu9vQ5kOjccLmqg6ya8+Uu70Y9rYrqOJGF4RFKR35rPYJM04zlqDynM1b5lgA9P0z5P1oZsX7Wc2C+uilu7cmJbW/
NiWstwlCR38LlG4WBBIrjjOWx7FLLrmrwbceHWFfqGvTYrlag60FFl2PiuOQyNjQlZyhKw5DR8qJv7nhGBmT8lyakhfqszIiHRhtWwJgQ1Pk6AY/spyVqEaA
bDyYXcMGmFyx4AhMiKV0A3HwV9yrAV+JBo++kVR4WiauXOc2xoQKHqa+ZlJZGvwAKowgDOgQwK+wPa9kd5oAbjvpkg2ydJnWj5KyTufpNIX2gOtRw4QGdTtA
fbT0K2w/JeLuco3WxnRdVugF+/HgKFiCxAeMClBGGWwLYK0EXMsXWDrKgJxTW5VUE+1IwKdP+mTq7Spy9ltqEAaxq1HO8dR1mcmu7NkZKlcBM61RhdOwY1uL
Dr9uaxBMIDACsDVcTGYdSYsPDT3TOIKIMZtFHomahB6Nb2SUzW8aX+uTRMvdChvGpbURdg9Af0RGSnNYUAGnTCW4apEAj6UzsMC4uQA9X9WLath1d8RErTtJ
319ZWVSmF4nWbqSFRQ+NoMXA/0RFh1vNJQPDEBSgXgn6N+vJNvv9iRGMX3H/g2YPNANgFRXXrGquUr2eVFG8TCp0TIyHo2BAWey8vqBd/LWoF1YX6ByQ84F/
8pVordCQ1L5k02QNk4EObZx8BJeafV6lvIFlAto9ywo0OQu9mNkqrYoZM4jLghhkB2C9TD7Zs2oPILS/ae7BsXm0G4k+WPoWUUiXHWLbZHUCIHb3cknQ9vFn
T+wEVtnXVFy4Fb52Jsj6OAialcgoBCfZeHFsd0DJbD3OB5s0zVo+RMzyOOfaN4IRMxMXDVlBe8osp5ao+R6qWmLXVkY0iw1Iz1mUYvZhtURjNvjW3UiPVQvc
iCSI4NJb4oYCGwk6T4oV7F7SXHe2FVcANASsi0xTCjzep1IAhCjKOVOtH3wPbNjQZQyAqLdkSW6Rc4glUruq6pn9DQp6tvPKhzHwGTYi0AMjq4ft7CE9oa4w
3Q/xX5SRB2hJ9fJ8+FOBDniH1Kc1HgyVs+Cn12+DVZGl05tgjUZdUnGJgDt+MFVt8fm4TEBVS3N3HxdaHhqAoOZtMtLwpME11OyyRToDCY9lkXKiUxV7vWKg
Ww51Z64imrP6Y8FdujDqU/ZhjbtFkjnGVj58DYNADV4jRTruhy70Gfp6oLilFVP1wZoxpCHNEJ0aVgOKNK45h9KpQuikphh/TuSyXud6dc7UdIPRkSoXjVtP
8iAla0921GC0Q35esonTajYQux4TujCs5l96Vf9hGOlPpuHNjO/EKxvmeVgBLdivrDcYtwiXLRMPZtEl4DlDNzkr5+j95hLm8oaLE9hIYUpAtUCthSF/+MT5
P4904Z4pYBNHwFpTozGJPDjZeEU+FB00yfy2en7EaSXFS07L7RDbrde/yLiAgp9Syr2MO+imNWWxFml0mmBYRFBgeAAfvQmpAFWmxOMVKZs+LlIAnKMb+iPw
iLGOFrzqGrYk4cIOirnYr0AbzIJVUiZLPLpzxJrj/sbFDoqeQyI9yUPTDHW5yRP+js13+vw/qWLXPe/xT2510Dec9N6pB6VE2NRgri/R3iL6CEVNAK1ztZZH
pA0h/xFIj1ydaWhE0gpPJUmQRk/UisR/qGYpJjTidYYVn2lCPeXhUgDqb+WmtN2xCinB3NZJi1dD4iTRPs9hWrNlr++IU9vRuQmGo4Dnk4tkxcXf+eiiCU+E
rJReoPgKJ99GQVW5UqpqyCK6Zo5k2wo4SLKSJbMbZa6Bmob8jxK1FKc2LazfOt10Fm8x4dpn2TKlxB8gAyEUoPyTSljtoWlMe7Vt3jUe9lTKTuxC6imUc9jp
6LO7QAYJIBJMnCMEFZ6WV9MUtYI5yNlpWk7XaQ0yLGApEL5UlYbcBuKYsFieu8byW0+sZHEaPpFscx19WGYqaogAhB9TYL1oPxTHCccgLaWcTefzGMTSophF
XS2iBtUindc8iIk0EkXdLL1a1LjBDq9W666MY0KEAMWV4K+/IAYfchCdiEBoNuqoy6VTN7Q6Jb8byS9G3PsUAjHqhDAxtn3yd/ggxjPqP1NFHFIWj2Xx4fGb
s54oOh+F4wsolrONX/n8Wp95N58i61h0NHz872FDuCpPIO/VrbA/9lUIS9ADsTO+zckYD362jHhEzpCfBb2BqA6s2OMgIDD2wMAd7/f7PPKEq2B9at2Jswhe
K63E8Zmo2h9isEtf+HLs77y7IaKExxEHxwev/3726jD+WYQvxMcHPx1xdumguRxPoYUUQ5ZQRJw77BB25YnbkEd1dS8m0jt5YyawhZdNwzByPxIG5FlwCdJK
OCfYpylb1cER/w93dOsjezx5Bgb2tobBCDiqScc4EE/WObqvhQuxe1zwEwNQH9E38BZDEF9jCOIHZ1nP8XS9K88YLtdpNouTPMluMMRHjVLoH0LCcC+k0YNU
BFxkR8mdW9FxFwpWxMX5YWXMXPMk4ZefD7UGBNIFMM+SfApS/r/3fgiQCVIdpSC9+y+TWbJEOx5DXEAm8E14hR7KvBaWvKym2kWNi9ecwm6RrEQ8SYEMyvUy
cfSSiJDPAZj8QH25q7qKOQb7EBGmZ9M7i0bmilWsCOaenuJMh9T9QoSO5UJrSECqG/kZCdQkZxmYHcojHZXOa4QkVSMN4k2pSbZTr/RUA1Qy1GE+FcETBh8Z
olvZnuXoDj/Ua6x4Si0Pm6/u25Guj0KIzzF3mUtPuZj0vq2sIoEVH0shz+H6DzjqXzAiRQYyi+UX7PH4ahDIg/UK5P6M2hkPMXjeDR286JdOpuhqcMsfpzp3
sIC+My2sIdyxbVsf3jiHFigHF4wLFvSMfWo4vLXCxunytZanMBH8lwZwP/g3I4ObvfHgJ4wgkcvl3Or+olnhq+AnWMUlpRaodgk0ki5h2ePhDPqX0yytbxqV
pe7iHdFo+A2Mh+MTegEIUzcB+h1/Z//Z1tn+A/Z2b+47QSriRQfEByMkfy8GDAbB2MeEWpvcQJtz7zde20816zMw7NgPcrGN3O3IbUKMDtnf8ahZbOPy0NN+
ht47fhbjxphW/0gJtGmVys20fe65YArvyhqjFpb48hLhCw9s/EUG1rDuOp7VMNJBuKS2qmkpTUKpv2K19MHE6CQpYOtYJnWZyhNp1+XqM/06xAHnu+/jif7J
1st8UNXFdJHwywfm2GxVVPVARp6jnBTISNX9KJkuiI2CmjYgDAo4j79eYgjajFucfEuqeOCDDBGvC66s07h0paTz7y9g1TzWMbOagP89Gv2AoT3SVzciH8b0
w9h8GFs19skHUWOdp7BElxjvzG+L8AOmZFoWVcWjgTUWArGzBR/aewxlFzUdCujx1injROCRB7yuqvB4ILGxfMUrDE4or5HKq8VNpYLK0xls5o7pks4JJ6BV
sE8kiZgiUCvs5XQ+HgIzYkx5oH5zFsC59ZH/RiCkOsKyRuePb9W5ajp4FDzeBYUNcAbYB9eO8JPbI7ydWqOtSG6E2ETxiu0af4Q3Xar1CsPzOBdzLpvhnU4M
s5gEvxlCfG5EHmlS+EP1xWc35ID/2xKdH9oXFY6uWXmjV4uSFDw4St+sZNB5DYy/xMO6SK7jKRdQcbVeVjw8h4sg+KuHhz3qQADvnep7n7Bwp1lREYOXtBE6
V7lAEFVxlr5nPQJE/ORJXWQY7TEeWQOSUlygI10y0jJVItySBQIXLiSou5eKdEcw/7xBsBiRZpU3QhT5HoJxMps3lh2O9sR8E/yH/rm3IJp8QEhTWRaWwvUv
lEYbasqbdlmyXDXG4LjTl2nOJ3Df4chjfXsHKtTCY74ZSevDI28x58y+h1MokDqPllO8OdSFmv5q+citQxz3dSx2MYd/ExI2KH0G/+//GmUcft8XVX/CYBmm
73+beo8R6AmHBDrsPyGNf7Fj7tu4KW/jqrzteblGVx1DciveExMvYcwfToCGonekh+YCyMmJ9FgcVLj3Dj7v4PM1Tj3pd2g7MNdkbnoyLaqqX/zHuML1FksN
vuEywRvQTmCfWLeobOVNa6DZmYtu88t+s6hVLzZDaAtHUDQ1I4Jl+VZxvUtja/Tek1X00zB1aXNrgMFJ8tFSinV0vLocqda/uJPXclIqw6bV6Ys61nc7boYA
+M7/W2IAdiRzx3+gXxc9X8CA317ZuUvL3/pWiPTNtzbZp7pkS1a1hET4rm+20KNxjbP1Kqd7nTPstFuXTiAyR2qVelcUvdLpYUQ03zIWe8KpWljxZ3unsdUP
9GqgAUfuRTS4ULC+uXbKl5e1Hpo09ZGCqAOqFec0o9Fb2AyMcda1l+it275AY6tyZ4M7ip4vVMcbP9XfwA+2DnGHeMpTYR1U8vDfFgbneqswPovCheGpIsLA
A7qzDKp8QqjaTQpV9xNDKu6oags8ogGFzpppQaitK5FTQzHTeZMl3SVpr13lPBTLJ83d3i4azCEoAGDT9y7rakwcaYoG1JeJkUvyRoic0lvIKXGL6LAWu59M
myJ+bBnMQx6MNeFd842wr+1xX56wyJ0F+UOEeikLC6Z7AWrsdLWGf3kCpd4/MMhLBnWpWC9kYIzsat532Dzn3iXnLIDGJFZaCa8aCkbTHvVJD7wp5N0Z7slk
7aFmHi7ytrA9Au3esWUOnI6ZfnlzWaYzaqmKAN1m5IeIMN4j/mMRCCcNVF7X8M/h23eDeTJFh+fbmzOkv8w8UtFYi3oRvFYhD4Mf374LwGpDLypWU8pwlS7X
GQ0qoQhal0VURypoHlb+4bvnB18uZlvmYvKHbvvjsy2fwINGZ3/50OvGcEnnuCZloppbbzRuegSTFiHwa2skGJq35tWLyQRtt4PuEQftGjL9loBmFTosxH9z
PFPP7LhjsLURuXXsvDG52Sf8GSg8WSf8AqTTmGJ9yKV8GFSj7fEJs9ULJV9ecpYK/lqUjstT3MnAXFZ40jIrwYIm2TyEA5mJ++HyVinf4ysZoBgk61laS5Gl
YepyXS+G8sSH3QRJiXdMr1kpepulJViSmNvohms2IlcITPc05ZkXy3XG7NxViksFS7Zrfo1Zsbd5XSz2e/On3PhNQVIhc/bc6+T9Do0eFIQSie52CCDkEW6/
WyzhC5KTBXcXkTqK4xAktaD7MfrFMpiY7EtG9z1wIF0j8s+NITRZO9xAOzlhAoktYXY6Tq4lPu6B49uol1oGmd8tmux5a1DbHy+i6w8blLU5qupLRUUJ7EyQ
0/3DmazIpWC0JSbJBBX5g4m+QDCQCI2RITDB6HeM1rF7Hj90OA2NoLEP0qgQVEbKi/QTm52CzGw9UXtozZ5shn/EUyuxz0aB2Ys6X/hYa5sysvPJlthEnfl5
sCOvW58S/cucgW05r/oXPl56uHOlf7LzIXUi5D0F2u3sxz2v8Z7UuE7XW53J3PE0Rl002/3cpfXMpfW8ZXuigT/ygUbb/dTWg4Rz/0nW9oOMbYcZnnhcfjZx
nwv8D3WaQHy7D3FuoO6L/4FPCDb79Btc94/w3vsn1TsV/rvhYRNGO+V386trlVU42C3P+hdWVm/hld6qByq3tV8Df/DsIl7l7k/H9p+O7d/VsW1del0VsXhb
Q3TJM7kIaJFZXAf6FmUsD0JVoRygU2q91CFvFDvOUfdQFXdXwk3oL9WhYtix/rJnGESf0uVsXerDOnXPgcnjPHSTy/WEXnCFKiaZ4Pe1GbrBRSpGlabgJsgw
XyOPPUcCYCSbdS9b8k/VvNeAmeoE1frB93ivYbfA966KVkK6CPIFKgcCOoszlsAWsE9yQ3pSb0L3FuXti/uNiQF+phl27edVLlSM8eCOP7I6JhPd02kK79mk
m8uQJE1sZCwWK4lnnozEhAytJMZ6qejPbm4SkopSgTSyHXOGMek1FZw3K61JtanBPKlmt6RHFiA7pEimgedWRsbWJJE2qB1pfk8u4HupOh2f4Ml5QNO93psl
uGjaYrSQVJSS/FKR2/BwA7VbtoZ4WmH+VQzqaisuEgFL79ysDe7ae5HNYh00sQsOngiLhyGD5qSNWLQnA71n//fmWaFxGab9UTGtzE56X6alusHDcq3nuREP
gaTsjK82zI6EuXdfeawOrSO6Q3Y6JroP3+7C21Ep8ShS6ea88mV5omXjDzv/z9PkKi/w1mf1gEIqK/j7glFwfkG5wC0WplhKi5IVLNNP8fuM1saX1ealMUAV
LO9LP2TW7M/9JEcY3fFHVkflRegP921P5/vmzZnTJktJsSx12HGleqa5Gf3TWO6kElXs4tO9pY+nJAdcnQ13zB3Ws+NVkU9NH87xWJrPUIEEXAnm5z4PW1lP
BEZ7vnadSNo7kt2pL4VrcHAI4jV49/b5wdnRvbuwGlhexs6mTf9s0kGSa9OwRZO2Iwa3nrs2Zu2k0GRjZ70zllQds7fJOzXpOYFhH+OWyMnW6En8iW0bFcUx
2oLSXG/3rrXMatgCYOs8nhNcxJ1n0vIGPFija3wdNOZuU0+qFyEw2KdVszuNzKZ2QI8HUbxis7i4xGe50Hz04d5sSByoWWyxqSPVjY24c4vU6q05CyKDP9lc
vQff9rOdzewm/nb3HrjdluFvprJFpPtQWxzf8t0ZOpJJBi0VZQMDhK14tYD0VYJ6X6xzLNPZ99qWM81u3zIco4J4GyKj9SyrrfNqvba6JRuOmAc6tO14aycT
f1NEuSZbBkizPnpAtP7jJYR+fHldpxm+qQQMq2vE/qgSKilp+mJ/VMk2YnKC2o/R7poDpn8bSuLDKb3+Q+sPh68PTk9fHR68Dg5PXp29OvwiKgTYLbHRouRv
jZMX/BEGySaygEBwzbHNztyWXD8Ep9ulgwIEjC/LI8iNrfbAHa9KBrVqEEo6A7BnX+e83ebk9hHwdlxJjCA8iBkuK8b/8IgoB12vitGwWtu73EGmEORahYpr
VT2wVJG3UP55xUqDml9Grrgm+YPJEn5Nxzn08sQzahO8NRC3J/SyAepCfS/EoEWhbFcAGoZ+e++tky7TylxW7SBGHebYt8L1W7/8sBP/7aADtusM7QgMZ8X6
0sp4b331KEcbFaSWl0JN6IK3G6J/NG483FYS3g0DKq3uhoJ2Pd2le6rR3bF/4+e60wyo2nedAMujdqcZoC3clQaO/+7uvKjbuB9D3g8Vt5Fb4nJvv+TperlM
ypsHcUrCBlzC/gMy+DdDqa5Z+N2JJ84pXwkBROVLnz4r1CXrdmMLloCwm5Brb2N1vbjtqnrVbMZer0wHd8rxm7G3VpeDg82vO9DRMKWXlrs11WBwuy35pIc4
dJg0Qty0A1gqZJ/t3GqCV/gTF9E9fvCFDP2ad3D0fw4Oz4IXr45fnR0NXr45efWfb44D2HXPTt68fn10ct++Op2fz07enZ7Fpy/fwD+HL4+ev3t9pDNAPN4P
g6dPwmC8/20Y7H/zNAy+GUPReLQPhfujJ1D6ZPTd086FfEl4Bbz6kYHNmTMg4ozNeu/DIA+DJFst1GsV6Tx4j5Gfo8a7WCOpiKibE3zqLvEVhtVq3uNthAE2
CErLe7wx0+/rftcgqHbrN2/0O97Yr3Cgqd55NmKBgeqdfQIOFUpezDMfSb0LXZfVeorZSJXtgq5KpwiYJsnk7xX6qGLRFX26EXPBue+84S3yOslZsa4CTvRH
nATBtMjn+HLrlIk0TCKtwSoGbB6t0Fc6NDEzK7zUx59NnifLNLv5iFErDANWgsv17IrVGFRCkBrKdKdrnk9nwCkdXKZ5scQnD+oELB0edoPXThWVoV+BX8h/
5UiGSAhVir/yUn2DtRTPGTPM2IqN4tubLJUJwxUuj57oi/o8/enfgmq6YHh9ZibAqpCCY2QrPgH66G9QrHKcipuxwV+BZDjqPMWsp0UuCIeUBPmZiXcd5uss
k3d5eNVFUaa/4iOwNb4FzV/LrBdJLjqxb9PiEDQSFKVHwRP12q1wc3/E2JbmGupQZ7hhH8U6pANyARNhF+nVQjTpLo87NSkc7duQtHl8e4s7ILlrkx1t1kg8
FVkf8XdyVW+heMLTAEsUrIP4NJ/TsHU19u+jQD20zBcovmShSf1IgXkSKgqpopHrh1YJVle3nT/UKP3VDSPJaHHNKqlw8VUcyl+hb5qaM7TDMaRYYepaA7+u
pNZJ5BP9KiMzcHHjshcvVYfx1KizIpMsa695UVo/1T5IPqKsINffnavv8q05aFt3uMKLmvriusnrxdO4cGeUzlacpCrV5NsCg+iumSHLRN7Ah9al0FRt85Wv
tMexeOwVB4KZU81Dd8fsKmltkS+M1hYHzRa1OJYCJ2Mlf0xaXxyfpyW+9I4UEtL646JAQY37juFiKvn53e5rfHwDowoAPAPkkKRILYCRtJX8mTEchcZ0GLzB
bFt8M0jrILmETvFB+t6748Ojk7ODV8d9wEyFT8oJxAUGW9JSbhxOPgF+RpfomFr+97mYKfm4FOdkCwT+NiAdcTiHIgcpq1+mpswmo1FXNsyeDSOGnF8hBuLJ
YpFJeyhfaIrhWw8XTF9D4ra1CRJVgng0GuH/SWCWFlj4qnlTIdCl+HxQCrt4rILXR1IvSPjTiTKUA6icZLFm/Sjo6tmQ8ah/k+E21hrvu3qF2gdBVv3NBF5w
ABSoJaa7EIhgklWee5BfqaKNhiJSIRqTi0TJbJaKcFk9Dqu1gTNOKlsbdb+3tENvQG33lC8HiZFIzHyJogB0cMx1kebTkiV4f2bYJVauPTF74qaA3ZPgjaFS
a3oudqFiZxrP7d1OW9sHqM3t87XgjxdXG0ho711tSqgXscijmnopFHkUVhPhirtwRGc5dEP1FddFDc3WGRbfmpNaRl4YCj6yuMgiswC3Cd4CLimlK+BvRCVI
8EY4fz+uY7OlFO7BDw1+hM+E/MCvRu5Mms6Sxto9PXhxFB+/id+9fXt0Eh++fvXW9ovyPPBGkfnhls3zrfjkp6PnGzqgedrbWyJCpr2qjWtE5PRtaPH6zf/e
RgtB7Fu2b4jR3sOtiUHivEFMe91kvzVa7J4iN3UnUtx6vnPp9wg0FjybB8D2xcXhT5AqA+BfANU83gYXvC4+ajgUIG1wL4HKpkHUl1tGEhzgelbjcVe3hj1D
E84DKuwfT4XnkuYAbk+CDfvZJ0gau6klEOjydub3f1kTbLODeFVRbstrpuNxlNIiZIncFmxZ0kyRNBtizpgXeLrX49xDjvdsnumazpAUXC8yRc6pYHe2G9W6
JauK7JrNGnAOCZxqnIAxSqGV4k7c22x6uzjx9xHIJegbASbr7gdfB5vrC0M+Rn8C1EHtxqXY59DKFIVTLeKN6Y0HGYE8+fNOyT/oTkljFfhuAlgn+dK3ho/X
mPRr1SKZFR+FQ9gxK08YmD+g/a0xiXZusrfS+53i5JdrStJWw/eGzVOg+sQ9SOB/lmVKny7gr+rgbas3ZyLBGrdDGzlD9UUxgXVrrkRzsVOmW9t0STbSlynb
XyIQIM30GzpjiAKgmTvkRG05GxdDcRMiDKd4CthrHmo1ABtHQX1jg6z4TXvRAQ1+IA/XDq2XguPeC1BAmc1iogHpFubuhJtYOlTotUcuqXn+Pm2ZkT+tm5AO
p97OmaKIVwJ7guQg8feWMWtoRw/jrTA8x69CJ82yef1N7e3YFE5EasLhyfUNiw3kNV3rksp5etHMN2DffUKQpgjwX7U1dKcNW0nc7nzfQ1RvTdI4IUB8gb85
fv13ThopEpZgWa5LJh490Olc8fIpqXlVohk24MkZVbICK6HwHQdgloP0YbRnqGwYdoTHWy5UU1VKej52bN9ZQ9vbv/cEvnh1fPBazYlyyHPxrnJcGo/ag1E9
DoNKnDA3InO2OW6t2DacvIi4cu2PQMyIOHebl8Mj3x1xa91EjhfYvsfOZpH/+B4/eT/sBZi4Zzwafdc8yPf0YAlH233UTCHjczf7446FD3qXAPU2B/WDxKl7
2JlYipJBzlsUdF9WfMuu8GUe8DXpGgkdfzzVQ77sDOuKu+SNg9vOd/CVyq7OPfh5dvNgjzx7PDKOP2rN87GtQc419sImmTvNgDazj7YFyQhPwYN2S90ZG9Im
ae6CvdmxEr3enrANhjhBiBU1aQsUlkPCaadk2IyRx70V+iHuhg3X+zbOSqMKpkawCOpA6OwJToAYTbQcgw7HD4z5aUCe5BtRaK27KdC5wQX4E0UNLmvjFKln
7uqQEvEs3YlXJqZhMzS1q5dgSyWy6WwUVf7aBsBX3Xg4vJWNv6NZ1Tg9/Jtem3RtNtAWQu0hbsNjcse+Gw3tjoNwwEinyx27b3Pi3AILh3s3zT4B8zXlX1Ut
DfqBt3gR7x2AuD8MDq6uSjwjZm42d7z6S2+Z3y880Wo8ns3RJUndi65YsIzGXLE0DTV3NA/bGnW7O+9sWSIXNBJb9ks4MnCOx7a0T3mZdNF4wRL71B04A7pb
F81RLEFgP3wv2Kb7Gqeco7L4aF/X2dzH5q+bVpKtQw7zos4T4rCxvAUyBZKFZB8UKaqaOd0k0+ka2O3Gr+VaLd0ZZyf8v0+fJt4BL7q3NxyXLmGJjHKUIM8C
84gpD5Qj9iS7SOOyTZQTnnLr8/WwpbpZM83anNO31jfrwW2hhdKeVlogiWYoue/eEbb/UcC0JtmgWoHSAB0a5y/PFvaxRAWqrADybMEqFswKnk81yQAiONJv
zwYHz0QKMSYSqYMiN5iXxa/oVho8H947OlfkXpTIbkrBKPQfkoGRJO7jbmKaaFEmWDQe4yehcQ4/Dr0JF2lWb51pkbz94EmzGN0xyWJLYsWW5yYa7m8RqS3z
1JOBiXL5joMndIPk9ovcTIxeNDclXLTSLGLr27MsuhmppTDQef9EEtDNGRXptczIyvIj3HC+jD5WIh8nJ6J1zhUJgU1zINL+VJbDvv8ZSDMQkbNQvgrZ7qiP
/0uwveuwJ45G5ai3jvTaHPQ8M6F0NWGWcx2ZiEmJKWQzk57dUAYaXM+NjUxmyQpDAmMLtnvRp+7/SAZzGYs/Ag1rsMU1b9wSCL23BXqjB99adpHrvn+mfIsN
r/0z4lYUXGC55/u2Tzpq9xibaXQ4V7g82yuaSXcqxqG0YKJd3a/U9Uo9rbZjdasflftQuTSCX/bSr9Ez2g939nsKRhTE5L/3wx1Daj2yS5m+kbLmjK0Mc2es
arWEFRQxyMkkunZYRKIOfjDc2Ldccj9EIxENTYC/N4zeb3Vmtfiv/C6rzxPHWRJxz1S7Y8rri/K5n5otCy+T5eFxQbhXSQtfoY1GQn9s9zEJMl0WBSZ2tpqL
XMr32xw7jq6p/DhpSB00hnEtzwt1srT4V7QrhThONNPoooutISK6TuPTRdhuE5lqba6HRu2Gg6HhS2j1HDS8Ap/l5jyPLGNazkDfsZ7lkkI5DmYWIY40Q/qW
HRcBDIK1IKPtLZXZTSjAsg/VyIbqsk/XMuvLiwrCsiF7MxgwhpX8JospNCTy2iQoC3FsdCqleU50NZ85IqVgo64cjVXZY420VxcGNW2g1RixjY3PUjOp8Y0k
2FJAkY0/YO5kSylh+XWcA29ojVr2X4OsDgN9m1D8yfeLnw5Oz0DinB4dPfdlTx63KuVKWQHsQVmiHQX/ZvfE79bJUIU6vspg7QAkdE7C2gHxiIfWwC89PQqO
oQEySu/R8S9qL1LAF+dd/b2rNYcN0AZA7mA820W01bYxundIjAFBbx3jYuv1Tt4bqe/yv4cHs2TZa03fk5XWvitv8JZqc3QzX7S17Evh4TStrvbqtoXOFZ2I
qfwr/6vXfxaoCYSpjUbPFL/k6+UlQEMBW6UVPn0iS8YC0bUISrhW+dvEmkB48gktkRSsWPNRtu6oq8J2iCOYzSHIAiZUHs4pe3b3gK9QLUkmU2p5qAceH+9r
FivrCKjHhvhSLgiyNd5ckebRx0UK2jghwPeE7dsNJWFfxJGeZWklPWsjy57PjuWRegJ3EfYXmpi8UMfdSarwDCjyrYxnpCqlA2nQTwypHJluJiSCMBoRV2wz
YDmvbXOv0dntbb7WcDCKlRC9VsqgvG5agsqgmMkHNUJ3ziSJxdSZ9v00dx0Bph/CK3tyKRA/pbSDosiSlzYhrfUFbTzjEd7vYWXhY29n4ndLgPCnhORzHz7q
iSo+lxm/Lx55MvOLpPzSuHVT8Tey8NtpUu20WfYKR5JJfIZ8KVVDsLh6QjvrhqM+yhtVhX1i09Ya+HHNrUNezfY9t0mYvcj+u+Mhx1CSwHNYKb6AQkKnKOyS
Oe9OyB+ewHK66EHx80sDTz0b7VghYheHXYt0kjoTq9DTdiu1LASdrjY2Y6nSbSJvx7Y/932rwzFG5NQB88h1Ps1YUvZapFqAyeHVWraXhb2Z8NX35fYeY9TJ
alpkyuOCZFVBT57NacC3rWeoRcFQi0rlIVKJIPjqtWwHm259qnOLZ3ZBie1yHWjwM2gJg9/4VHwGSxX6SMsiX/L5QXFl83jXSI3KXRmdW/J99w7817XPQ2TF
UagMBCnb7s+MXTkfoMlOi3wGjcmCsKsfrJzYrt+ufplyYvl+fRq+NChatPvLpGLxF1fxLd8k5ydoHZPQxUAoYnCixyP0OTfvYiO0Oj/v7/T8A5sfhCH+tEFa
bBAwqNkKXVtSPCEgj8WdxZypdfG9LBbykToF/n83ZgyD/mnR/EtZNGTivqRZ07zyze/NRN77VkRU9RsV5dWeyDKJWq79oLaUu1kbsREtzSST4YUiqMVW+Euv
mcdYiGRYDI7pQc4B42jjXUAnhvdPE+93MPH4Xf278BkqmRhrwwV8tMNpsZ+7Q4OAprxhGL4DWOsEj/QwqcftbjcILY7/22nPdviBThcM49znUr/ot25we9GH
JoGJkigb9Xn0L34YORlf8S05Dcjf0POrnhPPZRSxyYvDMu8G70qbdim8SYSIK4ZE4PQ7/hyYRkyAIJ/FRFZ4BE1/hzwAeoiodz9rqDXu8B7ej6Ee61M987MK
/pvHD+A9wtnEDOHWKDtzRiLbafvua8tz2iNbaX65uLPLZkcPzQbPTrvLpktx1C1/2MER80D+HJt6LR++pEtob6c+/3QV/X6uIpMgdPCb0Rgf0F9kyzgjcVRJ
2PVI+u7EU/gPdENtZdsH8lJtXx8P7r0idyEnVgxR179zdyc83sT/0fGIcd1MpAbBa8vceVP1uJ8sza/4n+i3kkuqih6PpN8JOTGSOtOeVCtNnT6qU5T5uUtJ
RKlhTe5U2RNjVW33+yqxh36PSoZ3cZ1LvmGjVD+xmuipPGKOfjUhUvjNC76NK0cZD1MlaodyiWGs7wIkQ12XwvoIpQLdn3hSkZEbj+YZ32b6iC71DBn1T2Z9
4CbfMxIp+IzGdKM/Y7N/QbyfJ6RcOvsU6qlzErlxkjTzt21wyemW+q0ytgFyG8lq5L3SIrXM12olN6bjar1ErwH2gZlxjRIrnCs4a72W3aM/2cENIEJs5YQA
32bAbkXJ3wDxqItkevYiMquep7yFni+eZSiveLIg7Eq7FURQHOFSEa+DtQ6h36uixGBkUSmSVUXMV6/pmjBertu6eW49dSZHDs7Nno5TwpK+nCaqIJNt8Fkj
pK0r2ReEI+HjLrIVFCkOw5A0bB42G90xAImtjv9HKiI77MptIV7PdOIfEdLF+7sA6xi3ytVNzybI0yfoIYtEvBWv1tf3kCI7/7j6zmkzMx/hD/UtnM2KOSxN
zhP5D2PBCYLrGcafzx5hhQ8lVGzAtId2mQQUWrh2J3kotHBOXEVWLEGCzuxiKGjGYVkQZKg87koPNsT3XNtgUwpIOkg+tdWA1UOa5kDT9LtvMKeZAsMxDMbD
755+DTSjzRroBc9yZsD3DDghiuQrE2EmCnyhaYTCjkIhFr8WFhJpCQsLEjm0qRnIPfl/AGWGCOw35wAA
""".replace(
    "\n",
    "",
).strip()


EXPECTED_ENGINE_SHA256 = (
    "6c7283af1a0091388b47424ce36672409"
    "ab1db6cfd35a2fada9cc6004daca188"
)


engine_bytes = gzip.decompress(
    base64.b64decode(
        ENGINE_GZIP_BASE64
    )
)


actual_engine_sha256 = (
    hashlib.sha256(
        engine_bytes
    )
    .hexdigest()
)


if (
    actual_engine_sha256
    != EXPECTED_ENGINE_SHA256
):

    raise RuntimeError(
        "Embedded journal engine failed SHA-256 verification.\n"
        f"Expected: {EXPECTED_ENGINE_SHA256}\n"
        f"Found   : {actual_engine_sha256}"
    )


ENGINE_PATH = (
    JOURNAL_ROOT
    / "journal_engine.py"
)


ENGINE_PATH.write_bytes(
    engine_bytes
)


print()
print(
    "Journal engine : written"
)

print(
    "Engine SHA256  :",
    actual_engine_sha256
)



# IMPORT VERIFIED ENGINE


if str(
    JOURNAL_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            JOURNAL_ROOT
        ),
    )


if (
    "journal_engine"
    in sys.modules
):

    del sys.modules[
        "journal_engine"
    ]


importlib.invalidate_caches()


import journal_engine as qe



# HARD SCIENTIFIC INVARIANT CHECKS


assert (
    qe.PARENT_PROTOCOL_HASH
    ==
    EXPECTED_PROTOCOL_HASH
)


assert (
    qe.PPO_CONFIG[
        "gamma"
    ]
    == 0.99
)


assert (
    qe.PPO_CONFIG[
        "gae_lambda"
    ]
    == 0.95
)


assert (
    qe.PPO_CONFIG[
        "clip_epsilon"
    ]
    == 0.20
)


assert (
    qe.PPO_CONFIG[
        "actor_lr"
    ]
    == 3e-4
)


assert (
    qe.PPO_CONFIG[
        "critic_lr"
    ]
    == 1e-3
)


assert (
    qe.PPO_CONFIG[
        "update_epochs"
    ]
    == 1
)


assert (
    qe.PPO_CONFIG[
        "minibatch_size"
    ]
    == 64
)


assert (
    qe.PPO_CONFIG[
        "entropy_coef"
    ]
    == 0.01
)


assert (
    qe.PPO_CONFIG[
        "value_coef"
    ]
    == 0.50
)


assert (
    qe.PPO_CONFIG[
        "max_grad_norm"
    ]
    == 0.50
)


assert (
    qe.QUANTUM_CONFIG[
        "main_qubits"
    ]
    == 4
)


assert (
    qe.QUANTUM_CONFIG[
        "main_layers"
    ]
    == 3
)


assert (
    qe.QUANTUM_CONFIG[
        "adaptive_shot_schedule"
    ]
    ==
    [
        32,
        64,
        128,
        256,
        512,
        1024,
        2048,
        4096,
    ]
)


assert (
    qe.QUANTUM_CONFIG[
        "delta"
    ]
    == 0.05
)


if (
    qe.ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):

    raise RuntimeError(
        "Journal engine did not select lightning.gpu.\n"
        f"Selected: {qe.ANALYTIC_QDEVICE_NAME}"
    )



# VERIFY MAIN VQC = 24 TRAINABLE QUANTUM PARAMETERS


test_actor = (
    qe.FixedShotQuantumActor(
        state_dim=4,
        action_dim=2,
        shots=32,
        n_qubits=4,
        n_layers=3,
    )
)


quantum_parameter_count = int(
    test_actor
    .quantum_weights
    .numel()
)


if (
    quantum_parameter_count
    != 24
):

    raise RuntimeError(
        "Main VQC parameter-count mismatch.\n"
        f"Expected: 24\n"
        f"Found   : {quantum_parameter_count}"
    )


with torch.no_grad():

    test_probs = (
        test_actor
        .single_forward(
            np.zeros(
                4,
                dtype=np.float32,
            )
        )
    )


if (
    test_probs.shape[0]
    != 2
):

    raise RuntimeError(
        "CartPole action-probability dimension mismatch."
    )


if abs(
    float(
        test_probs.sum()
    )
    - 1.0
) > 1e-6:

    raise RuntimeError(
        "Finite-shot action probabilities "
        "do not sum to 1."
    )


del test_actor

torch.cuda.empty_cache()



# VERIFY QTRUST FOUR-TAIL FINITE-HORIZON ALLOCATION


trace, qtrust_summary = (
    qe.qtrust_finite_horizon_test(

        p_old=np.array(
            [
                0.50,
                0.50,
            ],
            dtype=float,
        ),

        p_new=np.array(
            [
                0.65,
                0.35,
            ],
            dtype=float,
        ),

        action=0,

        advantage=1.0,

        seed=20260920,

        shot_schedule=[
            32,
            64,
            128,
        ],

        delta=0.05,

        clip_epsilon=0.20,
    )
)


if len(
    trace
) < 1:

    raise RuntimeError(
        "QTrust smoke test produced no stages."
    )


if not np.allclose(
    trace[
        "Tail Alpha"
    ].to_numpy(
        dtype=float
    ),
    trace[
        "Stage Alpha"
    ].to_numpy(
        dtype=float
    )
    / 4.0,
):

    raise RuntimeError(
        "QTrust four-tail allocation check failed."
    )


print()
print(
    "QTrust 4-tail  : PASS"
)



# DUAL-T4 ISOLATED PROCESS SMOKE TEST

# Each worker sees exactly one physical GPU as local cuda:0.
# No RL training is performed.


SMOKE_SCRIPT = f"""
import sys
import json
import numpy as np

sys.path.insert(
    0,
    {repr(str(JOURNAL_ROOT))}
)

import journal_engine as qe

if qe.ANALYTIC_QDEVICE_NAME != "lightning.gpu":
    raise RuntimeError(
        "lightning.gpu was not selected"
    )

actor = qe.FixedShotQuantumActor(
    state_dim=4,
    action_dim=2,
    shots=32,
    n_qubits=4,
    n_layers=1,
)

with qe.torch.no_grad():
    probs = actor.single_forward(
        np.array(
            [0.1, -0.1, 0.05, 0.0],
            dtype=np.float32,
        )
    )

payload = {{
    "backend":
        qe.ANALYTIC_QDEVICE_NAME,

    "device":
        str(
            qe.DEVICE
        ),

    "prob_sum":
        float(
            probs.detach()
            .cpu()
            .sum()
        ),

    "parameter_count":
        int(
            actor
            .quantum_weights
            .numel()
        ),
}}

print(
    json.dumps(
        payload
    )
)
"""


smoke_results = []


for physical_gpu in [
    0,
    1,
]:

    worker_env = (
        os.environ.copy()
    )


    worker_env[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        physical_gpu
    )


    worker_env[
        "OMP_NUM_THREADS"
    ] = "1"


    worker_env[
        "MKL_NUM_THREADS"
    ] = "1"


    worker_env[
        "OPENBLAS_NUM_THREADS"
    ] = "1"


    worker_env[
        "NUMEXPR_NUM_THREADS"
    ] = "1"


    worker_env[
        "CUDA_MODULE_LOADING"
    ] = "LAZY"


    completed = subprocess.run(

        [
            sys.executable,
            "-c",
            SMOKE_SCRIPT,
        ],

        env=worker_env,

        capture_output=True,

        text=True,

        timeout=240,
    )


    if (
        completed.returncode
        != 0
    ):

        raise RuntimeError(
            f"\nGPU {physical_gpu} smoke test failed.\n"
            f"\nSTDOUT:\n{completed.stdout}\n"
            f"\nSTDERR:\n{completed.stderr}"
        )


    output_lines = [
        line
        for line
        in completed.stdout.splitlines()
        if line.strip()
    ]


    if not output_lines:

        raise RuntimeError(
            f"GPU {physical_gpu} returned no smoke-test output."
        )


    result = json.loads(
        output_lines[
            -1
        ]
    )


    result[
        "physical_gpu"
    ] = physical_gpu


    smoke_results.append(
        result
    )



# HARD DUAL-GPU VALIDATION


for result in smoke_results:

    if (
        result[
            "backend"
        ]
        != "lightning.gpu"
    ):

        raise RuntimeError(
            "Worker did not use lightning.gpu."
        )


    if abs(
        result[
            "prob_sum"
        ]
        - 1.0
    ) > 1e-6:

        raise RuntimeError(
            "Worker probability sum failed."
        )


    # 1 layer × 4 qubits × 2 trainable angles
    if (
        result[
            "parameter_count"
        ]
        != 8
    ):

        raise RuntimeError(
            "Worker one-layer VQC parameter count failed."
        )



# SAVE SOFTWARE / ENGINE PROVENANCE


current_environment = {

    "python":
        sys.version.split()[0],

    "torch":
        torch.__version__,

    "pennylane":
        qml.__version__,

    "pennylane_lightning_gpu":
        package_version(
            "pennylane-lightning-gpu"
        ),

    "gymnasium":
        gym.__version__,

    "scipy":
        scipy.__version__,

    "gpu_count":
        GPU_COUNT,

    "gpu_names":
        GPU_NAMES,

    "quantum_backend":
        qe.ANALYTIC_QDEVICE_NAME,
}


engine_record = {

    "cell":
        3,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "engine_sha256":
        actual_engine_sha256,

    "engine_path":
        str(
            ENGINE_PATH
        ),

    "main_vqc_trainable_parameters":
        quantum_parameter_count,

    "qtrust_four_tail_rule":
        "PASS",

    "backend":
        qe.ANALYTIC_QDEVICE_NAME,

    "dual_gpu_smoke_test":
        smoke_results,

    "experiment_A_environment":
        experiment_A_environment,

    "journal_environment":
        current_environment,
}


(
    LOGS_DIR
    / "cell03_engine_verification.json"
).write_text(

    json.dumps(
        engine_record,
        indent=2,
    ),

    encoding="utf-8",
)



# PRESERVE EXPERIMENT-A ENVIRONMENT METADATA


if metadata_path.exists():

    reference_directory = (
        JOURNAL_ROOT
        / "experiment_A_frozen"
        / "reference"
    )


    reference_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        metadata_path,
        reference_directory
        / "environment_metadata.json",
    )






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


with open(
    manifest_path,
    "r",
    encoding="utf-8",
) as f:

    manifest = json.load(
        f
    )


manifest[
    "engine"
] = {

    "status":
        "VERIFIED",

    "sha256":
        actual_engine_sha256,

    "backend":
        qe.ANALYTIC_QDEVICE_NAME,

    "main_vqc_parameters":
        quantum_parameter_count,

    "pennylane":
        qml.__version__,

    "pennylane_lightning_gpu":
        package_version(
            "pennylane-lightning-gpu"
        ),

    "gymnasium":
        gym.__version__,
}


manifest[
    "current_completed_cell"
] = 3


with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )






print()
print("=" * 80)
print("JOURNAL ENGINE VERIFIED")
print("=" * 80)

print(
    "Parent protocol hash :",
    EXPECTED_PROTOCOL_HASH
)

print(
    "Journal protocol hash:",
    JOURNAL_PROTOCOL_HASH
)

print(
    "Engine SHA256        :",
    actual_engine_sha256
)

print(
    "PennyLane            :",
    qml.__version__
)

print(
    "Lightning-GPU        :",
    package_version(
        "pennylane-lightning-gpu"
    )
)

print(
    "Gymnasium            :",
    gym.__version__
)

print(
    "Quantum backend      :",
    qe.ANALYTIC_QDEVICE_NAME
)

print(
    "Main VQC parameters  :",
    quantum_parameter_count
)

print(
    "GPU 0 smoke test     : PASS"
)

print(
    "GPU 1 smoke test     : PASS"
)

print(
    "QTrust 4-tail rule   : PASS"
)

print(
    "Training performed   : NO"
)

print("=" * 80)






CELL3_CHECKPOINT = (
    save_journal_checkpoint(
        3,
        "Journal_Engine_Verified",
    )
)


print()
print("=" * 80)
print("CELL 3 COMPLETE")
print("=" * 80)

print(
    "Next: Cell 4 will run Experiment B "
    "(30 frozen ablation runs)."
)

print(
    "Both T4s will be used in parallel."
)

print(
    "Every completed run will be saved atomically "
    "and skipped automatically on resume."
)

In [ ]:
# Experiment B: confidence and rollback ablations

# 30 New Runs:
#   B_NoRollback  × 10 seeds
#   B_Delta_001   × 10 seeds
#   B_Delta_010   × 10 seeds

# Full QTrust delta=0.05 + rollback is already Experiment A.

# Execution:
#   2 physical T4 GPUs
#   2 independent workers / T4
#   4 total workers

# Recovery:
#   - each completed run is written atomically
#   - completed runs are skipped automatically

#   - Cell-4 master ZIP created at the end


from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import shutil
import zipfile
import subprocess
import hashlib

import numpy as np
import pandas as pd



# REQUIRE CELLS 1–3


required = [
    "JOURNAL_ROOT",
    "EXP_B_DIR",
    "LOGS_DIR",
    "RUN_MANIFEST",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1–3 first. Missing: "
        + ", ".join(missing)
    )


print("=" * 88)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 4 / 10")
print("EXPERIMENT B — ABLATIONS")
print("=" * 88)



# EXPERIMENT-B DIRECTORIES


B_RUNS_DIR = (
    EXP_B_DIR
    / "runs"
)

B_UPDATES_DIR = (
    EXP_B_DIR
    / "updates"
)

B_EVAL_DIR = (
    EXP_B_DIR
    / "evaluation"
)

B_LOG_DIR = (
    EXP_B_DIR
    / "worker_logs"
)

B_TASK_DIR = (
    EXP_B_DIR
    / "worker_tasks"
)


for directory in [
    B_RUNS_DIR,
    B_UPDATES_DIR,
    B_EVAL_DIR,
    B_LOG_DIR,
    B_TASK_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )



# EXACT EXPERIMENT-B MANIFEST


B_MANIFEST = (
    RUN_MANIFEST[
        RUN_MANIFEST[
            "experiment"
        ]
        == "B"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(B_MANIFEST) != 30:
    raise RuntimeError(
        f"Experiment B must contain 30 runs. "
        f"Found {len(B_MANIFEST)}."
    )


expected_variants = {
    "B_NoRollback",
    "B_Delta_001",
    "B_Delta_010",
}


if set(
    B_MANIFEST[
        "variant"
    ].unique()
) != expected_variants:

    raise RuntimeError(
        "Experiment-B variant set mismatch."
    )



# COMPLETION VALIDATION


def summary_path_for_run(
    run_id,
):
    return (
        B_RUNS_DIR
        / f"{run_id}.json"
    )


def valid_completed_run(
    run_id,
):
    path = (
        summary_path_for_run(
            run_id
        )
    )

    if not path.exists():
        return False

    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        return (
            payload.get(
                "status"
            )
            == "COMPLETE"
            and
            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH
            and
            payload.get(
                "parent_protocol_hash"
            )
            == EXPECTED_PROTOCOL_HASH
            and
            payload.get(
                "run_id"
            )
            == run_id
        )

    except Exception:
        return False


completed_before = [
    run_id
    for run_id in B_MANIFEST[
        "run_id"
    ]
    if valid_completed_run(
        run_id
    )
]


pending_manifest = (
    B_MANIFEST[
        ~B_MANIFEST[
            "run_id"
        ].isin(
            completed_before
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Completed before Cell 4 :",
    len(
        completed_before
    ),
    "/ 30"
)

print(
    "Pending Experiment-B runs:",
    len(
        pending_manifest
    )
)



# EXPERIMENT-B WORKER SOURCE


WORKER_SOURCE = r'''
import os
import sys
import json
import time
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# Arguments
# ------------------------------------------------------------

if len(sys.argv) != 4:
    raise RuntimeError(
        "Usage: expB_worker.py "
        "<task_json> <journal_root> <protocol_hash>"
    )

TASK_FILE = Path(
    sys.argv[1]
)

JOURNAL_ROOT = Path(
    sys.argv[2]
)

JOURNAL_PROTOCOL_HASH = (
    sys.argv[3]
)

PARENT_PROTOCOL_HASH = (
    "3cf4bcb18e1111b3"
)


# ------------------------------------------------------------
# CPU oversubscription guard
# ------------------------------------------------------------

os.environ[
    "OMP_NUM_THREADS"
] = "1"

os.environ[
    "MKL_NUM_THREADS"
] = "1"

os.environ[
    "OPENBLAS_NUM_THREADS"
] = "1"

os.environ[
    "NUMEXPR_NUM_THREADS"
] = "1"

os.environ[
    "CUDA_MODULE_LOADING"
] = "LAZY"

torch.set_num_threads(
    1
)

try:
    torch.set_num_interop_threads(
        1
    )
except Exception:
    pass


# ------------------------------------------------------------
# Import exact verified journal engine
# ------------------------------------------------------------

sys.path.insert(
    0,
    str(
        JOURNAL_ROOT
    ),
)

import journal_engine as qe


if (
    qe.PARENT_PROTOCOL_HASH
    != PARENT_PROTOCOL_HASH
):
    raise RuntimeError(
        "Parent protocol hash mismatch."
    )


if (
    qe.ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):
    raise RuntimeError(
        "Worker is not using lightning.gpu."
    )


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

EXP_B_DIR = (
    JOURNAL_ROOT
    / "experiment_B_ablations"
)

RUNS_DIR = (
    EXP_B_DIR
    / "runs"
)

UPDATES_DIR = (
    EXP_B_DIR
    / "updates"
)

EVAL_DIR = (
    EXP_B_DIR
    / "evaluation"
)


for directory in [
    RUNS_DIR,
    UPDATES_DIR,
    EVAL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Atomic helpers
# ------------------------------------------------------------

def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    temporary = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary,
        path,
    )


def atomic_csv(
    path,
    dataframe,
):
    path = Path(
        path
    )

    temporary = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    dataframe.to_csv(
        temporary,
        index=False,
    )

    os.replace(
        temporary,
        path,
    )


def valid_existing_summary(
    run_id,
):
    path = (
        RUNS_DIR
        / f"{run_id}.json"
    )

    if not path.exists():
        return False

    try:
        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        return (
            payload.get(
                "status"
            )
            == "COMPLETE"
            and
            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH
            and
            payload.get(
                "parent_protocol_hash"
            )
            == PARENT_PROTOCOL_HASH
        )

    except Exception:
        return False


# ------------------------------------------------------------
# Load worker task list
# ------------------------------------------------------------

tasks = json.loads(
    TASK_FILE.read_text(
        encoding="utf-8"
    )
)


print(
    f"[worker] visible CUDA devices="
    f"{os.environ.get('CUDA_VISIBLE_DEVICES')}"
)

print(
    f"[worker] tasks={len(tasks)}"
)


# ------------------------------------------------------------
# Execute assigned runs sequentially
# ------------------------------------------------------------

for task_number, task in enumerate(
    tasks,
    start=1,
):

    run_id = (
        task[
            "run_id"
        ]
    )


    if valid_existing_summary(
        run_id
    ):
        print(
            f"[skip] {run_id}"
        )
        continue


    run_start = (
        time.perf_counter()
    )


    try:

        seed = int(
            task[
                "seed"
            ]
        )

        delta = float(
            task[
                "delta"
            ]
        )

        rollback = bool(
            task[
                "rollback"
            ]
        )

        base_shots = int(
            task[
                "base_shots"
            ]
        )

        layers = int(
            task[
                "layers"
            ]
        )

        qubits = int(
            task[
                "qubits"
            ]
        )


        print(
            f"[{task_number}/{len(tasks)}] "
            f"START {run_id}"
        )


        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        (
            model,
            update_df,
            training_stats,
        ) = qe.train_qtrust_journal(

            env_name=
                "CartPole-v1",

            base_shots=
                base_shots,

            total_steps=
                2048,

            rollout_steps=
                32,

            seed=
                seed,

            update_epochs=
                1,

            n_qubits=
                qubits,

            n_layers=
                layers,

            delta=
                delta,

            rollback_on_unresolved=
                rollback,

            shot_schedule=[
                32,
                64,
                128,
                256,
                512,
                1024,
                2048,
                4096,
            ],
        )


        # ----------------------------------------------------
        # Evaluate exactly 30 episodes
        # ----------------------------------------------------

        evaluation_seeds = (
            qe.make_final_eval_seeds(
                training_seed=
                    seed,
                n_episodes=
                    30,
            )
        )


        (
            evaluation_df,
            evaluation_stats,
        ) = (
            qe.evaluate_finite_shot_policy_journal(

                model=
                    model,

                env_name=
                    "CartPole-v1",

                seeds=
                    evaluation_seeds,

                deterministic=
                    True,
            )
        )


        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------

        if len(
            update_df
        ):

            mean_unresolved = float(
                update_df[
                    "unresolved_fraction"
                ].mean()
            )

            resolved_accuracy_values = (
                update_df[
                    "classification_accuracy"
                ]
                .dropna()
            )

            classification_accuracy = (
                float(
                    resolved_accuracy_values.mean()
                )
                if len(
                    resolved_accuracy_values
                )
                else None
            )

            mean_qtrust_shots = float(
                update_df[
                    "mean_qtrust_shots"
                ].mean()
            )

            mean_approx_kl = float(
                update_df[
                    "approx_kl"
                ].mean()
            )

            mean_clip_fraction = float(
                update_df[
                    "clip_fraction"
                ].mean()
            )

            mean_actor_loss = float(
                update_df[
                    "actor_loss"
                ].mean()
            )

            mean_critic_loss = float(
                update_df[
                    "critic_loss"
                ].mean()
            )

        else:

            mean_unresolved = None
            classification_accuracy = None
            mean_qtrust_shots = None
            mean_approx_kl = None
            mean_clip_fraction = None
            mean_actor_loss = None
            mean_critic_loss = None


        # ----------------------------------------------------
        # Write large artifacts FIRST
        # ----------------------------------------------------

        update_path = (
            UPDATES_DIR
            / f"{run_id}.csv"
        )

        evaluation_path = (
            EVAL_DIR
            / f"{run_id}.csv"
        )


        atomic_csv(
            update_path,
            update_df,
        )

        atomic_csv(
            evaluation_path,
            evaluation_df,
        )


        # ----------------------------------------------------
        # Final summary marker is written LAST.
        #
        # Therefore the run is considered complete only after
        # all associated artifacts are safely present.
        # ----------------------------------------------------

        wall_runtime = float(
            time.perf_counter()
            - run_start
        )


        summary = {

            "status":
                "COMPLETE",

            "experiment":
                "B",

            "run_id":
                run_id,

            "variant":
                task[
                    "variant"
                ],

            "environment":
                "CartPole-v1",

            "method":
                "QTrust-PPO-128",

            "seed":
                seed,

            "parent_protocol_hash":
                PARENT_PROTOCOL_HASH,

            "journal_protocol_hash":
                JOURNAL_PROTOCOL_HASH,

            "delta":
                delta,

            "rollback_on_unresolved":
                rollback,

            "base_shots":
                base_shots,

            "qubits":
                qubits,

            "layers":
                layers,

            "training_steps":
                2048,

            "rollout_steps":
                32,

            "evaluation_episodes":
                30,

            "mean_eval_reward":
                float(
                    evaluation_stats[
                        "mean_eval_reward"
                    ]
                ),

            "std_eval_reward":
                float(
                    evaluation_stats[
                        "std_eval_reward"
                    ]
                ),

            "median_eval_reward":
                float(
                    evaluation_stats[
                        "median_eval_reward"
                    ]
                ),

            "eval_ci95_low":
                float(
                    evaluation_stats[
                        "eval_ci95_low"
                    ]
                ),

            "eval_ci95_high":
                float(
                    evaluation_stats[
                        "eval_ci95_high"
                    ]
                ),

            "mean_eval_steps":
                float(
                    evaluation_stats[
                        "mean_eval_steps"
                    ]
                ),

            "accepted_updates":
                int(
                    training_stats[
                        "accepted_updates"
                    ]
                ),

            "rolled_back_updates":
                int(
                    training_stats[
                        "rolled_back_updates"
                    ]
                ),

            "mean_unresolved_fraction":
                mean_unresolved,

            "classification_accuracy":
                classification_accuracy,

            "mean_adaptive_shots":
                mean_qtrust_shots,

            "mean_approx_kl":
                mean_approx_kl,

            "mean_clip_fraction":
                mean_clip_fraction,

            "mean_actor_loss":
                mean_actor_loss,

            "mean_critic_loss":
                mean_critic_loss,

            "action_shots":
                int(
                    training_stats[
                        "action_shots"
                    ]
                ),

            "gradient_shots":
                int(
                    training_stats[
                        "gradient_shots"
                    ]
                ),

            "certification_shots":
                int(
                    training_stats[
                        "certification_shots"
                    ]
                ),

            "training_quantum_shots":
                int(
                    training_stats[
                        "total_quantum_shots"
                    ]
                ),

            "evaluation_quantum_shots":
                int(
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "overall_quantum_shots":
                int(
                    training_stats[
                        "total_quantum_shots"
                    ]
                    +
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "training_runtime_seconds":
                float(
                    training_stats[
                        "elapsed_seconds"
                    ]
                ),

            "evaluation_runtime_seconds":
                float(
                    evaluation_stats[
                        "evaluation_runtime_seconds"
                    ]
                ),

            "wall_runtime_seconds":
                wall_runtime,

            "updates_file":
                str(
                    update_path
                    .relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "evaluation_file":
                str(
                    evaluation_path
                    .relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "completed_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        summary_path = (
            RUNS_DIR
            / f"{run_id}.json"
        )


        atomic_json(
            summary_path,
            summary,
        )


        print(
            f"[complete] {run_id} | "
            f"reward="
            f"{summary['mean_eval_reward']:.3f} | "
            f"unresolved="
            f"{summary['mean_unresolved_fraction']:.4f} | "
            f"wall="
            f"{wall_runtime / 60:.1f} min"
        )


        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    except Exception as exc:

        failure_payload = {

            "status":
                "FAILED",

            "run_id":
                run_id,

            "error_type":
                type(
                    exc
                ).__name__,

            "error":
                str(
                    exc
                ),

            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        failure_path = (
            RUNS_DIR
            / f"{run_id}.failed.json"
        )


        atomic_json(
            failure_path,
            failure_payload,
        )


        traceback.print_exc()


        # Fail this worker so the parent cell does not
        # accidentally report success.
        raise


print(
    "[worker] ALL ASSIGNED TASKS COMPLETE"
)
'''


WORKER_PATH = (
    EXP_B_DIR
    / "expB_worker.py"
)


WORKER_PATH.write_text(
    WORKER_SOURCE,
    encoding="utf-8",
)



# BUILD FOUR BALANCED WORKER QUEUES

# Mapping:
#   Worker 0 -> physical GPU 0
#   Worker 1 -> physical GPU 1
#   Worker 2 -> physical GPU 0
#   Worker 3 -> physical GPU 1

# This uses both T4s while allowing two independent
# experiment streams per device.


WORKER_GPU_MAP = [
    0,
    1,
    0,
    1,
]


TOTAL_WORKERS = len(
    WORKER_GPU_MAP
)


tasks = (
    pending_manifest
    .to_dict(
        orient="records"
    )
)


# Interleave variants/seeds rather than placing a whole
# variant on one worker.
worker_queues = [
    []
    for _ in range(
        TOTAL_WORKERS
    )
]


for index, task in enumerate(
    tasks
):

    # JSON cannot directly serialize NaN safely.
    cleaned_task = {}

    for key, value in task.items():

        if pd.isna(
            value
        ):
            cleaned_task[
                key
            ] = None

        elif isinstance(
            value,
            np.generic,
        ):
            cleaned_task[
                key
            ] = value.item()

        else:
            cleaned_task[
                key
            ] = value


    worker_queues[
        index
        % TOTAL_WORKERS
    ].append(
        cleaned_task
    )


task_files = []


for worker_id, queue in enumerate(
    worker_queues
):

    path = (
        B_TASK_DIR
        / f"worker_{worker_id}_tasks.json"
    )

    path.write_text(
        json.dumps(
            queue,
            indent=2,
        ),
        encoding="utf-8",
    )

    task_files.append(
        path
    )


print()
print(
    "Worker assignment:"
)

for worker_id, (
    gpu_id,
    queue,
) in enumerate(
    zip(
        WORKER_GPU_MAP,
        worker_queues,
    )
):

    print(
        f"  Worker {worker_id} "
        f"-> T4 GPU {gpu_id} "
        f"-> {len(queue)} runs"
    )





# Updated by the parent process while workers are running.
# This ZIP contains:
#   summaries
#   update traces
#   evaluation traces
#   manifest
#   protocol
#   engine

# It does NOT contain earlier checkpoint ZIPs.


B_RECOVERY_ZIP = (
    Path("/kaggle/working")
    / "QTrust_Journal_Experiment_B_LATEST.zip"
)


def save_B_recovery_zip():

    temporary = (
        B_RECOVERY_ZIP
        .with_suffix(
            ".zip.tmp"
        )
    )


    if temporary.exists():
        temporary.unlink()


    with zipfile.ZipFile(
        temporary,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
        compresslevel=1,
    ) as zf:

        # Experiment B
        if EXP_B_DIR.exists():

            for file in sorted(
                EXP_B_DIR.rglob("*")
            ):

                if not file.is_file():
                    continue

                if file.suffix == ".tmp":
                    continue

                relative = (
                    file.relative_to(
                        JOURNAL_ROOT
                    )
                )

                zf.write(
                    file,
                    arcname=str(
                        Path(
                            JOURNAL_ROOT.name
                        )
                        / relative
                    ),
                )


        # Core reproducibility files
        for core_name in [
            "journal_manifest.json",
            "journal_extension_protocol.json",
            "journal_run_manifest.csv",
            "journal_engine.py",
            "JOURNAL_EXPERIMENT_PLAN.txt",
        ]:

            core_file = (
                JOURNAL_ROOT
                / core_name
            )

            if core_file.exists():

                zf.write(
                    core_file,
                    arcname=str(
                        Path(
                            JOURNAL_ROOT.name
                        )
                        / core_name
                    ),
                )


    with zipfile.ZipFile(
        temporary,
        "r",
    ) as zf:

        bad = zf.testzip()


    if bad is not None:

        temporary.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Experiment-B recovery ZIP "
            f"failed verification: {bad}"
        )


    os.replace(
        temporary,
        B_RECOVERY_ZIP,
    )



# START WORKERS


processes = []


if len(
    pending_manifest
) > 0:

    print()
    print("=" * 88)
    print("STARTING EXPERIMENT B")
    print("=" * 88)


    for worker_id, (
        physical_gpu,
        task_file,
    ) in enumerate(
        zip(
            WORKER_GPU_MAP,
            task_files,
        )
    ):

        if len(
            worker_queues[
                worker_id
            ]
        ) == 0:
            continue


        worker_env = (
            os.environ.copy()
        )


        worker_env[
            "CUDA_VISIBLE_DEVICES"
        ] = str(
            physical_gpu
        )


        worker_env[
            "OMP_NUM_THREADS"
        ] = "1"


        worker_env[
            "MKL_NUM_THREADS"
        ] = "1"


        worker_env[
            "OPENBLAS_NUM_THREADS"
        ] = "1"


        worker_env[
            "NUMEXPR_NUM_THREADS"
        ] = "1"


        worker_env[
            "CUDA_MODULE_LOADING"
        ] = "LAZY"


        log_path = (
            B_LOG_DIR
            / f"worker_{worker_id}.log"
        )


        log_handle = open(
            log_path,
            "w",
            buffering=1,
            encoding="utf-8",
        )


        process = subprocess.Popen(

            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    task_file
                ),
                str(
                    JOURNAL_ROOT
                ),
                JOURNAL_PROTOCOL_HASH,
            ],

            env=
                worker_env,

            stdout=
                log_handle,

            stderr=
                subprocess.STDOUT,

            text=True,
        )


        processes.append(
            {
                "worker_id":
                    worker_id,

                "gpu":
                    physical_gpu,

                "process":
                    process,

                "log_handle":
                    log_handle,

                "log_path":
                    log_path,
            }
        )



# MONITOR PROGRESS

# Every time the number of completed runs increases:
#   - print progress



def get_completed_B():

    completed = []

    for run_id in B_MANIFEST[
        "run_id"
    ]:

        if valid_completed_run(
            run_id
        ):
            completed.append(
                run_id
            )

    return completed


last_completed_count = (
    len(
        completed_before
    )
)


last_zip_count = (
    -1
)


if last_completed_count > 0:

    save_B_recovery_zip()

    last_zip_count = (
        last_completed_count
    )


while any(
    item[
        "process"
    ].poll()
    is None
    for item in processes
):

    completed_now = (
        get_completed_B()
    )

    count_now = len(
        completed_now
    )


    if (
        count_now
        != last_completed_count
    ):

        print(
            f"[progress] Experiment B: "
            f"{count_now}/30 complete "
            f"({100.0 * count_now / 30:.1f}%)"
        )


        last_completed_count = (
            count_now
        )


        
        # has completed.
        if (
            count_now
            != last_zip_count
        ):

            save_B_recovery_zip()

            last_zip_count = (
                count_now
            )


    time.sleep(
        20
    )


# Close log handles.
for item in processes:

    item[
        "log_handle"
    ].close()



# VERIFY WORKER EXIT STATUS


failed_workers = []


for item in processes:

    return_code = (
        item[
            "process"
        ].returncode
    )


    if return_code != 0:

        failed_workers.append(
            item
        )


if failed_workers:

    print()
    print("=" * 88)
    print("ONE OR MORE WORKERS FAILED")
    print("=" * 88)


    for item in failed_workers:

        print(
            f"\nWorker {item['worker_id']} "
            f"(GPU {item['gpu']})"
        )

        print(
            "Log:",
            item[
                "log_path"
            ]
        )


        try:

            tail = (
                item[
                    "log_path"
                ]
                .read_text(
                    encoding="utf-8"
                )
                .splitlines()[
                    -40:
                ]
            )

            print(
                "\n".join(
                    tail
                )
            )

        except Exception:
            pass


    # Preserve everything completed so far.
    save_B_recovery_zip()


    raise RuntimeError(
        "Experiment B stopped because at least "
        "one worker failed. Completed runs are saved. "
        "Rerunning Cell 4 will automatically skip them."
    )



# HARD COMPLETION CHECK — 30 / 30


completed_after = (
    get_completed_B()
)


if len(
    completed_after
) != 30:

    save_B_recovery_zip()

    raise RuntimeError(
        f"Experiment B expected 30 COMPLETE runs, "
        f"found {len(completed_after)}."
    )



# LOAD ALL 30 SUMMARIES


summary_rows = []


for run_id in B_MANIFEST[
    "run_id"
]:

    path = (
        summary_path_for_run(
            run_id
        )
    )

    payload = json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )

    summary_rows.append(
        payload
    )


B_RESULTS = pd.DataFrame(
    summary_rows
)



# HARD SCIENTIFIC VALIDATION


if len(
    B_RESULTS
) != 30:

    raise RuntimeError(
        "Experiment-B result count != 30."
    )


if not B_RESULTS[
    "status"
].eq(
    "COMPLETE"
).all():

    raise RuntimeError(
        "Experiment B contains incomplete runs."
    )


if not B_RESULTS[
    "journal_protocol_hash"
].eq(
    JOURNAL_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Journal protocol hash mismatch in Experiment B."
    )


if not B_RESULTS[
    "parent_protocol_hash"
].eq(
    EXPECTED_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Parent protocol hash mismatch in Experiment B."
    )


counts = (
    B_RESULTS
    .groupby(
        "variant"
    )
    .size()
)


if not (
    len(
        counts
    )
    == 3
    and
    counts.eq(
        10
    ).all()
):

    raise RuntimeError(
        "Every Experiment-B variant must contain "
        "exactly 10 independent seeds."
    )


# Exact seed check.
for variant in sorted(
    expected_variants
):

    seeds = sorted(
        B_RESULTS[
            B_RESULTS[
                "variant"
            ]
            == variant
        ][
            "seed"
        ]
        .astype(int)
        .tolist()
    )

    if seeds != [
        11,
        23,
        37,
        53,
        71,
        89,
        107,
        131,
        157,
        181,
    ]:

        raise RuntimeError(
            f"Seed mismatch for {variant}: {seeds}"
        )



# SAVE MASTER EXPERIMENT-B TABLE


B_RESULTS_PATH = (
    EXP_B_DIR
    / "experiment_B_all_runs.csv"
)


B_RESULTS.to_csv(
    B_RESULTS_PATH,
    index=False,
)



# CREATE A SMALL SUMMARY TABLE

# Descriptive only.



B_SUMMARY = (
    B_RESULTS
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),

        Mean_Wall_Runtime_s=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


B_SUMMARY_PATH = (
    EXP_B_DIR
    / "experiment_B_summary.csv"
)


B_SUMMARY.to_csv(
    B_SUMMARY_PATH,
    index=False,
)



# UPDATE MASTER RUN MANIFEST


RUN_MANIFEST.loc[
    RUN_MANIFEST[
        "experiment"
    ]
    == "B",
    "status",
] = "COMPLETE"


RUN_MANIFEST.to_csv(
    JOURNAL_ROOT
    / "journal_run_manifest.csv",
    index=False,
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "experiment_B_ablations"
] = {

    "status":
        "COMPLETE",

    "completed_runs":
        30,

    "variants":
        sorted(
            expected_variants
        ),

    "seeds_per_variant":
        10,

    "result_file":
        str(
            B_RESULTS_PATH
            .relative_to(
                JOURNAL_ROOT
            )
        ),

    "summary_file":
        str(
            B_SUMMARY_PATH
            .relative_to(
                JOURNAL_ROOT
            )
        ),
}


manifest[
    "current_completed_cell"
] = 4


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)






save_B_recovery_zip()



# CELL-4 PROVENANCE RECORD


cell4_record = {

    "cell":
        4,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        "B",

    "completed_runs":
        30,

    "variants":
        sorted(
            expected_variants
        ),

    "seeds_per_variant":
        10,

    "workers":
        TOTAL_WORKERS,

    "physical_gpus":
        2,

    "worker_gpu_map":
        WORKER_GPU_MAP,

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,
}


(
    LOGS_DIR
    / "cell04_experiment_B.json"
).write_text(
    json.dumps(
        cell4_record,
        indent=2,
    ),
    encoding="utf-8",
)



# REPORT RESULTS


print()
print("=" * 88)
print("EXPERIMENT B COMPLETE")
print("=" * 88)

print(
    "Completed runs :",
    len(
        B_RESULTS
    ),
    "/ 30"
)

print(
    "Variants       : 3"
)

print(
    "Seeds/variant  : 10"
)

print(
    "GPUs           : 2 × Tesla T4"
)

print(
    "Workers        :",
    TOTAL_WORKERS
)

print(
    "Journal hash   :",
    JOURNAL_PROTOCOL_HASH
)

print()
print(
    B_SUMMARY.to_string(
        index=False
    )
)

print()
print(
    "Recovery ZIP:",
    B_RECOVERY_ZIP
)

print("=" * 88)






CELL4_CHECKPOINT = (
    save_journal_checkpoint(
        4,
        "Experiment_B_Ablations_COMPLETE",
    )
)


print()
print("=" * 88)
print("CELL 4 COMPLETE")
print("=" * 88)

print(
    "Experiment B is now frozen."
)

print(
    "Next: Cell 5 will run Experiment C "
    "(60 noise-robustness runs)."
)

print(
    "Do not modify Experiment-B settings "
    "after seeing these results."
)

In [ ]:
# Verify rollback logic in the generated noise engine.
from pathlib import Path

p = Path(
    "/kaggle/working/QTrust_PPO_Journal/"
    "journal_noise_engine.py"
)

lines = p.read_text(
    encoding="utf-8"
).splitlines()

for n in range(
    1658,
    1680,
):
    print(
        f"{n+1:04d}: {lines[n]}"
    )

In [ ]:
# Experiment C: output-noise robustness

# 60 New Runs:

#   C_Readout_002
#       readout bit-flip probability = 0.02

#   C_Depolarizing_0005
#       depolarizing probability = 0.005

# ×

#   Fixed-QPPO-128
#   Fixed-QPPO-256
#   QTrust-PPO-128

# × 10 seeds

# Execution:
#   2 × Tesla T4
#   4 independent workers

# Recovery:
#   Atomic save after every completed run
#   Automatic resume



from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import zipfile
import subprocess
import hashlib
import importlib
import py_compile

import numpy as np
import pandas as pd



# REQUIRE CELLS 1–4


required = [
    "JOURNAL_ROOT",
    "EXP_C_DIR",
    "LOGS_DIR",
    "RUN_MANIFEST",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1–4 first. Missing: "
        + ", ".join(missing)
    )


print("=" * 90)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 5 / 10")
print("EXPERIMENT C — NOISE ROBUSTNESS")
print("=" * 90)



# DIRECTORIES


C_RUNS_DIR = EXP_C_DIR / "runs"
C_UPDATES_DIR = EXP_C_DIR / "updates"
C_EVAL_DIR = EXP_C_DIR / "evaluation"
C_LOG_DIR = EXP_C_DIR / "worker_logs"
C_TASK_DIR = EXP_C_DIR / "worker_tasks"


for directory in [
    C_RUNS_DIR,
    C_UPDATES_DIR,
    C_EVAL_DIR,
    C_LOG_DIR,
    C_TASK_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )






C_MANIFEST = (
    RUN_MANIFEST[
        RUN_MANIFEST["experiment"] == "C"
    ]
    .copy()
    .reset_index(drop=True)
)


if len(C_MANIFEST) != 60:
    raise RuntimeError(
        f"Experiment C must contain 60 runs. "
        f"Found {len(C_MANIFEST)}."
    )


EXPECTED_NOISE_VARIANTS = {
    "C_Readout_002",
    "C_Depolarizing_0005",
}

EXPECTED_METHODS_C = {
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
}


if set(
    C_MANIFEST["variant"].unique()
) != EXPECTED_NOISE_VARIANTS:
    raise RuntimeError(
        "Experiment-C noise variant mismatch."
    )


if set(
    C_MANIFEST["method"].unique()
) != EXPECTED_METHODS_C:
    raise RuntimeError(
        "Experiment-C method mismatch."
    )



# FREEZE NOISE IMPLEMENTATION ADDENDUM


NOISE_IMPLEMENTATION = {

    "experiment":
        "C",

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "created_before_results":
        True,

    "backend":
        "lightning.gpu",

    "readout_noise": {

        "variant":
            "C_Readout_002",

        "probability":
            0.02,

        "model":
            (
                "Independent symmetric bit-flip readout "
                "channel on each of the two measured "
                "action-readout qubits."
            ),

        "simulation":
            (
                "Unitary Stinespring dilation using "
                "one environment ancilla per measured qubit."
            ),
    },

    "depolarizing_noise": {

        "variant":
            "C_Depolarizing_0005",

        "probability":
            0.005,

        "model":
            (
                "Independent single-qubit depolarizing "
                "channel on each of the two measured "
                "action-readout qubits at circuit output."
            ),

        "channel":
            (
                "E(rho)=(1-p)rho + "
                "(p/3)(XrhoX + YrhoY + ZrhoZ)"
            ),

        "simulation":
            (
                "Unitary Stinespring dilation using "
                "two environment ancillas per measured qubit."
            ),
    },

    "scope":
        (
            "Output/readout noise robustness. "
            "Not a claim of full hardware gate-level noise."
        ),

    "finite_shot_measurement":
        True,

    "parameter_shift_gradients":
        True,

    "analytic_born_probabilities":
        (
            "Used only as hidden simulation/audit truth "
            "for QTrust finite-shot certification. "
            "They are not exposed to the controller."
        ),
}


NOISE_IMPLEMENTATION_PATH = (
    EXP_C_DIR
    / "noise_implementation_addendum.json"
)


NOISE_IMPLEMENTATION_PATH.write_text(
    json.dumps(
        NOISE_IMPLEMENTATION,
        indent=2,
    ),
    encoding="utf-8",
)


NOISE_IMPLEMENTATION_HASH = (
    hashlib.sha256(
        json.dumps(
            NOISE_IMPLEMENTATION,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    )
    .hexdigest()[:16]
)


print(
    "Noise implementation hash:",
    NOISE_IMPLEMENTATION_HASH
)






NOISE_ENGINE_SOURCE = r'''
import os
import copy
import time
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import pennylane as qml

from torch.distributions import Categorical

import journal_engine as qe


# ============================================================
# BASE VQC
# ============================================================

def _base_vqc(
    features,
    weights,
    input_dim,
    n_qubits,
    n_layers,
):

    for qubit in range(n_qubits):

        qml.Hadamard(
            wires=qubit
        )


    for layer in range(n_layers):

        for qubit in range(n_qubits):

            feature_index = (
                layer * n_qubits + qubit
            ) % input_dim


            angle = features[
                feature_index
            ]


            qml.RY(
                0.5 * angle,
                wires=qubit,
            )


            qml.RZ(
                0.25 * angle,
                wires=qubit,
            )


        for qubit in range(
            n_qubits - 1
        ):

            qml.CNOT(
                wires=[
                    qubit,
                    qubit + 1,
                ]
            )


        qml.CNOT(
            wires=[
                n_qubits - 1,
                0,
            ]
        )


        for qubit in range(
            n_qubits
        ):

            qml.RY(
                weights[
                    layer,
                    qubit,
                    0,
                ],
                wires=qubit,
            )


            qml.RZ(
                weights[
                    layer,
                    qubit,
                    1,
                ],
                wires=qubit,
            )


# ============================================================
# READOUT BIT-FLIP DILATION
# ============================================================

def _apply_readout_dilation(
    n_qubits,
    error_probability,
):

    p = float(
        error_probability
    )


    if p <= 0.0:
        return


    angle = (
        2.0
        * math.asin(
            math.sqrt(p)
        )
    )


    for target in [
        0,
        1,
    ]:

        ancilla = (
            n_qubits
            + target
        )


        qml.RY(
            angle,
            wires=ancilla,
        )


        qml.CNOT(
            wires=[
                ancilla,
                target,
            ]
        )


# ============================================================
# CONTROLLED PAULI HELPERS
# ============================================================

def _controlled_X_01(
    control_a,
    control_b,
    target,
):

    qml.PauliX(
        wires=control_a
    )


    qml.Toffoli(
        wires=[
            control_a,
            control_b,
            target,
        ]
    )


    qml.PauliX(
        wires=control_a
    )


def _controlled_Y_10(
    control_a,
    control_b,
    target,
):

    qml.PauliX(
        wires=control_b
    )


    qml.adjoint(
        qml.S
    )(
        wires=target
    )


    qml.Toffoli(
        wires=[
            control_a,
            control_b,
            target,
        ]
    )


    qml.S(
        wires=target
    )


    qml.PauliX(
        wires=control_b
    )


def _controlled_Z_11(
    control_a,
    control_b,
    target,
):

    qml.Hadamard(
        wires=target
    )


    qml.Toffoli(
        wires=[
            control_a,
            control_b,
            target,
        ]
    )


    qml.Hadamard(
        wires=target
    )


# ============================================================
# DEPOLARIZING DILATION
# ============================================================

def _apply_single_qubit_depolarizing_dilation(
    target,
    ancilla_a,
    ancilla_b,
    probability,
):

    p = float(
        probability
    )


    if p <= 0.0:
        return


    amplitudes = np.array(
        [
            math.sqrt(
                1.0 - p
            ),

            math.sqrt(
                p / 3.0
            ),

            math.sqrt(
                p / 3.0
            ),

            math.sqrt(
                p / 3.0
            ),
        ],
        dtype=np.float64,
    )


    qml.StatePrep(
        amplitudes,
        wires=[
            ancilla_a,
            ancilla_b,
        ],
    )


    _controlled_X_01(
        ancilla_a,
        ancilla_b,
        target,
    )


    _controlled_Y_10(
        ancilla_a,
        ancilla_b,
        target,
    )


    _controlled_Z_11(
        ancilla_a,
        ancilla_b,
        target,
    )


def _apply_depolarizing_dilation(
    n_qubits,
    probability,
):

    p = float(
        probability
    )


    if p <= 0.0:
        return


    for target in [
        0,
        1,
    ]:

        ancilla_a = (
            n_qubits
            + 2 * target
        )


        ancilla_b = (
            ancilla_a + 1
        )


        _apply_single_qubit_depolarizing_dilation(

            target=
                target,

            ancilla_a=
                ancilla_a,

            ancilla_b=
                ancilla_b,

            probability=
                p,
        )


# ============================================================
# NOISY QNODE
# ============================================================

def build_noisy_quantum_policy(
    input_dim,
    n_qubits,
    n_layers,
    shots,
    noise_type,
    readout_error=0.0,
    depolarizing_probability=0.0,
):

    if noise_type == "readout":

        total_wires = (
            n_qubits + 2
        )


    elif noise_type == "depolarizing":

        total_wires = (
            n_qubits + 4
        )


    else:

        raise ValueError(
            "Unsupported noise type: "
            f"{noise_type}"
        )


    dev = qml.device(
        "lightning.gpu",
        wires=total_wires,
        shots=shots,
    )


    @qml.qnode(
        dev,
        interface="torch",
        diff_method="parameter-shift",
    )
    def noisy_policy(
        features,
        weights,
    ):

        _base_vqc(

            features=
                features,

            weights=
                weights,

            input_dim=
                input_dim,

            n_qubits=
                n_qubits,

            n_layers=
                n_layers,
        )


        if noise_type == "readout":

            _apply_readout_dilation(

                n_qubits=
                    n_qubits,

                error_probability=
                    readout_error,
            )


        elif noise_type == "depolarizing":

            _apply_depolarizing_dilation(

                n_qubits=
                    n_qubits,

                probability=
                    depolarizing_probability,
            )


        return qml.probs(
            wires=[
                0,
                1,
            ]
        )


    return noisy_policy


# ============================================================
# NOISY QUANTUM ACTOR
# ============================================================

class NoisyQuantumActor(
    nn.Module
):

    def __init__(
        self,
        state_dim,
        action_dim,
        shots,
        n_qubits=4,
        n_layers=3,
        noise_type="readout",
        readout_error=0.0,
        depolarizing_probability=0.0,
    ):

        super().__init__()


        self.state_dim = int(
            state_dim
        )


        self.action_dim = int(
            action_dim
        )


        self.shots = (
            None
            if shots is None
            else int(shots)
        )


        self.n_qubits = int(
            n_qubits
        )


        self.n_layers = int(
            n_layers
        )


        self.noise_type = str(
            noise_type
        )


        self.readout_error = float(
            readout_error
        )


        self.depolarizing_probability = float(
            depolarizing_probability
        )


        self.qnode = (
            build_noisy_quantum_policy(

                input_dim=
                    self.state_dim,

                n_qubits=
                    self.n_qubits,

                n_layers=
                    self.n_layers,

                shots=
                    self.shots,

                noise_type=
                    self.noise_type,

                readout_error=
                    self.readout_error,

                depolarizing_probability=
                    self.depolarizing_probability,
            )
        )


        self.quantum_weights = (
            nn.Parameter(
                0.05
                * torch.randn(
                    self.n_layers,
                    self.n_qubits,
                    2,
                    dtype=torch.float64,
                )
            )
        )


    def encode_state(
        self,
        state,
    ):

        if not torch.is_tensor(
            state
        ):

            state = torch.tensor(
                state,
                dtype=torch.float64,
            )


        state = state.to(
            qe.QUANTUM_TORCH_DEVICE,
            dtype=torch.float64,
        )


        state = torch.nan_to_num(
            state,
            nan=0.0,
            posinf=10.0,
            neginf=-10.0,
        )


        return (
            torch.pi
            * torch.tanh(
                state
            )
        )


    def single_forward(
        self,
        state,
    ):

        encoded = (
            self.encode_state(
                state
            )
        )


        basis_probs = (
            self.qnode(
                encoded,
                self.quantum_weights,
            )
        )


        return (
            qe.balanced_action_probabilities(
                basis_probs,
                self.action_dim,
            )
        )


    def forward(
        self,
        states,
    ):

        if not torch.is_tensor(
            states
        ):

            states = torch.tensor(
                states,
                dtype=torch.float64,
            )


        if states.ndim == 1:

            return self.single_forward(
                states
            )


        return torch.stack(
            [
                self.single_forward(
                    state
                )
                for state
                in states
            ],
            dim=0,
        )


    @torch.no_grad()
    def act(
        self,
        state,
    ):

        probs = self.single_forward(
            state
        )


        dist = Categorical(
            probs=probs
        )


        action = (
            dist.sample()
        )


        log_prob = (
            dist.log_prob(
                action
            )
        )


        return (

            int(
                action.item()
            ),

            float(
                log_prob.item()
            ),

            probs.detach()
            .cpu()
            .numpy(),
        )


    def evaluate_actions(
        self,
        states,
        actions,
    ):

        probs = self.forward(
            states
        )


        actions = actions.to(
            probs.device,
            dtype=torch.long,
        )


        dist = Categorical(
            probs=probs
        )


        return (

            dist.log_prob(
                actions
            ),

            dist.entropy(),

            probs,
        )


# ============================================================
# HYBRID ACTOR-CRITIC
# ============================================================

class NoisyHybridActorCritic:

    def __init__(
        self,
        state_dim,
        action_dim,
        shots,
        n_qubits,
        n_layers,
        noise_type,
        readout_error,
        depolarizing_probability,
    ):

        self.actor = (
            NoisyQuantumActor(

                state_dim=
                    state_dim,

                action_dim=
                    action_dim,

                shots=
                    shots,

                n_qubits=
                    n_qubits,

                n_layers=
                    n_layers,

                noise_type=
                    noise_type,

                readout_error=
                    readout_error,

                depolarizing_probability=
                    depolarizing_probability,
            )
        )


        self.critic = (
            qe.ClassicalCritic(
                state_dim=
                    state_dim,
                hidden_dim=64,
            )
            .to(
                qe.DEVICE
            )
        )


    @torch.no_grad()
    def act(
        self,
        state,
    ):

        (
            action,
            log_prob,
            probs,
        ) = self.actor.act(
            state
        )


        state_tensor = (
            torch.tensor(
                state,
                dtype=torch.float32,
                device=qe.DEVICE,
            )
            .unsqueeze(0)
        )


        value = float(
            self.critic(
                state_tensor
            ).item()
        )


        return (
            action,
            log_prob,
            value,
            probs,
        )


# ============================================================
# ANALYTIC NOISY SHADOW
# ============================================================

def make_noisy_analytic_shadow(
    actor,
):

    shadow = (
        NoisyQuantumActor(

            state_dim=
                actor.state_dim,

            action_dim=
                actor.action_dim,

            shots=
                None,

            n_qubits=
                actor.n_qubits,

            n_layers=
                actor.n_layers,

            noise_type=
                actor.noise_type,

            readout_error=
                actor.readout_error,

            depolarizing_probability=
                actor.depolarizing_probability,
        )
    )


    with torch.no_grad():

        shadow.quantum_weights.copy_(
            actor
            .quantum_weights
            .detach()
        )


    for parameter in shadow.parameters():

        parameter.requires_grad_(
            False
        )


    return shadow


@torch.no_grad()
def analytic_action_probs(
    actor,
    state,
):

    probs = actor.single_forward(
        state
    )


    return (
        probs.detach()
        .cpu()
        .numpy()
        .astype(
            np.float64
        )
    )


# ============================================================
# NOISE-AWARE QTRUST CERTIFICATION
# ============================================================

def certify_noisy_update(
    old_actor,
    new_actor,
    buffer,
    advantages,
    seed,
    delta,
):

    records = []


    lower_clip = (
        1.0
        - qe.PPO_CONFIG[
            "clip_epsilon"
        ]
    )


    upper_clip = (
        1.0
        + qe.PPO_CONFIG[
            "clip_epsilon"
        ]
    )


    for i in range(
        len(buffer)
    ):

        state = (
            buffer.states[i]
        )


        action = int(
            buffer.actions[i]
        )


        advantage = float(
            advantages[i]
        )


        p_old = analytic_action_probs(
            old_actor,
            state,
        )


        p_new = analytic_action_probs(
            new_actor,
            state,
        )


        _, result = (
            qe.qtrust_finite_horizon_test(

                p_old=
                    p_old,

                p_new=
                    p_new,

                action=
                    action,

                advantage=
                    advantage,

                seed=
                    seed
                    + i * 1009,

                shot_schedule=[
                    32,
                    64,
                    128,
                    256,
                    512,
                    1024,
                    2048,
                    4096,
                ],

                delta=
                    delta,

                clip_epsilon=
                    qe.PPO_CONFIG[
                        "clip_epsilon"
                    ],
            )
        )


        true_ratio = float(
            result[
                "true_ratio"
            ]
        )


        if advantage >= 0:

            true_should_clip = (
                true_ratio
                > upper_clip
            )


        else:

            true_should_clip = (
                true_ratio
                < lower_clip
            )


        decision = result[
            "decision"
        ]


        if decision in {
            "CONFIRMED_UPPER_CLIP",
            "CONFIRMED_LOWER_CLIP",
        }:

            predicted_clip = True


        elif decision in {
            "SAFE_NO_UPPER_CLIP",
            "SAFE_NO_LOWER_CLIP",
        }:

            predicted_clip = False


        else:

            predicted_clip = None


        classification_correct = (

            np.nan

            if predicted_clip is None

            else bool(
                predicted_clip
                == true_should_clip
            )
        )


        records.append(
            {

                "sample":
                    i,

                "advantage":
                    advantage,

                "true_ratio":
                    true_ratio,

                "decision":
                    decision,

                "resolved":
                    bool(
                        result[
                            "resolved"
                        ]
                    ),

                "shots_per_policy":
                    int(
                        result[
                            "shots_per_policy"
                        ]
                    ),

                "total_shots":
                    int(
                        result[
                            "total_probability_shots"
                        ]
                    ),

                "classification_correct":
                    classification_correct,
            }
        )


    df = pd.DataFrame(
        records
    )


    unresolved_fraction = float(
        1.0
        - df[
            "resolved"
        ].mean()
    )


    resolved = df[
        df[
            "classification_correct"
        ].notna()
    ]


    accuracy = (

        float(
            resolved[
                "classification_correct"
            ].mean()
        )

        if len(resolved)

        else np.nan
    )


    return (

        df,

        {

            "unresolved_fraction":
                unresolved_fraction,

            "classification_accuracy":
                accuracy,

            "total_qtrust_shots":
                int(
                    df[
                        "total_shots"
                    ].sum()
                ),

            "mean_qtrust_shots":
                float(
                    df[
                        "total_shots"
                    ].mean()
                ),
        },
    )


# ============================================================
# TRAIN NOISY POLICY
# ============================================================

def train_noisy_policy(
    env_name,
    method,
    shots,
    total_steps,
    rollout_steps,
    seed,
    n_qubits,
    n_layers,
    noise_type,
    readout_error,
    depolarizing_probability,
    delta=0.05,
):

    assert (
        total_steps
        % rollout_steps
        == 0
    )


    qtrust_enabled = (
        method
        == "QTrust-PPO-128"
    )


    qe.set_global_seed(
        seed
    )


    env = qe.make_env(
        env_name,
        seed=seed,
    )


    state_dim = (
        qe.ENV_CONFIG[
            env_name
        ][
            "state_dim"
        ]
    )


    action_dim = (
        qe.ENV_CONFIG[
            env_name
        ][
            "action_dim"
        ]
    )


    model = (
        NoisyHybridActorCritic(

            state_dim=
                state_dim,

            action_dim=
                action_dim,

            shots=
                shots,

            n_qubits=
                n_qubits,

            n_layers=
                n_layers,

            noise_type=
                noise_type,

            readout_error=
                readout_error,

            depolarizing_probability=
                depolarizing_probability,
        )
    )


    actor_optimizer = (
        torch.optim.Adam(
            model.actor.parameters(),
            lr=
                qe.PPO_CONFIG[
                    "actor_lr"
                ],
        )
    )


    critic_optimizer = (
        torch.optim.Adam(
            model.critic.parameters(),
            lr=
                qe.PPO_CONFIG[
                    "critic_lr"
                ],
        )
    )


    buffer = qe.RolloutBuffer()


    global_step = 0
    update_number = 0
    episode_number = 1

    accepted_updates = 0
    rolled_back_updates = 0

    cumulative_action_shots = 0
    cumulative_gradient_shots = 0
    cumulative_certification_shots = 0

    update_records = []


    state, _ = env.reset(
        seed=
            seed
            + episode_number
    )


    state = np.asarray(
        state,
        dtype=np.float32,
    )


    start_time = (
        time.perf_counter()
    )


    while (
        global_step
        < total_steps
    ):

        (
            action,
            log_prob,
            value,
            _,
        ) = model.act(
            state
        )


        cumulative_action_shots += (
            shots
        )


        (
            next_state,
            reward,
            terminated,
            truncated,
            _,
        ) = env.step(
            action
        )


        next_state = np.asarray(
            next_state,
            dtype=np.float32,
        )


        if terminated:

            next_value = 0.0


        else:

            next_tensor = (
                torch.tensor(
                    next_state,
                    dtype=torch.float32,
                    device=qe.DEVICE,
                )
                .unsqueeze(0)
            )


            with torch.no_grad():

                next_value = float(
                    model.critic(
                        next_tensor
                    ).item()
                )


        buffer.add(

            state=
                state,

            action=
                action,

            log_prob=
                log_prob,

            reward=
                reward,

            value=
                value,

            next_value=
                next_value,

            terminated=
                terminated,

            truncated=
                truncated,

            shots=
                shots,
        )


        global_step += 1


        if (
            len(buffer)
            == rollout_steps
        ):

            update_number += 1


            if qtrust_enabled:

                old_shadow = (
                    make_noisy_analytic_shadow(
                        model.actor
                    )
                )


                old_weights = (
                    model.actor
                    .quantum_weights
                    .detach()
                    .clone()
                )


                old_optimizer_state = (
                    copy.deepcopy(
                        actor_optimizer
                        .state_dict()
                    )
                )


                advantages, _ = (
                    qe.get_buffer_advantages(
                        buffer
                    )
                )


            tracker = qml.Tracker(
                model.actor
                .qnode
                .device
            )


            with tracker:

                metrics = (
                    qe.quantum_ppo_update(

                        model=
                            model,

                        buffer=
                            buffer,

                        actor_optimizer=
                            actor_optimizer,

                        critic_optimizer=
                            critic_optimizer,

                        update_epochs=
                            1,
                    )
                )


            gradient_shots = int(
                tracker.totals.get(
                    "shots",
                    0,
                )
            )


            gradient_executions = int(
                tracker.totals.get(
                    "executions",
                    0,
                )
            )


            cumulative_gradient_shots += (
                gradient_shots
            )


            unresolved_fraction = 0.0
            classification_accuracy = np.nan
            mean_qtrust_shots = 0.0
            certification_shots = 0


            if qtrust_enabled:

                new_shadow = (
                    make_noisy_analytic_shadow(
                        model.actor
                    )
                )


                (
                    certification_df,
                    trust_metrics,
                ) = certify_noisy_update(

                    old_actor=
                        old_shadow,

                    new_actor=
                        new_shadow,

                    buffer=
                        buffer,

                    advantages=
                        advantages,

                    seed=
                        seed
                        + update_number
                        * 100_000,

                    delta=
                        delta,
                )


                unresolved_fraction = float(
                    trust_metrics[
                        "unresolved_fraction"
                    ]
                )


                classification_accuracy = (
                    trust_metrics[
                        "classification_accuracy"
                    ]
                )


                mean_qtrust_shots = float(
                    trust_metrics[
                        "mean_qtrust_shots"
                    ]
                )


                certification_shots = int(
                    trust_metrics[
                        "total_qtrust_shots"
                    ]
                )


                cumulative_certification_shots += (
                    certification_shots
                )


                if (
                    unresolved_fraction
                    > 0.0
                ):

                    accepted = False

                    rolled_back_updates += 1


                    # ----------------------------------------
                    # CORRECTED BLOCK
                    # ----------------------------------------
                    with torch.no_grad():

                        model.actor.quantum_weights.copy_(
                            old_weights
                        )


                    actor_optimizer.load_state_dict(
                        old_optimizer_state
                    )


                else:

                    accepted = True

                    accepted_updates += 1


            else:

                accepted = True

                accepted_updates += 1


            metrics.update(
                {

                    "update":
                        update_number,

                    "global_step":
                        global_step,

                    "update_accepted":
                        accepted,

                    "unresolved_fraction":
                        unresolved_fraction,

                    "classification_accuracy":
                        classification_accuracy,

                    "mean_qtrust_shots":
                        mean_qtrust_shots,

                    "gradient_shots_update":
                        gradient_shots,

                    "gradient_executions":
                        gradient_executions,

                    "certification_shots_update":
                        certification_shots,

                    "cumulative_action_shots":
                        cumulative_action_shots,

                    "cumulative_gradient_shots":
                        cumulative_gradient_shots,

                    "cumulative_certification_shots":
                        cumulative_certification_shots,

                    "cumulative_total_shots":
                        (
                            cumulative_action_shots
                            +
                            cumulative_gradient_shots
                            +
                            cumulative_certification_shots
                        ),
                }
            )


            update_records.append(
                metrics
            )


            buffer.clear()


        if (
            terminated
            or truncated
        ):

            episode_number += 1


            state, _ = env.reset(
                seed=
                    seed
                    + episode_number
            )


            state = np.asarray(
                state,
                dtype=np.float32,
            )


        else:

            state = next_state


    elapsed = (
        time.perf_counter()
        - start_time
    )


    env.close()


    return (

        model,

        pd.DataFrame(
            update_records
        ),

        {

            "method":
                method,

            "environment_steps":
                global_step,

            "ppo_updates":
                update_number,

            "accepted_updates":
                accepted_updates,

            "rolled_back_updates":
                rolled_back_updates,

            "action_shots":
                cumulative_action_shots,

            "gradient_shots":
                cumulative_gradient_shots,

            "certification_shots":
                cumulative_certification_shots,

            "total_quantum_shots":
                (
                    cumulative_action_shots
                    +
                    cumulative_gradient_shots
                    +
                    cumulative_certification_shots
                ),

            "elapsed_seconds":
                elapsed,
        },
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_noisy_policy(
    model,
    env_name,
    seeds,
):

    actor = model.actor

    records = []

    total_shots = 0


    start_time = (
        time.perf_counter()
    )


    for (
        episode_number,
        eval_seed,
    ) in enumerate(
        seeds,
        start=1,
    ):

        env = qe.make_env(
            env_name,
            seed=eval_seed,
        )


        state, _ = env.reset(
            seed=eval_seed
        )


        state = np.asarray(
            state,
            dtype=np.float32,
        )


        terminated = False
        truncated = False

        reward_sum = 0.0
        steps = 0


        while not (
            terminated
            or truncated
        ):

            with torch.no_grad():

                probs = (
                    actor.single_forward(
                        state
                    )
                )


            total_shots += int(
                actor.shots
            )


            action = int(
                torch.argmax(
                    probs
                ).item()
            )


            (
                state,
                reward,
                terminated,
                truncated,
                _,
            ) = env.step(
                action
            )


            state = np.asarray(
                state,
                dtype=np.float32,
            )


            reward_sum += float(
                reward
            )

            steps += 1


        env.close()


        records.append(
            {

                "episode":
                    episode_number,

                "seed":
                    eval_seed,

                "reward":
                    reward_sum,

                "steps":
                    steps,
            }
        )


    elapsed = (
        time.perf_counter()
        - start_time
    )


    df = pd.DataFrame(
        records
    )


    rewards = (
        df[
            "reward"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    return (

        df,

        {

            "evaluation_episodes":
                len(rewards),

            "mean_eval_reward":
                float(
                    np.mean(
                        rewards
                    )
                ),

            "std_eval_reward":
                float(
                    np.std(
                        rewards,
                        ddof=1,
                    )
                ),

            "median_eval_reward":
                float(
                    np.median(
                        rewards
                    )
                ),

            "min_eval_reward":
                float(
                    np.min(
                        rewards
                    )
                ),

            "max_eval_reward":
                float(
                    np.max(
                        rewards
                    )
                ),

            "mean_eval_steps":
                float(
                    df[
                        "steps"
                    ].mean()
                ),

            "evaluation_quantum_shots":
                int(
                    total_shots
                ),

            "evaluation_runtime_seconds":
                float(
                    elapsed
                ),
        },
    )
'''



# SAVE NOISE ENGINE


NOISE_ENGINE_PATH = (
    JOURNAL_ROOT
    / "journal_noise_engine.py"
)


NOISE_ENGINE_PATH.write_text(
    NOISE_ENGINE_SOURCE,
    encoding="utf-8",
)



# NEW: HARD SYNTAX CHECK BEFORE ANY EXPERIMENT


py_compile.compile(
    str(
        NOISE_ENGINE_PATH
    ),
    doraise=True,
)


print(
    "Noise engine syntax check: PASS"
)



# HASH VERIFIED SOURCE


NOISE_ENGINE_SHA256 = (
    hashlib.sha256(
        NOISE_ENGINE_SOURCE.encode(
            "utf-8"
        )
    )
    .hexdigest()
)


print(
    "Noise engine SHA256:",
    NOISE_ENGINE_SHA256
)



# IMPORT NOISE ENGINE


if str(
    JOURNAL_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            JOURNAL_ROOT
        ),
    )


if (
    "journal_noise_engine"
    in sys.modules
):

    del sys.modules[
        "journal_noise_engine"
    ]


importlib.invalidate_caches()


import torch
import journal_noise_engine as qn



# PRE-FLIGHT TEST BOTH NOISE MODELS


for configuration in [

    {
        "noise_type":
            "readout",

        "readout_error":
            0.02,

        "depolarizing_probability":
            0.0,
    },

    {
        "noise_type":
            "depolarizing",

        "readout_error":
            0.0,

        "depolarizing_probability":
            0.005,
    },
]:

    actor = (
        qn.NoisyQuantumActor(

            state_dim=4,

            action_dim=2,

            shots=32,

            n_qubits=4,

            n_layers=3,

            noise_type=
                configuration[
                    "noise_type"
                ],

            readout_error=
                configuration[
                    "readout_error"
                ],

            depolarizing_probability=
                configuration[
                    "depolarizing_probability"
                ],
        )
    )


    state = torch.tensor(
        [
            0.1,
            -0.2,
            0.05,
            0.0,
        ],
        dtype=torch.float64,
    )


    probs = (
        actor.single_forward(
            state
        )
    )


    if not torch.isfinite(
        probs
    ).all():

        raise RuntimeError(
            "Noise preflight produced "
            "non-finite probabilities."
        )


    if abs(
        float(
            probs.detach()
            .sum()
        )
        - 1.0
    ) > 1e-6:

        raise RuntimeError(
            "Noise probabilities do not sum to one."
        )


    loss = (
        -torch.log(
            probs[0]
            + 1e-12
        )
    )


    loss.backward()


    gradient = (
        actor
        .quantum_weights
        .grad
    )


    if (
        gradient is None
        or
        not torch.isfinite(
            gradient
        ).all()
    ):

        raise RuntimeError(
            "Noise parameter-shift gradient failed."
        )


    print(
        f"{configuration['noise_type']} "
        "noise preflight: PASS"
    )


    del actor

    torch.cuda.empty_cache()


print(
    "Noise engine preflight: PASS"
)



# COMPLETION / RESUME VALIDATION


def C_summary_path(
    run_id,
):

    return (
        C_RUNS_DIR
        / f"{run_id}.json"
    )


def valid_C_run(
    run_id,
):

    path = (
        C_summary_path(
            run_id
        )
    )


    if not path.exists():
        return False


    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )


        return (

            payload.get(
                "status"
            )
            == "COMPLETE"

            and

            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH

            and

            payload.get(
                "parent_protocol_hash"
            )
            == EXPECTED_PROTOCOL_HASH

            and

            payload.get(
                "noise_implementation_hash"
            )
            == NOISE_IMPLEMENTATION_HASH

            and

            payload.get(
                "run_id"
            )
            == run_id
        )


    except Exception:

        return False


completed_before = [

    run_id

    for run_id
    in C_MANIFEST[
        "run_id"
    ]

    if valid_C_run(
        run_id
    )
]


pending_manifest = (

    C_MANIFEST[
        ~C_MANIFEST[
            "run_id"
        ].isin(
            completed_before
        )
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print()
print(
    "Completed before Cell 5 :",
    len(
        completed_before
    ),
    "/ 60"
)


print(
    "Pending Experiment-C runs:",
    len(
        pending_manifest
    )
)



# EXPERIMENT-C WORKER SOURCE


WORKER_SOURCE = r'''
import os
import sys
import json
import time
import traceback

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


if len(sys.argv) != 5:

    raise RuntimeError(
        "Usage: expC_worker.py "
        "<task_file> <journal_root> "
        "<journal_hash> <noise_hash>"
    )


TASK_FILE = Path(
    sys.argv[1]
)

JOURNAL_ROOT = Path(
    sys.argv[2]
)

JOURNAL_PROTOCOL_HASH = (
    sys.argv[3]
)

NOISE_IMPLEMENTATION_HASH = (
    sys.argv[4]
)


PARENT_PROTOCOL_HASH = (
    "3cf4bcb18e1111b3"
)


# ============================================================
# RESOURCE GUARDS
# ============================================================

os.environ[
    "OMP_NUM_THREADS"
] = "1"

os.environ[
    "MKL_NUM_THREADS"
] = "1"

os.environ[
    "OPENBLAS_NUM_THREADS"
] = "1"

os.environ[
    "NUMEXPR_NUM_THREADS"
] = "1"

os.environ[
    "CUDA_MODULE_LOADING"
] = "LAZY"


torch.set_num_threads(
    1
)


try:

    torch.set_num_interop_threads(
        1
    )

except Exception:

    pass


# ============================================================
# IMPORT ENGINES
# ============================================================

sys.path.insert(
    0,
    str(
        JOURNAL_ROOT
    ),
)


import journal_engine as qe
import journal_noise_engine as qn


if (
    qe.PARENT_PROTOCOL_HASH
    != PARENT_PROTOCOL_HASH
):

    raise RuntimeError(
        "Parent protocol mismatch."
    )


if (
    qe.ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):

    raise RuntimeError(
        "lightning.gpu unavailable."
    )


# ============================================================
# PATHS
# ============================================================

EXP_C_DIR = (
    JOURNAL_ROOT
    / "experiment_C_noise"
)


RUNS_DIR = (
    EXP_C_DIR
    / "runs"
)


UPDATES_DIR = (
    EXP_C_DIR
    / "updates"
)


EVAL_DIR = (
    EXP_C_DIR
    / "evaluation"
)


for directory in [
    RUNS_DIR,
    UPDATES_DIR,
    EVAL_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# ATOMIC WRITES
# ============================================================

def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )


    temporary = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )


    os.replace(
        temporary,
        path,
    )


def atomic_csv(
    path,
    dataframe,
):

    path = Path(
        path
    )


    temporary = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary,
        index=False,
    )


    os.replace(
        temporary,
        path,
    )


def valid_existing(
    run_id,
):

    path = (
        RUNS_DIR
        / f"{run_id}.json"
    )


    if not path.exists():

        return False


    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )


        return (

            payload.get(
                "status"
            )
            == "COMPLETE"

            and

            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH

            and

            payload.get(
                "noise_implementation_hash"
            )
            == NOISE_IMPLEMENTATION_HASH
        )


    except Exception:

        return False


# ============================================================
# LOAD TASKS
# ============================================================

tasks = json.loads(
    TASK_FILE.read_text(
        encoding="utf-8"
    )
)


print(
    "[worker] CUDA_VISIBLE_DEVICES =",
    os.environ.get(
        "CUDA_VISIBLE_DEVICES"
    )
)


print(
    "[worker] tasks =",
    len(tasks)
)


# ============================================================
# RUN TASKS
# ============================================================

for task_number, task in enumerate(
    tasks,
    start=1,
):

    run_id = (
        task[
            "run_id"
        ]
    )


    if valid_existing(
        run_id
    ):

        print(
            "[skip]",
            run_id
        )

        continue


    run_start = (
        time.perf_counter()
    )


    try:

        method = str(
            task[
                "method"
            ]
        )


        seed = int(
            task[
                "seed"
            ]
        )


        noise_type = str(
            task[
                "noise_type"
            ]
        )


        readout_error = float(
            task[
                "readout_error"
            ]
        )


        depolarizing_probability = float(
            task[
                "depolarizing_probability"
            ]
        )


        if method == "Fixed-QPPO-128":

            shots = 128


        elif method == "Fixed-QPPO-256":

            shots = 256


        elif method == "QTrust-PPO-128":

            shots = 128


        else:

            raise ValueError(
                "Unexpected method: "
                + method
            )


        print(
            f"[{task_number}/{len(tasks)}] "
            f"START {run_id}"
        )


        # ====================================================
        # TRAIN
        # ====================================================

        (
            model,
            update_df,
            train_stats,
        ) = qn.train_noisy_policy(

            env_name=
                "CartPole-v1",

            method=
                method,

            shots=
                shots,

            total_steps=
                2048,

            rollout_steps=
                32,

            seed=
                seed,

            n_qubits=
                4,

            n_layers=
                3,

            noise_type=
                noise_type,

            readout_error=
                readout_error,

            depolarizing_probability=
                depolarizing_probability,

            delta=
                0.05,
        )


        # ====================================================
        # EVALUATE
        # ====================================================

        evaluation_seeds = (
            qe.make_final_eval_seeds(

                training_seed=
                    seed,

                n_episodes=
                    30,
            )
        )


        (
            evaluation_df,
            evaluation_stats,
        ) = qn.evaluate_noisy_policy(

            model=
                model,

            env_name=
                "CartPole-v1",

            seeds=
                evaluation_seeds,
        )


        # ====================================================
        # DIAGNOSTICS
        # ====================================================

        mean_unresolved = float(
            update_df[
                "unresolved_fraction"
            ].mean()
        )


        accuracy_values = (
            update_df[
                "classification_accuracy"
            ]
            .dropna()
        )


        classification_accuracy = (

            float(
                accuracy_values.mean()
            )

            if len(
                accuracy_values
            )

            else None
        )


        mean_adaptive_shots = float(
            update_df[
                "mean_qtrust_shots"
            ].mean()
        )


        # ====================================================
        # WRITE TRACES FIRST
        # ====================================================

        update_path = (
            UPDATES_DIR
            / f"{run_id}.csv"
        )


        evaluation_path = (
            EVAL_DIR
            / f"{run_id}.csv"
        )


        atomic_csv(
            update_path,
            update_df,
        )


        atomic_csv(
            evaluation_path,
            evaluation_df,
        )


        wall_time = float(
            time.perf_counter()
            - run_start
        )


        # ====================================================
        # WRITE COMPLETION MARKER LAST
        # ====================================================

        summary = {

            "status":
                "COMPLETE",

            "experiment":
                "C",

            "run_id":
                run_id,

            "variant":
                task[
                    "variant"
                ],

            "environment":
                "CartPole-v1",

            "method":
                method,

            "seed":
                seed,

            "parent_protocol_hash":
                PARENT_PROTOCOL_HASH,

            "journal_protocol_hash":
                JOURNAL_PROTOCOL_HASH,

            "noise_implementation_hash":
                NOISE_IMPLEMENTATION_HASH,

            "noise_type":
                noise_type,

            "readout_error":
                readout_error,

            "depolarizing_probability":
                depolarizing_probability,

            "shots":
                shots,

            "qubits":
                4,

            "layers":
                3,

            "training_steps":
                2048,

            "rollout_steps":
                32,

            "evaluation_episodes":
                30,

            "mean_eval_reward":
                float(
                    evaluation_stats[
                        "mean_eval_reward"
                    ]
                ),

            "std_eval_reward":
                float(
                    evaluation_stats[
                        "std_eval_reward"
                    ]
                ),

            "median_eval_reward":
                float(
                    evaluation_stats[
                        "median_eval_reward"
                    ]
                ),

            "mean_eval_steps":
                float(
                    evaluation_stats[
                        "mean_eval_steps"
                    ]
                ),

            "accepted_updates":
                int(
                    train_stats[
                        "accepted_updates"
                    ]
                ),

            "rolled_back_updates":
                int(
                    train_stats[
                        "rolled_back_updates"
                    ]
                ),

            "mean_unresolved_fraction":
                mean_unresolved,

            "classification_accuracy":
                classification_accuracy,

            "mean_adaptive_shots":
                mean_adaptive_shots,

            "action_shots":
                int(
                    train_stats[
                        "action_shots"
                    ]
                ),

            "gradient_shots":
                int(
                    train_stats[
                        "gradient_shots"
                    ]
                ),

            "certification_shots":
                int(
                    train_stats[
                        "certification_shots"
                    ]
                ),

            "training_quantum_shots":
                int(
                    train_stats[
                        "total_quantum_shots"
                    ]
                ),

            "evaluation_quantum_shots":
                int(
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "overall_quantum_shots":
                int(
                    train_stats[
                        "total_quantum_shots"
                    ]
                    +
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "training_runtime_seconds":
                float(
                    train_stats[
                        "elapsed_seconds"
                    ]
                ),

            "evaluation_runtime_seconds":
                float(
                    evaluation_stats[
                        "evaluation_runtime_seconds"
                    ]
                ),

            "wall_runtime_seconds":
                wall_time,

            "updates_file":
                str(
                    update_path.relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "evaluation_file":
                str(
                    evaluation_path.relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "completed_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        atomic_json(
            RUNS_DIR
            / f"{run_id}.json",

            summary,
        )


        print(
            f"[complete] {run_id} | "
            f"reward="
            f"{summary['mean_eval_reward']:.3f} | "
            f"wall="
            f"{wall_time / 60:.1f} min"
        )


        del model


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    except Exception as exc:

        atomic_json(

            RUNS_DIR
            / f"{run_id}.failed.json",

            {

                "status":
                    "FAILED",

                "run_id":
                    run_id,

                "error_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    str(
                        exc
                    ),

                "timestamp_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
        )


        traceback.print_exc()

        raise


print(
    "[worker] ALL ASSIGNED TASKS COMPLETE"
)
'''



# SAVE + COMPILE WORKER BEFORE LAUNCH


WORKER_PATH = (
    EXP_C_DIR
    / "expC_worker.py"
)


WORKER_PATH.write_text(
    WORKER_SOURCE,
    encoding="utf-8",
)


py_compile.compile(
    str(
        WORKER_PATH
    ),
    doraise=True,
)


print(
    "Experiment-C worker syntax check: PASS"
)



# BUILD 4 BALANCED WORKER QUEUES


WORKER_GPU_MAP = [
    0,
    1,
    0,
    1,
]


TOTAL_WORKERS = 4


tasks = (
    pending_manifest
    .to_dict(
        orient="records"
    )
)


worker_queues = [
    []
    for _ in range(
        TOTAL_WORKERS
    )
]


for index, task in enumerate(
    tasks
):

    clean_task = {}


    for key, value in task.items():

        if pd.isna(
            value
        ):

            clean_task[
                key
            ] = None


        elif isinstance(
            value,
            np.generic,
        ):

            clean_task[
                key
            ] = value.item()


        else:

            clean_task[
                key
            ] = value


    worker_queues[
        index
        % TOTAL_WORKERS
    ].append(
        clean_task
    )


task_files = []


for worker_id, queue in enumerate(
    worker_queues
):

    task_file = (
        C_TASK_DIR
        / f"worker_{worker_id}_tasks.json"
    )


    task_file.write_text(
        json.dumps(
            queue,
            indent=2,
        ),
        encoding="utf-8",
    )


    task_files.append(
        task_file
    )


print()
print(
    "Worker assignment:"
)


for worker_id, (
    gpu,
    queue,
) in enumerate(
    zip(
        WORKER_GPU_MAP,
        worker_queues,
    )
):

    print(
        f"  Worker {worker_id} "
        f"-> T4 GPU {gpu} "
        f"-> {len(queue)} runs"
    )






C_RECOVERY_ZIP = (
    Path(
        "/kaggle/working"
    )
    / "QTrust_Journal_Experiment_C_LATEST.zip"
)


def save_C_recovery_zip():

    temporary = (
        C_RECOVERY_ZIP
        .with_suffix(
            ".zip.tmp"
        )
    )


    if temporary.exists():

        temporary.unlink()


    with zipfile.ZipFile(

        temporary,

        "w",

        compression=
            zipfile.ZIP_DEFLATED,

        compresslevel=1,

    ) as zf:


        for file in sorted(
            EXP_C_DIR.rglob("*")
        ):

            if not file.is_file():

                continue


            if file.suffix == ".tmp":

                continue


            relative = (
                file.relative_to(
                    JOURNAL_ROOT
                )
            )


            zf.write(
                file,

                arcname=str(
                    Path(
                        JOURNAL_ROOT.name
                    )
                    / relative
                ),
            )


        for core_name in [

            "journal_manifest.json",
            "journal_extension_protocol.json",
            "journal_run_manifest.csv",
            "journal_engine.py",
            "journal_noise_engine.py",
            "JOURNAL_EXPERIMENT_PLAN.txt",

        ]:

            path = (
                JOURNAL_ROOT
                / core_name
            )


            if path.exists():

                zf.write(
                    path,

                    arcname=str(
                        Path(
                            JOURNAL_ROOT.name
                        )
                        / core_name
                    ),
                )


    with zipfile.ZipFile(
        temporary,
        "r",
    ) as zf:

        bad = (
            zf.testzip()
        )


    if bad is not None:

        temporary.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Experiment-C recovery ZIP "
            f"failed verification: {bad}"
        )


    os.replace(
        temporary,
        C_RECOVERY_ZIP,
    )



# START FOUR WORKERS


processes = []


if len(
    pending_manifest
) > 0:

    print()
    print("=" * 90)
    print("STARTING EXPERIMENT C")
    print("=" * 90)


    for worker_id, (
        gpu,
        task_file,
    ) in enumerate(
        zip(
            WORKER_GPU_MAP,
            task_files,
        )
    ):

        if not worker_queues[
            worker_id
        ]:

            continue


        worker_env = (
            os.environ.copy()
        )


        worker_env[
            "CUDA_VISIBLE_DEVICES"
        ] = str(
            gpu
        )


        worker_env[
            "OMP_NUM_THREADS"
        ] = "1"


        worker_env[
            "MKL_NUM_THREADS"
        ] = "1"


        worker_env[
            "OPENBLAS_NUM_THREADS"
        ] = "1"


        worker_env[
            "NUMEXPR_NUM_THREADS"
        ] = "1"


        worker_env[
            "CUDA_MODULE_LOADING"
        ] = "LAZY"


        log_path = (
            C_LOG_DIR
            / f"worker_{worker_id}.log"
        )


        log_handle = open(
            log_path,
            "w",
            buffering=1,
            encoding="utf-8",
        )


        process = subprocess.Popen(

            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    task_file
                ),
                str(
                    JOURNAL_ROOT
                ),
                JOURNAL_PROTOCOL_HASH,
                NOISE_IMPLEMENTATION_HASH,
            ],

            env=
                worker_env,

            stdout=
                log_handle,

            stderr=
                subprocess.STDOUT,

            text=True,
        )


        processes.append(
            {

                "worker_id":
                    worker_id,

                "gpu":
                    gpu,

                "process":
                    process,

                "log_handle":
                    log_handle,

                "log_path":
                    log_path,
            }
        )



# PROGRESS MONITOR


def completed_C_runs():

    return [

        run_id

        for run_id
        in C_MANIFEST[
            "run_id"
        ]

        if valid_C_run(
            run_id
        )
    ]


last_count = (
    len(
        completed_before
    )
)


last_zip_count = -1


if last_count > 0:

    save_C_recovery_zip()

    last_zip_count = (
        last_count
    )


while any(

    item[
        "process"
    ].poll()
    is None

    for item
    in processes

):

    current = len(
        completed_C_runs()
    )


    if current != last_count:

        print(
            f"[progress] Experiment C: "
            f"{current}/60 complete "
            f"({100.0 * current / 60:.1f}%)"
        )


        last_count = (
            current
        )


        if (
            current
            != last_zip_count
        ):

            save_C_recovery_zip()

            last_zip_count = (
                current
            )


    time.sleep(
        20
    )


for item in processes:

    item[
        "log_handle"
    ].close()



# CHECK WORKER FAILURES


failed_workers = [

    item

    for item in processes

    if item[
        "process"
    ].returncode
    != 0
]


if failed_workers:

    save_C_recovery_zip()


    print()
    print("=" * 90)
    print("ONE OR MORE EXPERIMENT-C WORKERS FAILED")
    print("=" * 90)


    for item in failed_workers:

        print()
        print(
            f"Worker {item['worker_id']} "
            f"(GPU {item['gpu']})"
        )


        print(
            "Log:",
            item[
                "log_path"
            ]
        )


        try:

            tail = (
                item[
                    "log_path"
                ]
                .read_text(
                    encoding="utf-8"
                )
                .splitlines()[
                    -60:
                ]
            )


            print(
                "\n".join(
                    tail
                )
            )


        except Exception:

            pass


    raise RuntimeError(
        "Experiment C stopped because a worker failed. "
        "Completed runs are preserved. "
        "Rerun Cell 5 to resume."
    )



# REQUIRE 60 / 60 COMPLETE


completed_final = (
    completed_C_runs()
)


if len(
    completed_final
) != 60:

    save_C_recovery_zip()


    raise RuntimeError(
        "Experiment C expected 60 COMPLETE runs, "
        f"found {len(completed_final)}."
    )



# LOAD ALL RESULTS


rows = []


for run_id in C_MANIFEST[
    "run_id"
]:

    rows.append(

        json.loads(

            C_summary_path(
                run_id
            )
            .read_text(
                encoding="utf-8"
            )
        )
    )


C_RESULTS = pd.DataFrame(
    rows
)



# HARD SCIENTIFIC VALIDATION


if len(
    C_RESULTS
) != 60:

    raise RuntimeError(
        "Experiment-C result count != 60."
    )


if not C_RESULTS[
    "status"
].eq(
    "COMPLETE"
).all():

    raise RuntimeError(
        "Experiment C contains incomplete runs."
    )


if not C_RESULTS[
    "journal_protocol_hash"
].eq(
    JOURNAL_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Journal protocol hash mismatch "
        "in Experiment C."
    )


if not C_RESULTS[
    "parent_protocol_hash"
].eq(
    EXPECTED_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Parent protocol hash mismatch "
        "in Experiment C."
    )


if not C_RESULTS[
    "noise_implementation_hash"
].eq(
    NOISE_IMPLEMENTATION_HASH
).all():

    raise RuntimeError(
        "Noise implementation hash mismatch."
    )


condition_counts = (
    C_RESULTS
    .groupby(
        [
            "variant",
            "method",
        ]
    )
    .size()
)


if (
    len(
        condition_counts
    )
    != 6
    or
    not condition_counts.eq(
        10
    ).all()
):

    raise RuntimeError(
        "Every Experiment-C condition must "
        "contain exactly 10 seeds."
    )


EXPECTED_SEED_LIST = [
    11,
    23,
    37,
    53,
    71,
    89,
    107,
    131,
    157,
    181,
]


for (
    variant,
    method,
), group in C_RESULTS.groupby(
    [
        "variant",
        "method",
    ]
):

    seeds = sorted(
        group[
            "seed"
        ]
        .astype(int)
        .tolist()
    )


    if seeds != EXPECTED_SEED_LIST:

        raise RuntimeError(
            f"Seed mismatch: "
            f"{variant} / {method}"
        )



# SAVE MASTER EXPERIMENT-C RESULTS


C_RESULTS_PATH = (
    EXP_C_DIR
    / "experiment_C_all_runs.csv"
)


C_RESULTS.to_csv(
    C_RESULTS_PATH,
    index=False,
)



# DESCRIPTIVE SUMMARY




C_SUMMARY = (

    C_RESULTS

    .groupby(
        [
            "variant",
            "method",
        ],
        as_index=False,
    )

    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),

        Mean_Wall_Runtime_s=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


C_SUMMARY_PATH = (
    EXP_C_DIR
    / "experiment_C_summary.csv"
)


C_SUMMARY.to_csv(
    C_SUMMARY_PATH,
    index=False,
)



# UPDATE RUN MANIFEST


RUN_MANIFEST.loc[
    RUN_MANIFEST[
        "experiment"
    ]
    == "C",
    "status",
] = "COMPLETE"


RUN_MANIFEST.to_csv(
    JOURNAL_ROOT
    / "journal_run_manifest.csv",
    index=False,
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "experiment_C_noise"
] = {

    "status":
        "COMPLETE",

    "completed_runs":
        60,

    "conditions":
        6,

    "seeds_per_condition":
        10,

    "noise_implementation_hash":
        NOISE_IMPLEMENTATION_HASH,

    "noise_engine_sha256":
        NOISE_ENGINE_SHA256,

    "result_file":
        str(
            C_RESULTS_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),

    "summary_file":
        str(
            C_SUMMARY_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),
}


manifest[
    "current_completed_cell"
] = 5


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)



# CELL-5 PROVENANCE


cell5_record = {

    "cell":
        5,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        "C",

    "completed_runs":
        60,

    "conditions":
        6,

    "seeds_per_condition":
        10,

    "workers":
        4,

    "physical_gpus":
        2,

    "worker_gpu_map":
        WORKER_GPU_MAP,

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "noise_implementation_hash":
        NOISE_IMPLEMENTATION_HASH,

    "noise_engine_sha256":
        NOISE_ENGINE_SHA256,
}


(
    LOGS_DIR
    / "cell05_experiment_C.json"
).write_text(
    json.dumps(
        cell5_record,
        indent=2,
    ),
    encoding="utf-8",
)






save_C_recovery_zip()



# REPORT


print()
print("=" * 90)
print("EXPERIMENT C COMPLETE")
print("=" * 90)


print(
    "Completed runs :",
    len(
        C_RESULTS
    ),
    "/ 60"
)


print(
    "Conditions     : 6"
)


print(
    "Seeds/condition:",
    10
)


print(
    "GPUs           : 2 × Tesla T4"
)


print(
    "Workers        : 4"
)


print(
    "Journal hash   :",
    JOURNAL_PROTOCOL_HASH
)


print(
    "Noise hash     :",
    NOISE_IMPLEMENTATION_HASH
)


print()
print(
    C_SUMMARY.to_string(
        index=False
    )
)


print()
print(
    "Recovery ZIP:",
    C_RECOVERY_ZIP
)


print("=" * 90)






CELL5_CHECKPOINT = (
    save_journal_checkpoint(
        5,
        "Experiment_C_Noise_COMPLETE",
    )
)


print()
print("=" * 90)
print("CELL 5 COMPLETE")
print("=" * 90)


print(
    "Experiment C is now frozen."
)


print(
    "Next: Cell 6 will run Experiment D "
    "(20 VQC depth-scalability runs)."
)


print(
    "Do not modify Experiment-C settings "
    "after observing these results."
)

In [ ]:
# Resume Cell




from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import shutil
import zipfile
import subprocess
import importlib



# RESTORE WORKSPACE IF NEEDED


JOURNAL_ROOT = Path(
    "/kaggle/working/QTrust_PPO_Journal"
)

LATEST_ZIP = Path(
    "/kaggle/working/QTrust_Journal_LATEST.zip"
)


if not JOURNAL_ROOT.exists():

    print(
        "Journal folder not found directly."
    )

    if not LATEST_ZIP.exists():

        raise RuntimeError(
            "Neither persistent journal folder nor "
            "QTrust_Journal_LATEST.zip was found."
        )

    print(
        "Restoring journal workspace from:",
        LATEST_ZIP
    )

    with zipfile.ZipFile(
        LATEST_ZIP,
        "r",
    ) as zf:

        zf.extractall(
            "/kaggle/working"
        )


if not JOURNAL_ROOT.exists():

    raise RuntimeError(
        "Journal restoration failed."
    )


print(
    "Journal workspace:",
    JOURNAL_ROOT
)



# RECONSTRUCT ALL PATH VARIABLES


EXP_A_DIR = (
    JOURNAL_ROOT
    / "experiment_A"
)

EXP_B_DIR = (
    JOURNAL_ROOT
    / "experiment_B_ablations"
)

EXP_C_DIR = (
    JOURNAL_ROOT
    / "experiment_C_noise"
)

EXP_D_DIR = (
    JOURNAL_ROOT
    / "experiment_D_scalability"
)

EXP_E_DIR = (
    JOURNAL_ROOT
    / "experiment_E_confidence"
)

STATS_DIR = (
    JOURNAL_ROOT
    / "statistics"
)

FIGURES_DIR = (
    JOURNAL_ROOT
    / "figures"
)

TABLES_DIR = (
    JOURNAL_ROOT
    / "tables"
)

LOGS_DIR = (
    JOURNAL_ROOT
    / "logs"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/qtrust_journal_checkpoints"
)


for directory in [
    EXP_B_DIR,
    EXP_C_DIR,
    EXP_D_DIR,
    EXP_E_DIR,
    STATS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    LOGS_DIR,
    CHECKPOINT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )






EXPECTED_PROTOCOL_HASH = (
    "3cf4bcb18e1111b3"
)

JOURNAL_PROTOCOL_HASH = (
    "4534e690e365694c"
)

EXPECTED_SEEDS = [
    11,
    23,
    37,
    53,
    71,
    89,
    107,
    131,
    157,
    181,
]



# LOAD RUN MANIFEST


import pandas as pd
import numpy as np


RUN_MANIFEST_PATH = (
    JOURNAL_ROOT
    / "journal_run_manifest.csv"
)


if not RUN_MANIFEST_PATH.exists():

    raise RuntimeError(
        "journal_run_manifest.csv is missing."
    )


RUN_MANIFEST = pd.read_csv(
    RUN_MANIFEST_PATH
)



# VERIFY CORE FILES


required_files = [

    JOURNAL_ROOT
    / "journal_manifest.json",

    JOURNAL_ROOT
    / "journal_extension_protocol.json",

    JOURNAL_ROOT
    / "journal_run_manifest.csv",

    JOURNAL_ROOT
    / "journal_engine.py",

]


for path in required_files:

    if not path.exists():

        raise RuntimeError(
            f"Required file missing: {path}"
        )



# VERIFY COMPLETED EXPERIMENT B


B_RESULTS_PATH = (
    EXP_B_DIR
    / "experiment_B_all_runs.csv"
)


if not B_RESULTS_PATH.exists():

    raise RuntimeError(
        "Experiment-B master result file is missing."
    )


B_RESULTS = pd.read_csv(
    B_RESULTS_PATH
)


if len(
    B_RESULTS
) != 30:

    raise RuntimeError(
        f"Experiment B expected 30 runs; "
        f"found {len(B_RESULTS)}."
    )



# VERIFY COMPLETED EXPERIMENT C


C_RESULTS_PATH = (
    EXP_C_DIR
    / "experiment_C_all_runs.csv"
)


if not C_RESULTS_PATH.exists():

    raise RuntimeError(
        "Experiment-C master result file is missing."
    )


C_RESULTS = pd.read_csv(
    C_RESULTS_PATH
)


if len(
    C_RESULTS
) != 60:

    raise RuntimeError(
        f"Experiment C expected 60 runs; "
        f"found {len(C_RESULTS)}."
    )



# VERIFY EXPERIMENT-D MANIFEST


D_MANIFEST = (
    RUN_MANIFEST[
        RUN_MANIFEST[
            "experiment"
        ]
        == "D"
    ]
    .copy()
)


if len(
    D_MANIFEST
) != 20:

    raise RuntimeError(
        f"Experiment D expected 20 planned runs; "
        f"found {len(D_MANIFEST)}."
    )



# RESTORE CHECKPOINT FUNCTION


def save_journal_checkpoint(
    cell_number,
    label,
):

    CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    checkpoint = (
        CHECKPOINT_DIR
        / (
            f"QTrust_Journal_Cell"
            f"{int(cell_number):02d}_"
            f"{label}.zip"
        )
    )


    temporary = (
        checkpoint
        .with_suffix(
            ".zip.tmp"
        )
    )


    if temporary.exists():

        temporary.unlink()


    with zipfile.ZipFile(

        temporary,

        "w",

        compression=
            zipfile.ZIP_DEFLATED,

        compresslevel=1,

    ) as zf:


        for file in sorted(
            JOURNAL_ROOT.rglob("*")
        ):

            if not file.is_file():

                continue


            if file.suffix == ".tmp":

                continue


            relative = (
                file.relative_to(
                    JOURNAL_ROOT.parent
                )
            )


            zf.write(
                file,
                arcname=str(
                    relative
                ),
            )


    with zipfile.ZipFile(
        temporary,
        "r",
    ) as zf:

        bad = (
            zf.testzip()
        )


    if bad is not None:

        temporary.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            f"Checkpoint ZIP failed: {bad}"
        )


    os.replace(
        temporary,
        checkpoint,
    )


    latest = Path(
        "/kaggle/working/"
        "QTrust_Journal_LATEST.zip"
    )


    shutil.copy2(
        checkpoint,
        latest,
    )


    print()
    print("-" * 80)
    print(
        f"CHECKPOINT SAVED AFTER CELL {cell_number}"
    )
    print("-" * 80)

    print(
        "Checkpoint :",
        checkpoint
    )

    print(
        "Latest copy:",
        latest
    )

    print(
        "Size       :",
        f"{checkpoint.stat().st_size / (1024**2):.2f} MB"
    )

    print("-" * 80)


    return checkpoint



# VERIFY / RESTORE EXACT SOFTWARE ENVIRONMENT


def version_of(
    package_name,
):

    try:

        from importlib.metadata import version

        return version(
            package_name
        )

    except Exception:

        return None


required_versions = {

    "pennylane":
        "0.45.1",

    "pennylane-lightning-gpu":
        "0.45.0",

    "gymnasium":
        "1.2.0",

    "scipy":
        "1.16.3",
}


needs_install = []


for package, expected in (
    required_versions.items()
):

    observed = version_of(
        package
    )


    print(
        f"{package}: "
        f"{observed} "
        f"(required {expected})"
    )


    if observed != expected:

        needs_install.append(
            f"{package}=={expected}"
        )


if needs_install:

    print()
    print(
        "Restoring exact journal dependencies..."
    )


    subprocess.check_call(

        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
        ]
        + needs_install
    )


    importlib.invalidate_caches()



# IMPORT ENGINE AND TEST BOTH GPUs


if str(
    JOURNAL_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            JOURNAL_ROOT
        ),
    )


import torch
import pennylane as qml
import gymnasium
import scipy


if (
    "journal_engine"
    in sys.modules
):

    del sys.modules[
        "journal_engine"
    ]


importlib.invalidate_caches()

import journal_engine as qe


if (
    qe.PARENT_PROTOCOL_HASH
    != EXPECTED_PROTOCOL_HASH
):

    raise RuntimeError(
        "Parent Experiment-A hash mismatch."
    )


if (
    qe.ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):

    raise RuntimeError(
        "Journal engine is not configured "
        "for lightning.gpu."
    )


gpu_count = (
    torch.cuda.device_count()
)


if gpu_count != 2:

    raise RuntimeError(
        f"Expected 2 T4 GPUs; found {gpu_count}."
    )


gpu_names = [

    torch.cuda.get_device_name(
        index
    )

    for index
    in range(
        gpu_count
    )
]






print()
print("=" * 88)
print("QTRUST-PPO JOURNAL SESSION RESUMED")
print("=" * 88)

print(
    "Experiment A      : FROZEN / existing authoritative dataset"
)

print(
    "Experiment B      :",
    f"{len(B_RESULTS)} / 30 COMPLETE"
)

print(
    "Experiment C      :",
    f"{len(C_RESULTS)} / 60 COMPLETE"
)

print(
    "Experiment D      :",
    "0 / 20 new runs expected"
)

print(
    "Parent hash       :",
    EXPECTED_PROTOCOL_HASH
)

print(
    "Journal hash      :",
    JOURNAL_PROTOCOL_HASH
)

print(
    "GPU count         :",
    gpu_count
)

for index, name in enumerate(
    gpu_names
):

    print(
        f"GPU {index}             :",
        name
    )


print(
    "PennyLane         :",
    qml.__version__
)

print(
    "Gymnasium         :",
    gymnasium.__version__
)

print(
    "SciPy             :",
    scipy.__version__
)

print()
print(
    "RESUME VERIFICATION: PASS"
)

print(
    "SAFE TO RUN CELL 6."
)

print("=" * 88)

In [ ]:
# Experiment D: VQC depth scalability

# Main Experiment A reference:
#   4 qubits
#   3 layers
#   24 trainable quantum parameters

# New:
#   D_Layers_1 -> 8 parameters
#   D_Layers_4 -> 32 parameters

# QTrust-PPO-128 only
# CartPole-v1
# 10 seeds per depth

# Total New Runs = 20

# Execution:
#   2 × Tesla T4
#   4 independent workers

# Recovery:
#   Atomic save after every completed run
#   Automatic resume



from pathlib import Path
from datetime import datetime, timezone

import os
import sys
import json
import time
import zipfile
import subprocess
import py_compile

import numpy as np
import pandas as pd



# REQUIRE CELLS 1–5


required = [
    "JOURNAL_ROOT",
    "EXP_D_DIR",
    "LOGS_DIR",
    "RUN_MANIFEST",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1–5 first. Missing: "
        + ", ".join(missing)
    )


print("=" * 90)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 6 / 10")
print("EXPERIMENT D — VQC DEPTH SCALABILITY")
print("=" * 90)



# DIRECTORIES


D_RUNS_DIR = (
    EXP_D_DIR
    / "runs"
)

D_UPDATES_DIR = (
    EXP_D_DIR
    / "updates"
)

D_EVAL_DIR = (
    EXP_D_DIR
    / "evaluation"
)

D_LOG_DIR = (
    EXP_D_DIR
    / "worker_logs"
)

D_TASK_DIR = (
    EXP_D_DIR
    / "worker_tasks"
)


for directory in [
    D_RUNS_DIR,
    D_UPDATES_DIR,
    D_EVAL_DIR,
    D_LOG_DIR,
    D_TASK_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )






D_MANIFEST = (
    RUN_MANIFEST[
        RUN_MANIFEST[
            "experiment"
        ]
        == "D"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(
    D_MANIFEST
) != 20:

    raise RuntimeError(
        f"Experiment D must contain exactly 20 runs. "
        f"Found {len(D_MANIFEST)}."
    )


EXPECTED_VARIANTS = {
    "D_Layers_1",
    "D_Layers_4",
}


if set(
    D_MANIFEST[
        "variant"
    ].unique()
) != EXPECTED_VARIANTS:

    raise RuntimeError(
        "Experiment-D variant mismatch."
    )


if set(
    D_MANIFEST[
        "method"
    ].unique()
) != {
    "QTrust-PPO-128"
}:

    raise RuntimeError(
        "Experiment D must use QTrust-PPO-128 only."
    )



# VERIFY DEPTH / PARAMETER DESIGN BEFORE RUNNING


expected_layer_map = {

    "D_Layers_1":
        1,

    "D_Layers_4":
        4,
}


expected_parameter_map = {

    "D_Layers_1":
        8,

    "D_Layers_4":
        32,
}


for variant, expected_layers in (
    expected_layer_map.items()
):

    observed = set(
        D_MANIFEST[
            D_MANIFEST[
                "variant"
            ]
            == variant
        ][
            "layers"
        ]
        .astype(int)
        .tolist()
    )


    if observed != {
        expected_layers
    }:

        raise RuntimeError(
            f"Layer mismatch for {variant}: "
            f"{observed}"
        )


print(
    "Depth design:"
)

print(
    "  1 layer -> 8 trainable quantum parameters"
)

print(
    "  3 layers -> 24 parameters "
    "(reference from Experiment A; NOT rerun)"
)

print(
    "  4 layers -> 32 trainable quantum parameters"
)



# COMPLETION / RESUME VALIDATION


def D_summary_path(
    run_id,
):

    return (
        D_RUNS_DIR
        / f"{run_id}.json"
    )


def valid_D_run(
    run_id,
):

    path = (
        D_summary_path(
            run_id
        )
    )


    if not path.exists():

        return False


    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )


        return (

            payload.get(
                "status"
            )
            == "COMPLETE"

            and

            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH

            and

            payload.get(
                "parent_protocol_hash"
            )
            == EXPECTED_PROTOCOL_HASH

            and

            payload.get(
                "run_id"
            )
            == run_id
        )


    except Exception:

        return False


completed_before = [

    run_id

    for run_id
    in D_MANIFEST[
        "run_id"
    ]

    if valid_D_run(
        run_id
    )
]


pending_manifest = (

    D_MANIFEST[
        ~D_MANIFEST[
            "run_id"
        ].isin(
            completed_before
        )
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print()
print(
    "Completed before Cell 6 :",
    len(
        completed_before
    ),
    "/ 20"
)

print(
    "Pending Experiment-D runs:",
    len(
        pending_manifest
    )
)



# WORKER SOURCE


WORKER_SOURCE = r'''
import os
import sys
import json
import time
import traceback

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch


if len(
    sys.argv
) != 4:

    raise RuntimeError(
        "Usage: expD_worker.py "
        "<task_file> <journal_root> <journal_hash>"
    )


TASK_FILE = Path(
    sys.argv[
        1
    ]
)


JOURNAL_ROOT = Path(
    sys.argv[
        2
    ]
)


JOURNAL_PROTOCOL_HASH = (
    sys.argv[
        3
    ]
)


PARENT_PROTOCOL_HASH = (
    "3cf4bcb18e1111b3"
)


# ============================================================
# RESOURCE GUARDS
# ============================================================

os.environ[
    "OMP_NUM_THREADS"
] = "1"

os.environ[
    "MKL_NUM_THREADS"
] = "1"

os.environ[
    "OPENBLAS_NUM_THREADS"
] = "1"

os.environ[
    "NUMEXPR_NUM_THREADS"
] = "1"

os.environ[
    "CUDA_MODULE_LOADING"
] = "LAZY"


torch.set_num_threads(
    1
)


try:

    torch.set_num_interop_threads(
        1
    )

except Exception:

    pass


# ============================================================
# LOAD VERIFIED JOURNAL ENGINE
# ============================================================

sys.path.insert(
    0,
    str(
        JOURNAL_ROOT
    ),
)


import journal_engine as qe


if (
    qe.PARENT_PROTOCOL_HASH
    != PARENT_PROTOCOL_HASH
):

    raise RuntimeError(
        "Parent protocol mismatch."
    )


if (
    qe.ANALYTIC_QDEVICE_NAME
    != "lightning.gpu"
):

    raise RuntimeError(
        "lightning.gpu unavailable."
    )


# ============================================================
# PATHS
# ============================================================

EXP_D_DIR = (
    JOURNAL_ROOT
    / "experiment_D_scalability"
)


RUNS_DIR = (
    EXP_D_DIR
    / "runs"
)


UPDATES_DIR = (
    EXP_D_DIR
    / "updates"
)


EVAL_DIR = (
    EXP_D_DIR
    / "evaluation"
)


for directory in [
    RUNS_DIR,
    UPDATES_DIR,
    EVAL_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# ATOMIC WRITES
# ============================================================

def atomic_json(
    path,
    payload,
):

    path = Path(
        path
    )


    temporary = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )


    os.replace(
        temporary,
        path,
    )


def atomic_csv(
    path,
    dataframe,
):

    path = Path(
        path
    )


    temporary = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary,
        index=False,
    )


    os.replace(
        temporary,
        path,
    )


def valid_existing(
    run_id,
):

    path = (
        RUNS_DIR
        / f"{run_id}.json"
    )


    if not path.exists():

        return False


    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )


        return (

            payload.get(
                "status"
            )
            == "COMPLETE"

            and

            payload.get(
                "journal_protocol_hash"
            )
            == JOURNAL_PROTOCOL_HASH

            and

            payload.get(
                "parent_protocol_hash"
            )
            == PARENT_PROTOCOL_HASH
        )


    except Exception:

        return False


# ============================================================
# LOAD TASKS
# ============================================================

tasks = json.loads(
    TASK_FILE.read_text(
        encoding="utf-8"
    )
)


print(
    "[worker] CUDA_VISIBLE_DEVICES =",
    os.environ.get(
        "CUDA_VISIBLE_DEVICES"
    )
)


print(
    "[worker] tasks =",
    len(
        tasks
    )
)


# ============================================================
# RUN ASSIGNED TASKS
# ============================================================

for task_number, task in enumerate(
    tasks,
    start=1,
):

    run_id = (
        task[
            "run_id"
        ]
    )


    if valid_existing(
        run_id
    ):

        print(
            "[skip]",
            run_id
        )

        continue


    run_start = (
        time.perf_counter()
    )


    try:

        seed = int(
            task[
                "seed"
            ]
        )


        layers = int(
            task[
                "layers"
            ]
        )


        qubits = int(
            task[
                "qubits"
            ]
        )


        variant = str(
            task[
                "variant"
            ]
        )


        expected_parameters = (
            layers
            * qubits
            * 2
        )


        print(
            f"[{task_number}/{len(tasks)}] "
            f"START {run_id} | "
            f"layers={layers} | "
            f"parameters={expected_parameters}"
        )


        # ====================================================
        # TRAIN QTRUST
        # ====================================================

        (
            model,
            update_df,
            train_stats,
        ) = qe.train_qtrust_journal(

            env_name=
                "CartPole-v1",

            base_shots=
                128,

            total_steps=
                2048,

            rollout_steps=
                32,

            seed=
                seed,

            update_epochs=
                1,

            n_qubits=
                qubits,

            n_layers=
                layers,

            delta=
                0.05,

            rollback_on_unresolved=
                True,

            shot_schedule=[
                32,
                64,
                128,
                256,
                512,
                1024,
                2048,
                4096,
            ],
        )


        # ====================================================
        # HARD PARAMETER COUNT CHECK
        # ====================================================

        observed_parameters = int(
            model.actor
            .quantum_weights
            .numel()
        )


        if (
            observed_parameters
            != expected_parameters
        ):

            raise RuntimeError(
                f"Parameter count mismatch: "
                f"expected {expected_parameters}, "
                f"found {observed_parameters}."
            )


        # ====================================================
        # FINAL EVALUATION — 30 EPISODES
        # ====================================================

        evaluation_seeds = (
            qe.make_final_eval_seeds(

                training_seed=
                    seed,

                n_episodes=
                    30,
            )
        )


        (
            evaluation_df,
            evaluation_stats,
        ) = (
            qe.evaluate_finite_shot_policy_journal(

                model=
                    model,

                env_name=
                    "CartPole-v1",

                seeds=
                    evaluation_seeds,

                deterministic=
                    True,
            )
        )


        # ====================================================
        # DIAGNOSTICS
        # ====================================================

        mean_unresolved = float(
            update_df[
                "unresolved_fraction"
            ].mean()
        )


        accuracy_values = (
            update_df[
                "classification_accuracy"
            ]
            .dropna()
        )


        classification_accuracy = (

            float(
                accuracy_values.mean()
            )

            if len(
                accuracy_values
            )

            else None
        )


        mean_adaptive_shots = float(
            update_df[
                "mean_qtrust_shots"
            ].mean()
        )


        mean_approx_kl = float(
            update_df[
                "approx_kl"
            ].mean()
        )


        mean_clip_fraction = float(
            update_df[
                "clip_fraction"
            ].mean()
        )


        # ====================================================
        # SAVE LARGE ARTIFACTS FIRST
        # ====================================================

        update_path = (
            UPDATES_DIR
            / f"{run_id}.csv"
        )


        evaluation_path = (
            EVAL_DIR
            / f"{run_id}.csv"
        )


        atomic_csv(
            update_path,
            update_df,
        )


        atomic_csv(
            evaluation_path,
            evaluation_df,
        )


        wall_time = float(
            time.perf_counter()
            - run_start
        )


        # ====================================================
        # SUMMARY COMPLETION MARKER LAST
        # ====================================================

        summary = {

            "status":
                "COMPLETE",

            "experiment":
                "D",

            "run_id":
                run_id,

            "variant":
                variant,

            "environment":
                "CartPole-v1",

            "method":
                "QTrust-PPO-128",

            "seed":
                seed,

            "parent_protocol_hash":
                PARENT_PROTOCOL_HASH,

            "journal_protocol_hash":
                JOURNAL_PROTOCOL_HASH,

            "qubits":
                qubits,

            "layers":
                layers,

            "quantum_parameters":
                observed_parameters,

            "base_shots":
                128,

            "delta":
                0.05,

            "rollback_on_unresolved":
                True,

            "training_steps":
                2048,

            "rollout_steps":
                32,

            "evaluation_episodes":
                30,

            "mean_eval_reward":
                float(
                    evaluation_stats[
                        "mean_eval_reward"
                    ]
                ),

            "std_eval_reward":
                float(
                    evaluation_stats[
                        "std_eval_reward"
                    ]
                ),

            "median_eval_reward":
                float(
                    evaluation_stats[
                        "median_eval_reward"
                    ]
                ),

            "mean_eval_steps":
                float(
                    evaluation_stats[
                        "mean_eval_steps"
                    ]
                ),

            "accepted_updates":
                int(
                    train_stats[
                        "accepted_updates"
                    ]
                ),

            "rolled_back_updates":
                int(
                    train_stats[
                        "rolled_back_updates"
                    ]
                ),

            "mean_unresolved_fraction":
                mean_unresolved,

            "classification_accuracy":
                classification_accuracy,

            "mean_adaptive_shots":
                mean_adaptive_shots,

            "mean_approx_kl":
                mean_approx_kl,

            "mean_clip_fraction":
                mean_clip_fraction,

            "action_shots":
                int(
                    train_stats[
                        "action_shots"
                    ]
                ),

            "gradient_shots":
                int(
                    train_stats[
                        "gradient_shots"
                    ]
                ),

            "certification_shots":
                int(
                    train_stats[
                        "certification_shots"
                    ]
                ),

            "training_quantum_shots":
                int(
                    train_stats[
                        "total_quantum_shots"
                    ]
                ),

            "evaluation_quantum_shots":
                int(
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "overall_quantum_shots":
                int(
                    train_stats[
                        "total_quantum_shots"
                    ]
                    +
                    evaluation_stats[
                        "evaluation_quantum_shots"
                    ]
                ),

            "training_runtime_seconds":
                float(
                    train_stats[
                        "elapsed_seconds"
                    ]
                ),

            "evaluation_runtime_seconds":
                float(
                    evaluation_stats[
                        "evaluation_runtime_seconds"
                    ]
                ),

            "wall_runtime_seconds":
                wall_time,

            "updates_file":
                str(
                    update_path.relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "evaluation_file":
                str(
                    evaluation_path.relative_to(
                        JOURNAL_ROOT
                    )
                ),

            "completed_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }


        atomic_json(

            RUNS_DIR
            / f"{run_id}.json",

            summary,
        )


        print(
            f"[complete] {run_id} | "
            f"layers={layers} | "
            f"params={observed_parameters} | "
            f"reward="
            f"{summary['mean_eval_reward']:.3f} | "
            f"wall="
            f"{wall_time / 60:.1f} min"
        )


        del model


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    except Exception as exc:

        atomic_json(

            RUNS_DIR
            / f"{run_id}.failed.json",

            {

                "status":
                    "FAILED",

                "run_id":
                    run_id,

                "error_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    str(
                        exc
                    ),

                "timestamp_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
        )


        traceback.print_exc()

        raise


print(
    "[worker] ALL ASSIGNED TASKS COMPLETE"
)
'''



# SAVE + COMPILE WORKER BEFORE ANY GPU JOB


WORKER_PATH = (
    EXP_D_DIR
    / "expD_worker.py"
)


WORKER_PATH.write_text(
    WORKER_SOURCE,
    encoding="utf-8",
)


py_compile.compile(
    str(
        WORKER_PATH
    ),
    doraise=True,
)


print(
    "Experiment-D worker syntax check: PASS"
)



# FOUR BALANCED WORKER QUEUES


WORKER_GPU_MAP = [
    0,
    1,
    0,
    1,
]


TOTAL_WORKERS = 4


tasks = (
    pending_manifest
    .to_dict(
        orient="records"
    )
)


worker_queues = [
    []
    for _ in range(
        TOTAL_WORKERS
    )
]


for index, task in enumerate(
    tasks
):

    clean_task = {}


    for key, value in task.items():

        if pd.isna(
            value
        ):

            clean_task[
                key
            ] = None


        elif isinstance(
            value,
            np.generic,
        ):

            clean_task[
                key
            ] = value.item()


        else:

            clean_task[
                key
            ] = value


    worker_queues[
        index
        % TOTAL_WORKERS
    ].append(
        clean_task
    )


task_files = []


for worker_id, queue in enumerate(
    worker_queues
):

    task_file = (
        D_TASK_DIR
        / f"worker_{worker_id}_tasks.json"
    )


    task_file.write_text(
        json.dumps(
            queue,
            indent=2,
        ),
        encoding="utf-8",
    )


    task_files.append(
        task_file
    )


print()
print(
    "Worker assignment:"
)


for worker_id, (
    gpu,
    queue,
) in enumerate(
    zip(
        WORKER_GPU_MAP,
        worker_queues,
    )
):

    print(
        f"  Worker {worker_id} "
        f"-> T4 GPU {gpu} "
        f"-> {len(queue)} runs"
    )






D_RECOVERY_ZIP = (
    Path(
        "/kaggle/working"
    )
    / "QTrust_Journal_Experiment_D_LATEST.zip"
)


def save_D_recovery_zip():

    temporary = (
        D_RECOVERY_ZIP
        .with_suffix(
            ".zip.tmp"
        )
    )


    if temporary.exists():

        temporary.unlink()


    with zipfile.ZipFile(

        temporary,

        "w",

        compression=
            zipfile.ZIP_DEFLATED,

        compresslevel=1,

    ) as zf:


        for file in sorted(
            EXP_D_DIR.rglob("*")
        ):

            if not file.is_file():

                continue


            if file.suffix == ".tmp":

                continue


            relative = (
                file.relative_to(
                    JOURNAL_ROOT
                )
            )


            zf.write(

                file,

                arcname=str(
                    Path(
                        JOURNAL_ROOT.name
                    )
                    / relative
                ),
            )


        for core_name in [

            "journal_manifest.json",
            "journal_extension_protocol.json",
            "journal_run_manifest.csv",
            "journal_engine.py",
            "JOURNAL_EXPERIMENT_PLAN.txt",

        ]:

            path = (
                JOURNAL_ROOT
                / core_name
            )


            if path.exists():

                zf.write(

                    path,

                    arcname=str(
                        Path(
                            JOURNAL_ROOT.name
                        )
                        / core_name
                    ),
                )


    with zipfile.ZipFile(
        temporary,
        "r",
    ) as zf:

        bad = (
            zf.testzip()
        )


    if bad is not None:

        temporary.unlink(
            missing_ok=True
        )


        raise RuntimeError(
            "Experiment-D recovery ZIP "
            f"failed verification: {bad}"
        )


    os.replace(
        temporary,
        D_RECOVERY_ZIP,
    )



# START FOUR WORKERS


processes = []


if len(
    pending_manifest
) > 0:

    print()
    print("=" * 90)
    print("STARTING EXPERIMENT D")
    print("=" * 90)


    for worker_id, (
        gpu,
        task_file,
    ) in enumerate(
        zip(
            WORKER_GPU_MAP,
            task_files,
        )
    ):

        if not worker_queues[
            worker_id
        ]:

            continue


        worker_env = (
            os.environ.copy()
        )


        worker_env[
            "CUDA_VISIBLE_DEVICES"
        ] = str(
            gpu
        )


        worker_env[
            "OMP_NUM_THREADS"
        ] = "1"


        worker_env[
            "MKL_NUM_THREADS"
        ] = "1"


        worker_env[
            "OPENBLAS_NUM_THREADS"
        ] = "1"


        worker_env[
            "NUMEXPR_NUM_THREADS"
        ] = "1"


        worker_env[
            "CUDA_MODULE_LOADING"
        ] = "LAZY"


        log_path = (
            D_LOG_DIR
            / f"worker_{worker_id}.log"
        )


        log_handle = open(
            log_path,
            "w",
            buffering=1,
            encoding="utf-8",
        )


        process = subprocess.Popen(

            [
                sys.executable,
                str(
                    WORKER_PATH
                ),
                str(
                    task_file
                ),
                str(
                    JOURNAL_ROOT
                ),
                JOURNAL_PROTOCOL_HASH,
            ],

            env=
                worker_env,

            stdout=
                log_handle,

            stderr=
                subprocess.STDOUT,

            text=True,
        )


        processes.append(
            {

                "worker_id":
                    worker_id,

                "gpu":
                    gpu,

                "process":
                    process,

                "log_handle":
                    log_handle,

                "log_path":
                    log_path,
            }
        )



# PROGRESS MONITOR


def completed_D_runs():

    return [

        run_id

        for run_id
        in D_MANIFEST[
            "run_id"
        ]

        if valid_D_run(
            run_id
        )
    ]


last_count = (
    len(
        completed_before
    )
)


last_zip_count = -1


if last_count > 0:

    save_D_recovery_zip()

    last_zip_count = (
        last_count
    )


while any(

    item[
        "process"
    ].poll()
    is None

    for item
    in processes

):

    current = len(
        completed_D_runs()
    )


    if current != last_count:

        print(
            f"[progress] Experiment D: "
            f"{current}/20 complete "
            f"({100.0 * current / 20:.1f}%)"
        )


        last_count = (
            current
        )


        if (
            current
            != last_zip_count
        ):

            save_D_recovery_zip()

            last_zip_count = (
                current
            )


    time.sleep(
        20
    )


for item in processes:

    item[
        "log_handle"
    ].close()



# CHECK WORKER FAILURES


failed_workers = [

    item

    for item in processes

    if item[
        "process"
    ].returncode
    != 0
]


if failed_workers:

    save_D_recovery_zip()


    print()
    print("=" * 90)
    print("ONE OR MORE EXPERIMENT-D WORKERS FAILED")
    print("=" * 90)


    for item in failed_workers:

        print()
        print(
            f"Worker {item['worker_id']} "
            f"(GPU {item['gpu']})"
        )


        print(
            "Log:",
            item[
                "log_path"
            ]
        )


        try:

            tail = (
                item[
                    "log_path"
                ]
                .read_text(
                    encoding="utf-8"
                )
                .splitlines()[
                    -60:
                ]
            )


            print(
                "\n".join(
                    tail
                )
            )


        except Exception:

            pass


    raise RuntimeError(
        "Experiment D stopped because a worker failed. "
        "Completed runs remain saved. "
        "Rerun Cell 6 to resume."
    )



# REQUIRE 20 / 20


completed_final = (
    completed_D_runs()
)


if len(
    completed_final
) != 20:

    save_D_recovery_zip()


    raise RuntimeError(
        "Experiment D expected 20 COMPLETE runs, "
        f"found {len(completed_final)}."
    )



# LOAD ALL 20 RESULTS


rows = []


for run_id in D_MANIFEST[
    "run_id"
]:

    rows.append(

        json.loads(

            D_summary_path(
                run_id
            )
            .read_text(
                encoding="utf-8"
            )
        )
    )


D_RESULTS = pd.DataFrame(
    rows
)



# HARD VALIDATION


if len(
    D_RESULTS
) != 20:

    raise RuntimeError(
        "Experiment-D result count != 20."
    )


if not D_RESULTS[
    "status"
].eq(
    "COMPLETE"
).all():

    raise RuntimeError(
        "Experiment D contains incomplete runs."
    )


if not D_RESULTS[
    "journal_protocol_hash"
].eq(
    JOURNAL_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Journal hash mismatch in Experiment D."
    )


if not D_RESULTS[
    "parent_protocol_hash"
].eq(
    EXPECTED_PROTOCOL_HASH
).all():

    raise RuntimeError(
        "Parent hash mismatch in Experiment D."
    )


counts = (
    D_RESULTS
    .groupby(
        "variant"
    )
    .size()
)


if (
    len(counts)
    != 2
    or
    not counts.eq(
        10
    ).all()
):

    raise RuntimeError(
        "Each depth must contain exactly 10 seeds."
    )


EXPECTED_SEED_LIST = [
    11,
    23,
    37,
    53,
    71,
    89,
    107,
    131,
    157,
    181,
]


for variant, group in (
    D_RESULTS.groupby(
        "variant"
    )
):

    seeds = sorted(
        group[
            "seed"
        ]
        .astype(int)
        .tolist()
    )


    if seeds != EXPECTED_SEED_LIST:

        raise RuntimeError(
            f"Seed mismatch for {variant}."
        )


parameter_counts = (

    D_RESULTS

    .groupby(
        "variant"
    )[
        "quantum_parameters"
    ]

    .unique()
)


for variant in (
    expected_parameter_map
):

    observed = list(
        parameter_counts[
            variant
        ]
    )


    expected = (
        expected_parameter_map[
            variant
        ]
    )


    if observed != [
        expected
    ]:

        raise RuntimeError(
            f"Parameter-count mismatch for "
            f"{variant}: {observed}"
        )



# SAVE COMPLETE EXPERIMENT-D TABLE


D_RESULTS_PATH = (
    EXP_D_DIR
    / "experiment_D_all_runs.csv"
)


D_RESULTS.to_csv(
    D_RESULTS_PATH,
    index=False,
)



# DESCRIPTIVE SUMMARY


D_SUMMARY = (

    D_RESULTS

    .groupby(
        [
            "variant",
            "layers",
            "quantum_parameters",
        ],
        as_index=False,
    )

    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),

        Mean_Wall_Runtime_s=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


D_SUMMARY_PATH = (
    EXP_D_DIR
    / "experiment_D_summary.csv"
)


D_SUMMARY.to_csv(
    D_SUMMARY_PATH,
    index=False,
)



# UPDATE RUN MANIFEST


RUN_MANIFEST.loc[
    RUN_MANIFEST[
        "experiment"
    ]
    == "D",
    "status",
] = "COMPLETE"


RUN_MANIFEST.to_csv(
    JOURNAL_ROOT
    / "journal_run_manifest.csv",
    index=False,
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "experiment_D_scalability"
] = {

    "status":
        "COMPLETE",

    "completed_runs":
        20,

    "new_depths":
        [
            1,
            4,
        ],

    "reference_depth":
        3,

    "reference_source":
        "Experiment A",

    "qubits":
        4,

    "seeds_per_new_depth":
        10,

    "result_file":
        str(
            D_RESULTS_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),

    "summary_file":
        str(
            D_SUMMARY_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),
}


manifest[
    "current_completed_cell"
] = 6


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)



# CELL-6 PROVENANCE


cell6_record = {

    "cell":
        6,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        "D",

    "completed_runs":
        20,

    "new_depths":
        [
            1,
            4,
        ],

    "reference_depth":
        3,

    "reference_source":
        "Experiment A",

    "seeds_per_depth":
        10,

    "workers":
        4,

    "physical_gpus":
        2,

    "worker_gpu_map":
        WORKER_GPU_MAP,

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,
}


(
    LOGS_DIR
    / "cell06_experiment_D.json"
).write_text(
    json.dumps(
        cell6_record,
        indent=2,
    ),
    encoding="utf-8",
)






save_D_recovery_zip()



# REPORT


print()
print("=" * 90)
print("EXPERIMENT D COMPLETE")
print("=" * 90)


print(
    "Completed runs :",
    len(
        D_RESULTS
    ),
    "/ 20"
)


print(
    "New depths     : 1, 4"
)


print(
    "Reference depth: 3 "
    "(Experiment A)"
)


print(
    "Qubits         : 4"
)


print(
    "Seeds/depth    : 10"
)


print(
    "GPUs           : 2 × Tesla T4"
)


print(
    "Workers        : 4"
)


print(
    "Journal hash   :",
    JOURNAL_PROTOCOL_HASH
)


print()
print(
    D_SUMMARY.to_string(
        index=False
    )
)


print()
print(
    "Recovery ZIP:",
    D_RECOVERY_ZIP
)


print("=" * 90)






CELL6_CHECKPOINT = (
    save_journal_checkpoint(
        6,
        "Experiment_D_Scalability_COMPLETE",
    )
)


print()
print("=" * 90)
print("CELL 6 COMPLETE")
print("=" * 90)


print(
    "Experiment D is now frozen."
)


print(
    "ALL NEW GPU TRAINING EXPERIMENTS ARE COMPLETE."
)


print(
    "Next: Cell 7 will run Experiment E, "
    "the controlled confidence-bound benchmark."
)


print(
    "Cell 7 is CPU-oriented and should be "
    "much faster than Cells 4–6."
)

In [ ]:
# Experiment E: confidence-bound benchmark

# Compare:
# Exact Clopper-Pearson
# One-sided Wilson score bounds
# One-sided Hoeffding bounds

# SAME:
#   - controlled PPO-boundary cases as Experiment A
#   - QTrust adaptive schedule
#   - delta = 0.05
#   - four one-sided tails per stage
#   - Bonferroni across finite horizon

# CPU ONLY


from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib

import numpy as np
import pandas as pd

from scipy.stats import beta, norm






required = [
    "JOURNAL_ROOT",
    "EXP_E_DIR",
    "LOGS_DIR",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing journal state: "
        + ", ".join(missing)
    )


print("=" * 90)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 7 / 10")
print("EXPERIMENT E — CONFIDENCE-BOUND BENCHMARK")
print("=" * 90)






RATIO_GRID = [
    0.70,
    0.75,
    0.78,
    0.79,
    0.80,
    0.81,
    0.82,
    0.85,
    0.90,
    1.00,
    1.10,
    1.15,
    1.18,
    1.19,
    1.20,
    1.21,
    1.22,
    1.25,
    1.30,
]

P_OLD_VALUES = [
    0.30,
    0.50,
    0.70,
]

SHOT_SCHEDULE = [
    32,
    64,
    128,
    256,
    512,
    1024,
    2048,
    4096,
]

DELTA = 0.05
CLIP_EPSILON = 0.20

REPEATS = 200
MASTER_SEED_E = 20260920

METHODS_E = [
    "Clopper-Pearson",
    "Wilson",
    "Hoeffding",
]



# FREEZE EXPERIMENT-E CONFIG BEFORE RESULTS


E_CONFIG = {

    "experiment":
        "E",

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "random_seed":
        MASTER_SEED_E,

    "ratio_grid":
        RATIO_GRID,

    "p_old_values":
        P_OLD_VALUES,

    "shot_schedule":
        SHOT_SCHEDULE,

    "delta":
        DELTA,

    "clip_epsilon":
        CLIP_EPSILON,

    "repeats":
        REPEATS,

    "confidence_methods":
        METHODS_E,

    "error_allocation":
        (
            "delta/K per stage; "
            "stage budget divided across "
            "four one-sided probability tails"
        ),

    "sampling":
        (
            "Common random binomial sample stream "
            "shared across confidence methods "
            "for each case and repeat."
        ),

    "truth_definition":
        (
            "Positive advantage clips only when r > 1.2; "
            "negative advantage clips only when r < 0.8."
        ),
}


E_CONFIG_HASH = (
    hashlib.sha256(
        json.dumps(
            E_CONFIG,
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    )
    .hexdigest()[:16]
)


EXP_E_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


(
    EXP_E_DIR
    / "experiment_E_protocol.json"
).write_text(
    json.dumps(
        {
            **E_CONFIG,
            "experiment_E_hash":
                E_CONFIG_HASH,
        },
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "Experiment-E hash:",
    E_CONFIG_HASH
)



# ONE-SIDED CONFIDENCE BOUNDS


def cp_bounds(
    successes,
    trials,
    alpha,
):

    k = int(successes)
    n = int(trials)


    lower = (
        0.0
        if k == 0
        else float(
            beta.ppf(
                alpha,
                k,
                n - k + 1,
            )
        )
    )


    upper = (
        1.0
        if k == n
        else float(
            beta.ppf(
                1.0 - alpha,
                k + 1,
                n - k,
            )
        )
    )


    return (
        lower,
        upper,
    )


def wilson_bounds(
    successes,
    trials,
    alpha,
):

    k = float(successes)
    n = float(trials)

    phat = k / n

    z = float(
        norm.ppf(
            1.0 - alpha
        )
    )


    denominator = (
        1.0
        + (z * z) / n
    )


    center = (
        phat
        + (z * z) / (2.0 * n)
    ) / denominator


    half_width = (
        z
        / denominator
        * np.sqrt(
            (
                phat
                * (1.0 - phat)
                / n
            )
            +
            (
                z * z
                / (4.0 * n * n)
            )
        )
    )


    lower = max(
        0.0,
        float(
            center - half_width
        ),
    )


    upper = min(
        1.0,
        float(
            center + half_width
        ),
    )


    return (
        lower,
        upper,
    )


def hoeffding_bounds(
    successes,
    trials,
    alpha,
):

    n = float(trials)

    phat = (
        float(successes)
        / n
    )


    radius = np.sqrt(
        np.log(
            1.0 / alpha
        )
        / (2.0 * n)
    )


    lower = max(
        0.0,
        phat - radius,
    )


    upper = min(
        1.0,
        phat + radius,
    )


    return (
        float(lower),
        float(upper),
    )


BOUND_FUNCTIONS = {

    "Clopper-Pearson":
        cp_bounds,

    "Wilson":
        wilson_bounds,

    "Hoeffding":
        hoeffding_bounds,
}



# RATIO CONFIDENCE BOUNDS


def ratio_bounds(
    method,
    new_successes,
    old_successes,
    trials,
    tail_alpha,
    eps=1e-12,
):

    bound_function = (
        BOUND_FUNCTIONS[
            method
        ]
    )


    new_low, new_high = (
        bound_function(
            new_successes,
            trials,
            tail_alpha,
        )
    )


    old_low, old_high = (
        bound_function(
            old_successes,
            trials,
            tail_alpha,
        )
    )


    ratio_low = (
        new_low
        / max(
            old_high,
            eps,
        )
    )


    ratio_high = (

        np.inf

        if old_low <= eps

        else (
            new_high
            / old_low
        )
    )


    return (
        float(ratio_low),
        float(ratio_high),
    )



# ADVANTAGE-AWARE DECISION


def clipping_decision(
    ratio_low,
    ratio_high,
    advantage,
):

    lower_boundary = (
        1.0
        - CLIP_EPSILON
    )

    upper_boundary = (
        1.0
        + CLIP_EPSILON
    )


    if advantage >= 0:

        if (
            ratio_high
            <= upper_boundary
        ):

            return (
                "SAFE_NO_UPPER_CLIP"
            )


        if (
            ratio_low
            > upper_boundary
        ):

            return (
                "CONFIRMED_UPPER_CLIP"
            )


        return "UNCERTAIN"


    else:

        if (
            ratio_low
            >= lower_boundary
        ):

            return (
                "SAFE_NO_LOWER_CLIP"
            )


        if (
            ratio_high
            < lower_boundary
        ):

            return (
                "CONFIRMED_LOWER_CLIP"
            )


        return "UNCERTAIN"



# BUILD SAME CONTROLLED CASES AS EXPERIMENT A


cases = []


for p_old_a in P_OLD_VALUES:

    for ratio in RATIO_GRID:

        p_new_a = (
            p_old_a
            * ratio
        )


        if not (
            0.0
            < p_new_a
            < 1.0
        ):

            continue


        for advantage in [
            +1.0,
            -1.0,
        ]:

            if advantage >= 0:

                true_clip = (
                    ratio
                    > 1.20
                )


            else:

                true_clip = (
                    ratio
                    < 0.80
                )


            cases.append(
                {

                    "p_old":
                        float(
                            p_old_a
                        ),

                    "p_new":
                        float(
                            p_new_a
                        ),

                    "ratio":
                        float(
                            ratio
                        ),

                    "advantage":
                        float(
                            advantage
                        ),

                    "true_clip":
                        bool(
                            true_clip
                        ),
                }
            )


EXPECTED_CASES = (
    len(P_OLD_VALUES)
    * len(RATIO_GRID)
    * 2
)


if len(
    cases
) != EXPECTED_CASES:

    raise RuntimeError(
        f"Controlled-case count mismatch: "
        f"{len(cases)} != {EXPECTED_CASES}"
    )


print(
    "Controlled cases :",
    len(cases)
)

print(
    "Repeats/case     :",
    REPEATS
)

print(
    "Trial paths      :",
    len(cases)
    * REPEATS
)



# RUN PAIRED COMMON-RANDOM BENCHMARK


K = len(
    SHOT_SCHEDULE
)

STAGE_ALPHA = (
    DELTA
    / K
)

TAIL_ALPHA = (
    STAGE_ALPHA
    / 4.0
)


records = []


for case_index, case in enumerate(
    cases
):

    for repeat in range(
        REPEATS
    ):

        base_seed = (
            MASTER_SEED_E
            + case_index
            * 100_000
            + repeat
        )


        rng_old = (
            np.random.default_rng(
                base_seed
            )
        )


        rng_new = (
            np.random.default_rng(
                base_seed
                + 50_000_000
            )
        )


        old_counts = {}
        new_counts = {}

        old_successes = 0
        new_successes = 0
        previous_shots = 0


        # Generate one common cumulative sample path.
        for shots in SHOT_SCHEDULE:

            additional = (
                shots
                - previous_shots
            )


            old_successes += int(
                rng_old.binomial(
                    additional,
                    case[
                        "p_old"
                    ],
                )
            )


            new_successes += int(
                rng_new.binomial(
                    additional,
                    case[
                        "p_new"
                    ],
                )
            )


            old_counts[
                shots
            ] = old_successes


            new_counts[
                shots
            ] = new_successes


            previous_shots = (
                shots
            )


        # Run each bound family on exactly the same samples.
        for method in METHODS_E:

            final_decision = (
                "UNCERTAIN"
            )

            final_shots = (
                SHOT_SCHEDULE[-1]
            )

            final_ratio_low = (
                np.nan
            )

            final_ratio_high = (
                np.nan
            )


            for shots in SHOT_SCHEDULE:

                (
                    ratio_low,
                    ratio_high,
                ) = ratio_bounds(

                    method=
                        method,

                    new_successes=
                        new_counts[
                            shots
                        ],

                    old_successes=
                        old_counts[
                            shots
                        ],

                    trials=
                        shots,

                    tail_alpha=
                        TAIL_ALPHA,
                )


                decision = (
                    clipping_decision(

                        ratio_low=
                            ratio_low,

                        ratio_high=
                            ratio_high,

                        advantage=
                            case[
                                "advantage"
                            ],
                    )
                )


                final_ratio_low = (
                    ratio_low
                )

                final_ratio_high = (
                    ratio_high
                )


                if (
                    decision
                    != "UNCERTAIN"
                ):

                    final_decision = (
                        decision
                    )

                    final_shots = (
                        shots
                    )

                    break


            resolved = (
                final_decision
                != "UNCERTAIN"
            )


            if resolved:

                predicted_clip = (
                    final_decision
                    in {
                        "CONFIRMED_UPPER_CLIP",
                        "CONFIRMED_LOWER_CLIP",
                    }
                )


                correct = bool(
                    predicted_clip
                    == case[
                        "true_clip"
                    ]
                )


                false_positive = bool(
                    predicted_clip
                    and not case[
                        "true_clip"
                    ]
                )


                false_negative = bool(
                    (
                        not predicted_clip
                    )
                    and case[
                        "true_clip"
                    ]
                )


            else:

                predicted_clip = None
                correct = None
                false_positive = False
                false_negative = False


            records.append(
                {

                    "Case":
                        case_index,

                    "Repeat":
                        repeat,

                    "Method":
                        method,

                    "p_old":
                        case[
                            "p_old"
                        ],

                    "p_new":
                        case[
                            "p_new"
                        ],

                    "True Ratio":
                        case[
                            "ratio"
                        ],

                    "Advantage":
                        case[
                            "advantage"
                        ],

                    "True Clip":
                        case[
                            "true_clip"
                        ],

                    "Decision":
                        final_decision,

                    "Resolved":
                        resolved,

                    "Correct":
                        correct,

                    "False Positive":
                        false_positive,

                    "False Negative":
                        false_negative,

                    "Shots / Policy":
                        int(
                            final_shots
                        ),

                    "Total Probability Shots":
                        int(
                            2
                            * final_shots
                        ),

                    "Final Ratio Low":
                        final_ratio_low,

                    "Final Ratio High":
                        final_ratio_high,

                    "Stage Alpha":
                        STAGE_ALPHA,

                    "Tail Alpha":
                        TAIL_ALPHA,
                }
            )


E_RESULTS = (
    pd.DataFrame(
        records
    )
)



# HARD VALIDATION


expected_rows = (
    len(cases)
    * REPEATS
    * len(METHODS_E)
)


if len(
    E_RESULTS
) != expected_rows:

    raise RuntimeError(
        f"Experiment-E row count mismatch: "
        f"{len(E_RESULTS)} != {expected_rows}"
    )


method_counts = (
    E_RESULTS[
        "Method"
    ]
    .value_counts()
)


if not all(
    method_counts.get(
        method,
        0,
    )
    == (
        len(cases)
        * REPEATS
    )

    for method in METHODS_E
):

    raise RuntimeError(
        "Experiment-E method-count mismatch."
    )



# SUMMARY


summary_rows = []


for method in METHODS_E:

    group = (
        E_RESULTS[
            E_RESULTS[
                "Method"
            ]
            == method
        ]
        .copy()
    )


    total = len(
        group
    )


    resolved = (
        group[
            group[
                "Resolved"
            ]
        ]
    )


    unresolved_count = (
        total
        - len(resolved)
    )


    correct_count = int(
        resolved[
            "Correct"
        ].fillna(
            False
        ).sum()
    )


    incorrect_count = (
        len(resolved)
        - correct_count
    )


    fp_count = int(
        resolved[
            "False Positive"
        ].sum()
    )


    fn_count = int(
        resolved[
            "False Negative"
        ].sum()
    )


    summary_rows.append(
        {

            "Method":
                method,

            "Total Trials":
                total,

            "Resolved Trials":
                len(resolved),

            "Unresolved Trials":
                unresolved_count,

            "Resolved Coverage":
                (
                    len(resolved)
                    / total
                ),

            "Unresolved Rate":
                (
                    unresolved_count
                    / total
                ),

            "Resolved Accuracy":
                (
                    correct_count
                    / len(resolved)

                    if len(resolved)
                    > 0

                    else np.nan
                ),

            "Incorrect Resolved Rate":
                (
                    incorrect_count
                    / total
                ),

            "False Positive Rate":
                (
                    fp_count
                    / total
                ),

            "False Negative Rate":
                (
                    fn_count
                    / total
                ),

            "Mean Shots / Policy":
                float(
                    group[
                        "Shots / Policy"
                    ].mean()
                ),

            "Mean Total Probability Shots":
                float(
                    group[
                        "Total Probability Shots"
                    ].mean()
                ),

            "Median Total Probability Shots":
                float(
                    group[
                        "Total Probability Shots"
                    ].median()
                ),

            "Maximum Total Probability Shots":
                int(
                    group[
                        "Total Probability Shots"
                    ].max()
                ),
        }
    )


E_SUMMARY = (
    pd.DataFrame(
        summary_rows
    )
)



# BOUNDARY-DISTANCE SUMMARY


def boundary_distance(
    row,
):

    boundary = (
        1.20
        if row[
            "Advantage"
        ]
        >= 0
        else 0.80
    )


    return abs(
        row[
            "True Ratio"
        ]
        - boundary
    )


E_RESULTS[
    "Boundary Distance"
] = (
    E_RESULTS.apply(
        boundary_distance,
        axis=1,
    )
)


E_DISTANCE_SUMMARY = (

    E_RESULTS

    .groupby(
        [
            "Method",
            "Boundary Distance",
        ],
        as_index=False,
    )

    .agg(

        Trials=(
            "Resolved",
            "size",
        ),

        Resolved_Coverage=(
            "Resolved",
            "mean",
        ),

        Mean_Total_Shots=(
            "Total Probability Shots",
            "mean",
        ),
    )
)



# SAVE RESULTS


E_RESULTS_PATH = (
    EXP_E_DIR
    / "experiment_E_all_trials.csv"
)


E_SUMMARY_PATH = (
    EXP_E_DIR
    / "experiment_E_summary.csv"
)


E_DISTANCE_PATH = (
    EXP_E_DIR
    / "experiment_E_boundary_distance.csv"
)


E_RESULTS.to_csv(
    E_RESULTS_PATH,
    index=False,
)


E_SUMMARY.to_csv(
    E_SUMMARY_PATH,
    index=False,
)


E_DISTANCE_SUMMARY.to_csv(
    E_DISTANCE_PATH,
    index=False,
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "experiment_E_confidence"
] = {

    "status":
        "COMPLETE",

    "experiment_E_hash":
        E_CONFIG_HASH,

    "methods":
        METHODS_E,

    "controlled_cases":
        len(cases),

    "repeats_per_case":
        REPEATS,

    "trial_paths":
        len(cases)
        * REPEATS,

    "delta":
        DELTA,

    "shot_schedule":
        SHOT_SCHEDULE,

    "result_file":
        str(
            E_RESULTS_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),

    "summary_file":
        str(
            E_SUMMARY_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),

    "distance_file":
        str(
            E_DISTANCE_PATH.relative_to(
                JOURNAL_ROOT
            )
        ),
}


manifest[
    "current_completed_cell"
] = 7


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)



# CELL-7 PROVENANCE


cell7_record = {

    "cell":
        7,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "experiment":
        "E",

    "experiment_E_hash":
        E_CONFIG_HASH,

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "controlled_cases":
        len(cases),

    "repeats_per_case":
        REPEATS,

    "methods":
        METHODS_E,

    "delta":
        DELTA,

    "stage_alpha":
        STAGE_ALPHA,

    "tail_alpha":
        TAIL_ALPHA,
}


(
    LOGS_DIR
    / "cell07_experiment_E.json"
).write_text(
    json.dumps(
        cell7_record,
        indent=2,
    ),
    encoding="utf-8",
)



# REPORT


print()
print("=" * 90)
print("EXPERIMENT E COMPLETE")
print("=" * 90)

print(
    "Controlled cases :",
    len(cases)
)

print(
    "Repeats/case     :",
    REPEATS
)

print(
    "Trial paths      :",
    len(cases)
    * REPEATS
)

print(
    "Methods          :",
    ", ".join(
        METHODS_E
    )
)

print(
    "Delta            :",
    DELTA
)

print(
    "Stages           :",
    len(
        SHOT_SCHEDULE
    )
)

print(
    "Stage alpha      :",
    STAGE_ALPHA
)

print(
    "Tail alpha       :",
    TAIL_ALPHA
)

print()
print(
    E_SUMMARY.to_string(
        index=False
    )
)

print()
print(
    "Important: resolved accuracy must always "
    "be reported together with unresolved rate "
    "or resolved coverage."
)

print("=" * 90)






CELL7_CHECKPOINT = (
    save_journal_checkpoint(
        7,
        "Experiment_E_Confidence_COMPLETE",
    )
)


print()
print("=" * 90)
print("CELL 7 COMPLETE")
print("=" * 90)

print(
    "Experiment E is now frozen."
)

print(
    "No further training experiments remain."
)

print(
    "Next: Cell 8 will perform the combined "
    "seed-level statistical analysis for A–D "
    "and paired benchmark analysis for E."
)

In [ ]:
# Combined statistical analysis

# A:
#   Main 3-environment benchmark
#   QTrust vs Classical / Fixed128 / Fixed256

# B:
#   Ablations vs full QTrust baseline

# C:
#   QTrust vs fixed-shot methods under noise
#   Noisy conditions vs clean Experiment-A reference

# D:
#   Depth 1 and depth 4 vs depth-3 Experiment-A reference

# E:
#   Paired confidence-bound benchmark analysis

# Inferential Unit:
#   training seed

# TESTS:
#   Wilcoxon signed-rank
#   paired rank-biserial effect size
#   paired bootstrap 95% CI of mean difference
#   Holm correction within predefined comparison families




from pathlib import Path
from datetime import datetime, timezone

import os
import re
import io
import json
import zipfile
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.stats import (
    t as student_t,
    wilcoxon,
    rankdata,
    binomtest,
)






required = [
    "JOURNAL_ROOT",
    "EXP_B_DIR",
    "EXP_C_DIR",
    "EXP_D_DIR",
    "EXP_E_DIR",
    "STATS_DIR",
    "LOGS_DIR",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "EXPECTED_SEEDS",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing journal state: "
        + ", ".join(missing)
    )


STATS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 92)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 8 / 10")
print("COMBINED SEED-LEVEL STATISTICAL ANALYSIS")
print("=" * 92)



# ANALYSIS CONSTANTS


SEEDS = sorted(
    int(x)
    for x in EXPECTED_SEEDS
)

N_SEEDS = len(
    SEEDS
)

BOOTSTRAP_REPS = 50_000
BOOTSTRAP_SEED = 20260920

MAIN_METHODS = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]

MAIN_ENVS = [
    "CartPole-v1",
    "Acrobot-v1",
    "LunarLander-v3",
]



# GENERIC COLUMN NORMALIZATION


def norm_name(
    value,
):

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).lower(),
    ).strip("_")


def canonical_method(
    value,
):

    text = norm_name(
        value
    )


    if (
        "classical" in text
        and "ppo" in text
    ):

        return "Classical-PPO"


    if (
        "qtrust" in text
        and "ppo" in text
    ):

        return "QTrust-PPO-128"


    if (
        "fixed" in text
        and "128" in text
    ):

        return "Fixed-QPPO-128"


    if (
        "fixed" in text
        and "256" in text
    ):

        return "Fixed-QPPO-256"


    return None


def canonical_env(
    value,
):

    text = norm_name(
        value
    )


    if "cartpole" in text:

        return "CartPole-v1"


    if "acrobot" in text:

        return "Acrobot-v1"


    if (
        "lunarlander" in text
        or "lunar_lander" in text
    ):

        return "LunarLander-v3"


    return None


COLUMN_ALIASES = {

    "environment": [
        "environment",
        "env",
        "env_name",
        "environment_name",
    ],

    "method": [
        "method",
        "algorithm",
        "model",
        "policy_method",
    ],

    "seed": [
        "seed",
        "training_seed",
        "train_seed",
        "random_seed",
    ],

    "reward": [
        "mean_eval_reward",
        "mean_evaluation_reward",
        "evaluation_mean_reward",
        "eval_mean_reward",
        "eval_reward_mean",
        "final_eval_reward",
        "final_mean_reward",
        "mean_reward",
        "evaluation_reward",
    ],
}


def detect_column(
    dataframe,
    aliases,
):

    mapping = {
        norm_name(col): col
        for col in dataframe.columns
    }


    for alias in aliases:

        if alias in mapping:

            return mapping[
                alias
            ]


    return None


def canonicalize_A_dataframe(
    dataframe,
):

    if dataframe is None:

        return None


    if len(
        dataframe
    ) == 0:

        return None


    env_col = detect_column(
        dataframe,
        COLUMN_ALIASES[
            "environment"
        ],
    )

    method_col = detect_column(
        dataframe,
        COLUMN_ALIASES[
            "method"
        ],
    )

    seed_col = detect_column(
        dataframe,
        COLUMN_ALIASES[
            "seed"
        ],
    )

    reward_col = detect_column(
        dataframe,
        COLUMN_ALIASES[
            "reward"
        ],
    )


    if any(
        col is None
        for col in [
            env_col,
            method_col,
            seed_col,
            reward_col,
        ]
    ):

        return None


    result = pd.DataFrame(
        {
            "environment":
                dataframe[
                    env_col
                ].map(
                    canonical_env
                ),

            "method":
                dataframe[
                    method_col
                ].map(
                    canonical_method
                ),

            "seed":
                pd.to_numeric(
                    dataframe[
                        seed_col
                    ],
                    errors="coerce",
                ),

            "mean_eval_reward":
                pd.to_numeric(
                    dataframe[
                        reward_col
                    ],
                    errors="coerce",
                ),
        }
    )


    result = (
        result
        .dropna()
        .copy()
    )


    result[
        "seed"
    ] = (
        result[
            "seed"
        ].astype(int)
    )


    result = (
        result[
            result[
                "environment"
            ].isin(
                MAIN_ENVS
            )
            &
            result[
                "method"
            ].isin(
                MAIN_METHODS
            )
            &
            result[
                "seed"
            ].isin(
                SEEDS
            )
        ]
        .copy()
    )


    result = (
        result
        .drop_duplicates(
            subset=[
                "environment",
                "method",
                "seed",
            ],
            keep="last",
        )
        .reset_index(
            drop=True
        )
    )


    return result


def valid_A_table(
    dataframe,
):

    if dataframe is None:

        return False


    if len(
        dataframe
    ) != 120:

        return False


    counts = (
        dataframe
        .groupby(
            [
                "environment",
                "method",
            ]
        )
        .size()
    )


    if len(
        counts
    ) != 12:

        return False


    if not counts.eq(
        10
    ).all():

        return False


    if set(
        dataframe[
            "environment"
        ].unique()
    ) != set(
        MAIN_ENVS
    ):

        return False


    if set(
        dataframe[
            "method"
        ].unique()
    ) != set(
        MAIN_METHODS
    ):

        return False


    return True



# ROBUSTLY LOCATE AUTHORITATIVE EXPERIMENT-A RESULTS

# No experiment is rerun.
# We only locate the completed 120-run result table.


A_RESULTS = None
A_SOURCE = None


search_roots = [

    JOURNAL_ROOT,

    Path(
        "/kaggle/input"
    ),

    Path(
        "/kaggle/working"
    ),
]



# 4A. Search ordinary CSV files


candidate_csvs = []


for root in search_roots:

    if not root.exists():

        continue


    try:

        candidate_csvs.extend(
            root.rglob(
                "*.csv"
            )
        )

    except Exception:

        pass


for path in candidate_csvs:

    # Avoid known B-E outputs.
    lower_path = str(
        path
    ).lower()


    if any(
        tag in lower_path
        for tag in [
            "experiment_b_",
            "experiment_c_",
            "experiment_d_",
            "experiment_e_",
        ]
    ):

        continue


    try:

        frame = pd.read_csv(
            path
        )

        candidate = (
            canonicalize_A_dataframe(
                frame
            )
        )


        if valid_A_table(
            candidate
        ):

            A_RESULTS = (
                candidate
            )

            A_SOURCE = str(
                path
            )

            break


    except Exception:

        continue



# 4B. Search CSVs inside ZIP archives if necessary


if A_RESULTS is None:

    candidate_zips = []


    for root in search_roots:

        if not root.exists():

            continue


        try:

            candidate_zips.extend(
                root.rglob(
                    "*.zip"
                )
            )

        except Exception:

            pass


    # Prefer likely Experiment-A artifact ZIPs first.
    candidate_zips = sorted(
        candidate_zips,
        key=lambda p: (
            0
            if (
                "final_paper_artifacts"
                in p.name.lower()
                or
                "qtrust_ppo_final"
                in p.name.lower()
            )
            else 1,
            len(
                str(p)
            ),
        ),
    )


    for zip_path in candidate_zips:

        try:

            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zf:

                csv_members = [
                    member
                    for member
                    in zf.namelist()
                    if member.lower().endswith(
                        ".csv"
                    )
                ]


                for member in csv_members:

                    lower_member = (
                        member.lower()
                    )


                    if any(
                        tag in lower_member
                        for tag in [
                            "experiment_b_",
                            "experiment_c_",
                            "experiment_d_",
                            "experiment_e_",
                        ]
                    ):

                        continue


                    try:

                        with zf.open(
                            member
                        ) as handle:

                            frame = pd.read_csv(
                                handle
                            )


                        candidate = (
                            canonicalize_A_dataframe(
                                frame
                            )
                        )


                        if valid_A_table(
                            candidate
                        ):

                            A_RESULTS = (
                                candidate
                            )

                            A_SOURCE = (
                                f"{zip_path}"
                                f" :: {member}"
                            )

                            break


                    except Exception:

                        continue


                if A_RESULTS is not None:

                    break


        except Exception:

            continue


if A_RESULTS is None:

    raise RuntimeError(
        "Could not locate the authoritative "
        "120-run Experiment-A result table. "
        "No analysis was performed and no data "
        "were modified."
    )


print()
print(
    "Experiment A source:"
)

print(
    A_SOURCE
)

print(
    "Experiment A runs  :",
    len(
        A_RESULTS
    ),
    "/ 120"
)



# HARD EXPERIMENT-A PROTOCOL VALIDATION


A_COUNTS = (
    A_RESULTS
    .groupby(
        [
            "environment",
            "method",
        ]
    )
    .size()
)


if not A_COUNTS.eq(
    10
).all():

    raise RuntimeError(
        "Experiment A does not have "
        "10 seeds per environment × method."
    )


for (
    env_name,
    method_name,
), group in (
    A_RESULTS.groupby(
        [
            "environment",
            "method",
        ]
    )
):

    observed_seeds = sorted(
        group[
            "seed"
        ]
        .astype(int)
        .tolist()
    )


    if observed_seeds != SEEDS:

        raise RuntimeError(
            "Experiment-A seed mismatch for "
            f"{env_name} / {method_name}"
        )


print(
    "Experiment A validation: PASS"
)






B_RESULTS_PATH = (
    EXP_B_DIR
    / "experiment_B_all_runs.csv"
)

C_RESULTS_PATH = (
    EXP_C_DIR
    / "experiment_C_all_runs.csv"
)

D_RESULTS_PATH = (
    EXP_D_DIR
    / "experiment_D_all_runs.csv"
)

E_RESULTS_PATH = (
    EXP_E_DIR
    / "experiment_E_all_trials.csv"
)

E_SUMMARY_PATH = (
    EXP_E_DIR
    / "experiment_E_summary.csv"
)


for required_path in [
    B_RESULTS_PATH,
    C_RESULTS_PATH,
    D_RESULTS_PATH,
    E_RESULTS_PATH,
    E_SUMMARY_PATH,
]:

    if not required_path.exists():

        raise RuntimeError(
            f"Missing frozen result file: "
            f"{required_path}"
        )


B_RESULTS = pd.read_csv(
    B_RESULTS_PATH
)

C_RESULTS = pd.read_csv(
    C_RESULTS_PATH
)

D_RESULTS = pd.read_csv(
    D_RESULTS_PATH
)

E_RESULTS = pd.read_csv(
    E_RESULTS_PATH
)

E_SUMMARY = pd.read_csv(
    E_SUMMARY_PATH
)


if len(
    B_RESULTS
) != 30:

    raise RuntimeError(
        "Experiment B != 30 runs."
    )


if len(
    C_RESULTS
) != 60:

    raise RuntimeError(
        "Experiment C != 60 runs."
    )


if len(
    D_RESULTS
) != 20:

    raise RuntimeError(
        "Experiment D != 20 runs."
    )


print(
    "Experiments B-E validation: PASS"
)



# STATISTICAL HELPER FUNCTIONS


def mean_t_ci(
    values,
    confidence=0.95,
):

    values = np.asarray(
        values,
        dtype=np.float64,
    )


    values = values[
        np.isfinite(
            values
        )
    ]


    n = len(
        values
    )


    if n == 0:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
        )


    mean = float(
        np.mean(
            values
        )
    )


    if n == 1:

        return (
            mean,
            np.nan,
            np.nan,
            np.nan,
        )


    sd = float(
        np.std(
            values,
            ddof=1,
        )
    )


    sem = (
        sd
        / np.sqrt(
            n
        )
    )


    alpha = (
        1.0
        - confidence
    )


    critical = float(
        student_t.ppf(
            1.0
            - alpha / 2.0,
            df=n - 1,
        )
    )


    lower = (
        mean
        - critical
        * sem
    )


    upper = (
        mean
        + critical
        * sem
    )


    return (
        mean,
        sd,
        float(lower),
        float(upper),
    )


def bootstrap_mean_ci(
    differences,
    reps=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):

    differences = np.asarray(
        differences,
        dtype=np.float64,
    )


    n = len(
        differences
    )


    if n == 0:

        return (
            np.nan,
            np.nan,
        )


    rng = (
        np.random.default_rng(
            seed
        )
    )


    indices = rng.integers(
        0,
        n,
        size=(
            reps,
            n,
        ),
    )


    bootstrap_means = (
        differences[
            indices
        ].mean(
            axis=1
        )
    )


    lower, upper = (
        np.quantile(
            bootstrap_means,
            [
                0.025,
                0.975,
            ],
        )
    )


    return (
        float(lower),
        float(upper),
    )


def paired_rank_biserial(
    differences,
):

    differences = np.asarray(
        differences,
        dtype=np.float64,
    )


    differences = (
        differences[
            differences != 0
        ]
    )


    if len(
        differences
    ) == 0:

        return 0.0


    ranks = rankdata(
        np.abs(
            differences
        ),
        method="average",
    )


    w_plus = float(
        ranks[
            differences > 0
        ].sum()
    )


    w_minus = float(
        ranks[
            differences < 0
        ].sum()
    )


    denominator = (
        w_plus
        + w_minus
    )


    if denominator == 0:

        return 0.0


    return float(
        (
            w_plus
            - w_minus
        )
        / denominator
    )


def safe_wilcoxon(
    differences,
):

    differences = np.asarray(
        differences,
        dtype=np.float64,
    )


    if np.allclose(
        differences,
        0.0,
    ):

        return (
            0.0,
            1.0,
        )


    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )


        result = wilcoxon(
            differences,
            zero_method="wilcox",
            alternative="two-sided",
            method="auto",
        )


    return (
        float(
            result.statistic
        ),

        float(
            result.pvalue
        ),
    )


def holm_adjust(
    pvalues,
):

    pvalues = np.asarray(
        pvalues,
        dtype=np.float64,
    )


    m = len(
        pvalues
    )


    if m == 0:

        return np.array(
            [],
            dtype=np.float64,
        )


    order = np.argsort(
        pvalues
    )


    sorted_p = (
        pvalues[
            order
        ]
    )


    adjusted_sorted = (
        np.empty(
            m,
            dtype=np.float64,
        )
    )


    running_max = 0.0


    for i, pvalue in enumerate(
        sorted_p
    ):

        adjusted = min(
            1.0,
            (
                m - i
            )
            * float(
                pvalue
            ),
        )


        running_max = max(
            running_max,
            adjusted,
        )


        adjusted_sorted[
            i
        ] = min(
            1.0,
            running_max,
        )


    adjusted = np.empty(
        m,
        dtype=np.float64,
    )


    adjusted[
        order
    ] = adjusted_sorted


    return adjusted


def paired_reward_analysis(
    target,
    reference,
    target_label,
    reference_label,
    family,
    seed_offset=0,
):

    left = (
        target[
            [
                "seed",
                "mean_eval_reward",
            ]
        ]
        .rename(
            columns={
                "mean_eval_reward":
                    "target_reward"
            }
        )
    )


    right = (
        reference[
            [
                "seed",
                "mean_eval_reward",
            ]
        ]
        .rename(
            columns={
                "mean_eval_reward":
                    "reference_reward"
            }
        )
    )


    merged = (
        left.merge(
            right,
            on="seed",
            how="inner",
            validate="one_to_one",
        )
        .sort_values(
            "seed"
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        merged
    ) != N_SEEDS:

        raise RuntimeError(
            f"Paired comparison "
            f"{target_label} vs {reference_label} "
            f"has {len(merged)} seeds; "
            f"expected {N_SEEDS}."
        )


    if sorted(
        merged[
            "seed"
        ].astype(int)
    ) != SEEDS:

        raise RuntimeError(
            "Seed mismatch in paired analysis: "
            f"{target_label} vs {reference_label}"
        )


    differences = (
        merged[
            "target_reward"
        ].to_numpy(
            dtype=np.float64
        )
        -
        merged[
            "reference_reward"
        ].to_numpy(
            dtype=np.float64
        )
    )


    statistic, pvalue = (
        safe_wilcoxon(
            differences
        )
    )


    ci_low, ci_high = (
        bootstrap_mean_ci(
            differences,
            seed=(
                BOOTSTRAP_SEED
                + seed_offset
            ),
        )
    )


    return {

        "Family":
            family,

        "Target":
            target_label,

        "Reference":
            reference_label,

        "N":
            len(
                differences
            ),

        "Target Mean Reward":
            float(
                merged[
                    "target_reward"
                ].mean()
            ),

        "Reference Mean Reward":
            float(
                merged[
                    "reference_reward"
                ].mean()
            ),

        "Mean Difference":
            float(
                np.mean(
                    differences
                )
            ),

        "Median Difference":
            float(
                np.median(
                    differences
                )
            ),

        "Bootstrap CI Low":
            ci_low,

        "Bootstrap CI High":
            ci_high,

        "Wilcoxon Statistic":
            statistic,

        "Raw P":
            pvalue,

        "Paired Rank-Biserial":
            paired_rank_biserial(
                differences
            ),
    }


def descriptive_seed_summary(
    dataframe,
    group_columns,
):

    rows = []


    for keys, group in (
        dataframe.groupby(
            group_columns,
            dropna=False,
        )
    ):

        if not isinstance(
            keys,
            tuple,
        ):

            keys = (
                keys,
            )


        mean, sd, low, high = (
            mean_t_ci(
                group[
                    "mean_eval_reward"
                ]
            )
        )


        row = {
            col: value
            for col, value
            in zip(
                group_columns,
                keys,
            )
        }


        row.update(
            {

                "N Seeds":
                    int(
                        len(
                            group
                        )
                    ),

                "Mean Reward":
                    mean,

                "SD Reward":
                    sd,

                "95% t-CI Low":
                    low,

                "95% t-CI High":
                    high,
            }
        )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )



# EXPERIMENT A DESCRIPTIVE STATISTICS


A_SEED_SUMMARY = (
    descriptive_seed_summary(

        A_RESULTS,

        [
            "environment",
            "method",
        ],
    )
)


A_SEED_SUMMARY.to_csv(
    STATS_DIR
    / "A_seed_level_summary.csv",
    index=False,
)



# EXPERIMENT A PAIRED INFERENCE

# QTrust minus comparator.
# Positive difference = higher QTrust reward.

# Holm correction separately within each environment.


A_TEST_ROWS = []

comparison_index = 0


for env_name in MAIN_ENVS:

    qtrust = (
        A_RESULTS[
            (
                A_RESULTS[
                    "environment"
                ]
                == env_name
            )
            &
            (
                A_RESULTS[
                    "method"
                ]
                == "QTrust-PPO-128"
            )
        ]
        .copy()
    )


    for comparator in [
        "Classical-PPO",
        "Fixed-QPPO-128",
        "Fixed-QPPO-256",
    ]:

        reference = (
            A_RESULTS[
                (
                    A_RESULTS[
                        "environment"
                    ]
                    == env_name
                )
                &
                (
                    A_RESULTS[
                        "method"
                    ]
                    == comparator
                )
            ]
            .copy()
        )


        result = (
            paired_reward_analysis(

                target=
                    qtrust,

                reference=
                    reference,

                target_label=
                    "QTrust-PPO-128",

                reference_label=
                    comparator,

                family=
                    env_name,

                seed_offset=
                    comparison_index,
            )
        )


        result[
            "Environment"
        ] = env_name


        A_TEST_ROWS.append(
            result
        )


        comparison_index += 1


A_INFERENCE = pd.DataFrame(
    A_TEST_ROWS
)


A_INFERENCE[
    "Holm P"
] = np.nan


for env_name in MAIN_ENVS:

    mask = (
        A_INFERENCE[
            "Environment"
        ]
        == env_name
    )


    A_INFERENCE.loc[
        mask,
        "Holm P",
    ] = (
        holm_adjust(
            A_INFERENCE.loc[
                mask,
                "Raw P",
            ].to_numpy()
        )
    )


A_INFERENCE[
    "Holm Significant 0.05"
] = (
    A_INFERENCE[
        "Holm P"
    ]
    < 0.05
)


A_INFERENCE.to_csv(
    STATS_DIR
    / "A_paired_inference.csv",
    index=False,
)



# EXPERIMENT B

# Full QTrust reference comes from Experiment A,
# CartPole-v1, QTrust-PPO-128, delta=.05,
# rollback=True.


B_SEED_SUMMARY = (
    descriptive_seed_summary(

        B_RESULTS,

        [
            "variant",
        ],
    )
)


B_SEED_SUMMARY.to_csv(
    STATS_DIR
    / "B_seed_level_summary.csv",
    index=False,
)


A_CARTPOLE_QTRUST = (
    A_RESULTS[
        (
            A_RESULTS[
                "environment"
            ]
            == "CartPole-v1"
        )
        &
        (
            A_RESULTS[
                "method"
            ]
            == "QTrust-PPO-128"
        )
    ]
    .copy()
)


B_TEST_ROWS = []


for index, variant in enumerate(
    [
        "B_Delta_001",
        "B_Delta_010",
        "B_NoRollback",
    ]
):

    target = (
        B_RESULTS[
            B_RESULTS[
                "variant"
            ]
            == variant
        ]
        .copy()
    )


    B_TEST_ROWS.append(

        paired_reward_analysis(

            target=
                target,

            reference=
                A_CARTPOLE_QTRUST,

            target_label=
                variant,

            reference_label=
                "A_Full_QTrust_delta_005",

            family=
                "Experiment_B_Ablations",

            seed_offset=
                100
                + index,
        )
    )


B_INFERENCE = pd.DataFrame(
    B_TEST_ROWS
)


B_INFERENCE[
    "Holm P"
] = (
    holm_adjust(
        B_INFERENCE[
            "Raw P"
        ].to_numpy()
    )
)


B_INFERENCE[
    "Holm Significant 0.05"
] = (
    B_INFERENCE[
        "Holm P"
    ]
    < 0.05
)


B_INFERENCE.to_csv(
    STATS_DIR
    / "B_ablation_paired_inference.csv",
    index=False,
)



# EXPERIMENT C DESCRIPTIVE


C_SEED_SUMMARY = (
    descriptive_seed_summary(

        C_RESULTS,

        [
            "variant",
            "method",
        ],
    )
)


C_SEED_SUMMARY.to_csv(
    STATS_DIR
    / "C_seed_level_summary.csv",
    index=False,
)



# EXPERIMENT C:
# QTrust Vs Fixed Under Each Noise Condition


C_WITHIN_ROWS = []


noise_variants = [
    "C_Readout_002",
    "C_Depolarizing_0005",
]


comparison_index = 200


for variant in noise_variants:

    qtrust = (
        C_RESULTS[
            (
                C_RESULTS[
                    "variant"
                ]
                == variant
            )
            &
            (
                C_RESULTS[
                    "method"
                ]
                == "QTrust-PPO-128"
            )
        ]
        .copy()
    )


    for comparator in [
        "Fixed-QPPO-128",
        "Fixed-QPPO-256",
    ]:

        reference = (
            C_RESULTS[
                (
                    C_RESULTS[
                        "variant"
                    ]
                    == variant
                )
                &
                (
                    C_RESULTS[
                        "method"
                    ]
                    == comparator
                )
            ]
            .copy()
        )


        result = (
            paired_reward_analysis(

                target=
                    qtrust,

                reference=
                    reference,

                target_label=
                    "QTrust-PPO-128",

                reference_label=
                    comparator,

                family=
                    variant,

                seed_offset=
                    comparison_index,
            )
        )


        result[
            "Noise Variant"
        ] = variant


        C_WITHIN_ROWS.append(
            result
        )


        comparison_index += 1


C_WITHIN_INFERENCE = (
    pd.DataFrame(
        C_WITHIN_ROWS
    )
)


C_WITHIN_INFERENCE[
    "Holm P"
] = np.nan


for variant in noise_variants:

    mask = (
        C_WITHIN_INFERENCE[
            "Noise Variant"
        ]
        == variant
    )


    C_WITHIN_INFERENCE.loc[
        mask,
        "Holm P",
    ] = (
        holm_adjust(
            C_WITHIN_INFERENCE.loc[
                mask,
                "Raw P",
            ].to_numpy()
        )
    )


C_WITHIN_INFERENCE[
    "Holm Significant 0.05"
] = (
    C_WITHIN_INFERENCE[
        "Holm P"
    ]
    < 0.05
)


C_WITHIN_INFERENCE.to_csv(
    STATS_DIR
    / "C_qtrust_vs_fixed_paired_inference.csv",
    index=False,
)



# EXPERIMENT C:
# Noisy Condition Vs Clean Experiment-A Reference

# Difference = noisy reward - clean reward.


C_CLEAN_ROWS = []


comparison_index = 300


for variant in noise_variants:

    for method in [
        "Fixed-QPPO-128",
        "Fixed-QPPO-256",
        "QTrust-PPO-128",
    ]:

        noisy = (
            C_RESULTS[
                (
                    C_RESULTS[
                        "variant"
                    ]
                    == variant
                )
                &
                (
                    C_RESULTS[
                        "method"
                    ]
                    == method
                )
            ]
            .copy()
        )


        clean = (
            A_RESULTS[
                (
                    A_RESULTS[
                        "environment"
                    ]
                    == "CartPole-v1"
                )
                &
                (
                    A_RESULTS[
                        "method"
                    ]
                    == method
                )
            ]
            .copy()
        )


        result = (
            paired_reward_analysis(

                target=
                    noisy,

                reference=
                    clean,

                target_label=
                    f"{variant}:{method}",

                reference_label=
                    f"Clean:{method}",

                family=
                    variant,

                seed_offset=
                    comparison_index,
            )
        )


        result[
            "Noise Variant"
        ] = variant


        result[
            "Method"
        ] = method


        C_CLEAN_ROWS.append(
            result
        )


        comparison_index += 1


C_NOISE_VS_CLEAN = (
    pd.DataFrame(
        C_CLEAN_ROWS
    )
)


C_NOISE_VS_CLEAN[
    "Holm P"
] = np.nan


for variant in noise_variants:

    mask = (
        C_NOISE_VS_CLEAN[
            "Noise Variant"
        ]
        == variant
    )


    C_NOISE_VS_CLEAN.loc[
        mask,
        "Holm P",
    ] = (
        holm_adjust(
            C_NOISE_VS_CLEAN.loc[
                mask,
                "Raw P",
            ].to_numpy()
        )
    )


C_NOISE_VS_CLEAN[
    "Holm Significant 0.05"
] = (
    C_NOISE_VS_CLEAN[
        "Holm P"
    ]
    < 0.05
)


C_NOISE_VS_CLEAN.to_csv(
    STATS_DIR
    / "C_noise_vs_clean_paired_inference.csv",
    index=False,
)



# EXPERIMENT D DESCRIPTIVE


D_SEED_SUMMARY = (
    descriptive_seed_summary(

        D_RESULTS,

        [
            "variant",
            "layers",
            "quantum_parameters",
        ],
    )
)


# Add Experiment-A depth-3 reference descriptively.
depth3_mean, depth3_sd, depth3_low, depth3_high = (
    mean_t_ci(
        A_CARTPOLE_QTRUST[
            "mean_eval_reward"
        ]
    )
)


D_REFERENCE_ROW = pd.DataFrame(
    [
        {
            "variant":
                "A_Layers_3_Reference",

            "layers":
                3,

            "quantum_parameters":
                24,

            "N Seeds":
                10,

            "Mean Reward":
                depth3_mean,

            "SD Reward":
                depth3_sd,

            "95% t-CI Low":
                depth3_low,

            "95% t-CI High":
                depth3_high,
        }
    ]
)


D_SEED_SUMMARY_WITH_REFERENCE = (
    pd.concat(
        [
            D_SEED_SUMMARY,
            D_REFERENCE_ROW,
        ],
        ignore_index=True,
    )
    .sort_values(
        "layers"
    )
    .reset_index(
        drop=True
    )
)


D_SEED_SUMMARY_WITH_REFERENCE.to_csv(
    STATS_DIR
    / "D_seed_level_summary_with_depth3_reference.csv",
    index=False,
)



# EXPERIMENT D PAIRED DEPTH INFERENCE

# Difference = new depth - depth 3.


D_TEST_ROWS = []


for index, variant in enumerate(
    [
        "D_Layers_1",
        "D_Layers_4",
    ]
):

    target = (
        D_RESULTS[
            D_RESULTS[
                "variant"
            ]
            == variant
        ]
        .copy()
    )


    result = (
        paired_reward_analysis(

            target=
                target,

            reference=
                A_CARTPOLE_QTRUST,

            target_label=
                variant,

            reference_label=
                "A_Layers_3_Reference",

            family=
                "Experiment_D_Depth",

            seed_offset=
                400
                + index,
        )
    )


    result[
        "Layers"
    ] = int(
        target[
            "layers"
        ].iloc[
            0
        ]
    )


    result[
        "Quantum Parameters"
    ] = int(
        target[
            "quantum_parameters"
        ].iloc[
            0
        ]
    )


    D_TEST_ROWS.append(
        result
    )


D_INFERENCE = pd.DataFrame(
    D_TEST_ROWS
)


D_INFERENCE[
    "Holm P"
] = (
    holm_adjust(
        D_INFERENCE[
            "Raw P"
        ].to_numpy()
    )
)


D_INFERENCE[
    "Holm Significant 0.05"
] = (
    D_INFERENCE[
        "Holm P"
    ]
    < 0.05
)


D_INFERENCE.to_csv(
    STATS_DIR
    / "D_depth_paired_inference.csv",
    index=False,
)



# EXPERIMENT E PAIRED ANALYSIS

# All methods were applied to identical Case × Repeat
# finite-shot sample paths.

# Compare Wilson and Hoeffding against CP.


E_KEYS = [
    "Case",
    "Repeat",
]


CP_E = (
    E_RESULTS[
        E_RESULTS[
            "Method"
        ]
        == "Clopper-Pearson"
    ][
        E_KEYS
        + [
            "Resolved",
            "Correct",
            "Total Probability Shots",
        ]
    ]
    .rename(
        columns={
            "Resolved":
                "CP_Resolved",

            "Correct":
                "CP_Correct",

            "Total Probability Shots":
                "CP_Total_Shots",
        }
    )
)


E_PAIRED_ROWS = []


for index, method in enumerate(
    [
        "Wilson",
        "Hoeffding",
    ]
):

    other = (
        E_RESULTS[
            E_RESULTS[
                "Method"
            ]
            == method
        ][
            E_KEYS
            + [
                "Resolved",
                "Correct",
                "Total Probability Shots",
            ]
        ]
        .rename(
            columns={
                "Resolved":
                    "Other_Resolved",

                "Correct":
                    "Other_Correct",

                "Total Probability Shots":
                    "Other_Total_Shots",
            }
        )
    )


    merged = (
        CP_E.merge(
            other,
            on=E_KEYS,
            how="inner",
            validate="one_to_one",
        )
    )


    if len(
        merged
    ) != 22_800:

        raise RuntimeError(
            f"Experiment-E paired path mismatch "
            f"for {method}: {len(merged)}"
        )


    cp_resolved = (
        merged[
            "CP_Resolved"
        ].astype(bool)
        .to_numpy()
    )


    other_resolved = (
        merged[
            "Other_Resolved"
        ].astype(bool)
        .to_numpy()
    )


    coverage_difference = (
        other_resolved.astype(
            np.int8
        )
        -
        cp_resolved.astype(
            np.int8
        )
    )


    # Discordant paths for exact McNemar test.
    other_only = int(
        np.sum(
            other_resolved
            &
            ~cp_resolved
        )
    )


    cp_only = int(
        np.sum(
            cp_resolved
            &
            ~other_resolved
        )
    )


    discordant = (
        other_only
        + cp_only
    )


    if discordant == 0:

        mcnemar_p = 1.0


    else:

        mcnemar_p = float(
            binomtest(
                other_only,
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )


    shot_difference = (
        merged[
            "Other_Total_Shots"
        ].to_numpy(
            dtype=np.float64
        )
        -
        merged[
            "CP_Total_Shots"
        ].to_numpy(
            dtype=np.float64
        )
    )


    shot_statistic, shot_p = (
        safe_wilcoxon(
            shot_difference
        )
    )


    coverage_ci_low, coverage_ci_high = (
        bootstrap_mean_ci(
            coverage_difference,
            seed=(
                BOOTSTRAP_SEED
                + 500
                + index
            ),
        )
    )


    shot_ci_low, shot_ci_high = (
        bootstrap_mean_ci(
            shot_difference,
            seed=(
                BOOTSTRAP_SEED
                + 600
                + index
            ),
        )
    )


    cp_resolved_accuracy = (

        float(
            merged.loc[
                cp_resolved,
                "CP_Correct",
            ]
            .astype(float)
            .mean()
        )

        if np.any(
            cp_resolved
        )

        else np.nan
    )


    other_resolved_accuracy = (

        float(
            merged.loc[
                other_resolved,
                "Other_Correct",
            ]
            .astype(float)
            .mean()
        )

        if np.any(
            other_resolved
        )

        else np.nan
    )


    E_PAIRED_ROWS.append(
        {

            "Method":
                method,

            "Reference":
                "Clopper-Pearson",

            "Paired Trial Paths":
                len(
                    merged
                ),

            "CP Coverage":
                float(
                    np.mean(
                        cp_resolved
                    )
                ),

            "Method Coverage":
                float(
                    np.mean(
                        other_resolved
                    )
                ),

            "Coverage Difference":
                float(
                    np.mean(
                        coverage_difference
                    )
                ),

            "Coverage Bootstrap CI Low":
                coverage_ci_low,

            "Coverage Bootstrap CI High":
                coverage_ci_high,

            "Method Only Resolved":
                other_only,

            "CP Only Resolved":
                cp_only,

            "Coverage McNemar Raw P":
                mcnemar_p,

            "CP Resolved Accuracy":
                cp_resolved_accuracy,

            "Method Resolved Accuracy":
                other_resolved_accuracy,

            "CP Mean Total Shots":
                float(
                    merged[
                        "CP_Total_Shots"
                    ].mean()
                ),

            "Method Mean Total Shots":
                float(
                    merged[
                        "Other_Total_Shots"
                    ].mean()
                ),

            "Mean Shot Difference":
                float(
                    np.mean(
                        shot_difference
                    )
                ),

            "Shot Bootstrap CI Low":
                shot_ci_low,

            "Shot Bootstrap CI High":
                shot_ci_high,

            "Shot Wilcoxon Statistic":
                shot_statistic,

            "Shot Wilcoxon Raw P":
                shot_p,

            "Shot Paired Rank-Biserial":
                paired_rank_biserial(
                    shot_difference
                ),
        }
    )


E_PAIRED = pd.DataFrame(
    E_PAIRED_ROWS
)


E_PAIRED[
    "Coverage Holm P"
] = (
    holm_adjust(
        E_PAIRED[
            "Coverage McNemar Raw P"
        ].to_numpy()
    )
)


E_PAIRED[
    "Shot Holm P"
] = (
    holm_adjust(
        E_PAIRED[
            "Shot Wilcoxon Raw P"
        ].to_numpy()
    )
)


E_PAIRED[
    "Coverage Holm Significant 0.05"
] = (
    E_PAIRED[
        "Coverage Holm P"
    ]
    < 0.05
)


E_PAIRED[
    "Shot Holm Significant 0.05"
] = (
    E_PAIRED[
        "Shot Holm P"
    ]
    < 0.05
)


E_PAIRED.to_csv(
    STATS_DIR
    / "E_paired_confidence_method_analysis.csv",
    index=False,
)



# RESOURCE / RELIABILITY SUMMARIES FOR B-D

# Descriptive only.


B_DIAGNOSTICS = (
    B_RESULTS
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),
    )
)


C_DIAGNOSTICS = (
    C_RESULTS
    .groupby(
        [
            "variant",
            "method",
        ],
        as_index=False,
    )
    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),
    )
)


D_DIAGNOSTICS = (
    D_RESULTS
    .groupby(
        [
            "variant",
            "layers",
            "quantum_parameters",
        ],
        as_index=False,
    )
    .agg(

        Seeds=(
            "seed",
            "count",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),

        Mean_Runtime_s=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


B_DIAGNOSTICS.to_csv(
    STATS_DIR
    / "B_diagnostics_summary.csv",
    index=False,
)


C_DIAGNOSTICS.to_csv(
    STATS_DIR
    / "C_diagnostics_summary.csv",
    index=False,
)


D_DIAGNOSTICS.to_csv(
    STATS_DIR
    / "D_diagnostics_summary.csv",
    index=False,
)






STATISTICAL_PROTOCOL = {

    "cell":
        8,

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "inferential_unit":
        "training seed",

    "n_training_seeds":
        N_SEEDS,

    "seed_values":
        SEEDS,

    "confidence_interval":
        "95% Student-t CI across training seeds",

    "paired_reward_test":
        "two-sided Wilcoxon signed-rank",

    "effect_size":
        "paired rank-biserial correlation",

    "paired_difference_ci":
        (
            "paired nonparametric bootstrap "
            f"mean-difference CI, {BOOTSTRAP_REPS} resamples"
        ),

    "multiplicity":
        (
            "Holm correction within each "
            "predefined comparison family"
        ),

    "experiment_A_source":
        A_SOURCE,

    "experiment_E_paired_design":
        (
            "confidence methods applied to identical "
            "Case × Repeat sample paths"
        ),
}


STATISTICAL_PROTOCOL_HASH = (
    hashlib.sha256(
        json.dumps(
            STATISTICAL_PROTOCOL,
            sort_keys=True,
            separators=(",", ":"),
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()[:16]
)


STATISTICAL_PROTOCOL[
    "statistical_protocol_hash"
] = (
    STATISTICAL_PROTOCOL_HASH
)


(
    STATS_DIR
    / "statistical_analysis_protocol.json"
).write_text(
    json.dumps(
        STATISTICAL_PROTOCOL,
        indent=2,
    ),
    encoding="utf-8",
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "statistical_analysis"
] = {

    "status":
        "COMPLETE",

    "cell":
        8,

    "statistical_protocol_hash":
        STATISTICAL_PROTOCOL_HASH,

    "experiment_A_source":
        A_SOURCE,

    "bootstrap_repetitions":
        BOOTSTRAP_REPS,

    "multiplicity":
        "Holm within predefined families",

    "main_files": [

        "statistics/A_seed_level_summary.csv",

        "statistics/A_paired_inference.csv",

        "statistics/B_ablation_paired_inference.csv",

        "statistics/C_qtrust_vs_fixed_paired_inference.csv",

        "statistics/C_noise_vs_clean_paired_inference.csv",

        "statistics/D_depth_paired_inference.csv",

        "statistics/E_paired_confidence_method_analysis.csv",
    ],
}


manifest[
    "current_completed_cell"
] = 8


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)



# CELL-8 PROVENANCE


cell8_record = {

    "cell":
        8,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "statistical_protocol_hash":
        STATISTICAL_PROTOCOL_HASH,

    "A_runs":
        len(
            A_RESULTS
        ),

    "B_runs":
        len(
            B_RESULTS
        ),

    "C_runs":
        len(
            C_RESULTS
        ),

    "D_runs":
        len(
            D_RESULTS
        ),

    "E_trial_rows":
        len(
            E_RESULTS
        ),
}


(
    LOGS_DIR
    / "cell08_statistical_analysis.json"
).write_text(
    json.dumps(
        cell8_record,
        indent=2,
    ),
    encoding="utf-8",
)



# REPORT


print()
print("=" * 92)
print("EXPERIMENT A — MAIN PAIRED INFERENCE")
print("=" * 92)

print(
    A_INFERENCE[
        [
            "Environment",
            "Reference",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
            "Paired Rank-Biserial",
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 92)
print("EXPERIMENT B — ABLATION INFERENCE")
print("=" * 92)

print(
    B_INFERENCE[
        [
            "Target",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
            "Paired Rank-Biserial",
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 92)
print("EXPERIMENT C — QTRUST VS FIXED UNDER NOISE")
print("=" * 92)

print(
    C_WITHIN_INFERENCE[
        [
            "Noise Variant",
            "Reference",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
            "Paired Rank-Biserial",
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 92)
print("EXPERIMENT C — NOISE VS CLEAN REFERENCE")
print("=" * 92)

print(
    C_NOISE_VS_CLEAN[
        [
            "Noise Variant",
            "Method",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 92)
print("EXPERIMENT D — DEPTH INFERENCE")
print("=" * 92)

print(
    D_INFERENCE[
        [
            "Target",
            "Layers",
            "Quantum Parameters",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
            "Paired Rank-Biserial",
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 92)
print("EXPERIMENT E — PAIRED CONFIDENCE-METHOD ANALYSIS")
print("=" * 92)

print(
    E_PAIRED[
        [
            "Method",
            "Coverage Difference",
            "Coverage Bootstrap CI Low",
            "Coverage Bootstrap CI High",
            "Coverage McNemar Raw P",
            "Coverage Holm P",
            "Mean Shot Difference",
            "Shot Bootstrap CI Low",
            "Shot Bootstrap CI High",
            "Shot Wilcoxon Raw P",
            "Shot Holm P",
        ]
    ].to_string(
        index=False
    )
)


print()
print(
    "Statistical protocol hash:",
    STATISTICAL_PROTOCOL_HASH
)

print()
print(
    "STATISTICAL ANALYSIS: PASS"
)

print("=" * 92)






CELL8_CHECKPOINT = (
    save_journal_checkpoint(
        8,
        "Statistical_Analysis_COMPLETE",
    )
)


print()
print("=" * 92)
print("CELL 8 COMPLETE")
print("=" * 92)

print(
    "All A-E statistical analyses are frozen."
)

print(
    "Next: Cell 9 will generate the final "
    "journal-quality figures and publication tables."
)

In [ ]:
# INPUT:



# OUTPUT:
#   Vector PDF figures
#   600-DPI PNG figures

#   LaTeX tables
#   Figure/table manifests


# No New Statistical Tests


from pathlib import Path
from datetime import datetime, timezone

import os
import json
import shutil
import hashlib
import warnings

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

from scipy.stats import t as student_t



# REQUIRE CELLS 1-8 / RESUMED STATE


required = [
    "JOURNAL_ROOT",
    "STATS_DIR",
    "FIGURES_DIR",
    "TABLES_DIR",
    "LOGS_DIR",
    "EXP_B_DIR",
    "EXP_C_DIR",
    "EXP_D_DIR",
    "EXP_E_DIR",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing journal state: "
        + ", ".join(missing)
    )


print("=" * 94)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 9 / 10")
print("FINAL JOURNAL FIGURES + PUBLICATION TABLES")
print("=" * 94)


FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



# REQUIRE CELL-8 DATA


required_cell8_objects = [
    "A_RESULTS",
    "A_SEED_SUMMARY",
    "A_INFERENCE",
    "B_RESULTS",
    "B_INFERENCE",
    "C_RESULTS",
    "C_WITHIN_INFERENCE",
    "C_NOISE_VS_CLEAN",
    "D_RESULTS",
    "D_INFERENCE",
    "E_RESULTS",
    "E_SUMMARY",
    "E_PAIRED",
]

missing_cell8 = [
    name
    for name in required_cell8_objects
    if name not in globals()
]

if missing_cell8:
    raise RuntimeError(
        "Cell 8 must be run in this session before Cell 9. "
        "Missing: "
        + ", ".join(missing_cell8)
    )






plt.rcParams.update(
    {
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "figure.titlesize": 12,
        "lines.linewidth": 1.5,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.bbox": "tight",
    }
)



# HELPERS


def t_ci(
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]


    n = len(values)

    mean = float(
        np.mean(values)
    )


    if n <= 1:

        return (
            mean,
            np.nan,
            np.nan,
        )


    sd = float(
        np.std(
            values,
            ddof=1,
        )
    )


    critical = float(
        student_t.ppf(
            0.975,
            df=n - 1,
        )
    )


    half = (
        critical
        * sd
        / np.sqrt(n)
    )


    return (
        mean,
        mean - half,
        mean + half,
    )


def save_figure(
    fig,
    stem,
):

    pdf_path = (
        FIGURES_DIR
        / f"{stem}.pdf"
    )


    png_path = (
        FIGURES_DIR
        / f"{stem}.png"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
    )


    plt.close(fig)


    return (
        pdf_path,
        png_path,
    )


def seed_offsets(
    n,
    width=0.16,
):

    if n <= 1:

        return np.array(
            [0.0]
        )


    return np.linspace(
        -width,
        width,
        n,
    )


def mean_ci_plot(
    ax,
    values,
    x,
):

    mean, low, high = (
        t_ci(values)
    )


    ax.errorbar(
        x,
        mean,
        yerr=[
            [mean - low],
            [high - mean],
        ],
        fmt="o",
        capsize=4,
        markersize=6,
        linewidth=1.5,
    )


def safe_to_latex(
    dataframe,
    path,
    caption,
    label,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )


        latex = dataframe.to_latex(
            index=False,
            escape=True,
            caption=caption,
            label=label,
            float_format=lambda x: (
                f"{x:.4f}"
                if isinstance(
                    x,
                    (float, np.floating),
                )
                else str(x)
            ),
        )


    path.write_text(
        latex,
        encoding="utf-8",
    )



# LABELS


METHOD_LABELS = {

    "Classical-PPO":
        "Classical",

    "Fixed-QPPO-128":
        "Fixed-128",

    "Fixed-QPPO-256":
        "Fixed-256",

    "QTrust-PPO-128":
        "QTrust",
}


ENV_LABELS = {

    "CartPole-v1":
        "CartPole",

    "Acrobot-v1":
        "Acrobot",

    "LunarLander-v3":
        "LunarLander",
}



# FIGURE 1
# Primary benchmark performance

# All 10 training seeds visible.
# Mean ± 95% t-CI over training seeds.


method_order = [
    "Classical-PPO",
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        11.4,
        3.8,
    ),
)


for ax, env_name in zip(
    axes,
    [
        "CartPole-v1",
        "Acrobot-v1",
        "LunarLander-v3",
    ],
):

    env_df = (
        A_RESULTS[
            A_RESULTS[
                "environment"
            ]
            == env_name
        ]
        .copy()
    )


    for x, method in enumerate(
        method_order
    ):

        values = (
            env_df[
                env_df[
                    "method"
                ]
                == method
            ]
            .sort_values(
                "seed"
            )[
                "mean_eval_reward"
            ]
            .to_numpy(
                dtype=float
            )
        )


        offsets = (
            seed_offsets(
                len(values)
            )
        )


        ax.scatter(
            x + offsets,
            values,
            s=18,
            alpha=0.70,
        )


        mean_ci_plot(
            ax,
            values,
            x,
        )


    ax.set_xticks(
        range(
            len(method_order)
        )
    )


    ax.set_xticklabels(
        [
            METHOD_LABELS[m]
            for m in method_order
        ],
        rotation=25,
        ha="right",
    )


    ax.set_title(
        ENV_LABELS[
            env_name
        ]
    )


    ax.grid(
        axis="y",
        alpha=0.25,
    )


axes[0].set_ylabel(
    "Final evaluation reward"
)


fig.suptitle(
    "Main benchmark across ten training seeds"
)


fig.tight_layout()


save_figure(
    fig,
    "Fig_J1_Main_Performance",
)



# FIGURE 2
# Experiment-B Ablation

# Baseline A QTrust plus three B variants.


baseline_B = (
    A_RESULTS[
        (
            A_RESULTS[
                "environment"
            ]
            == "CartPole-v1"
        )
        &
        (
            A_RESULTS[
                "method"
            ]
            == "QTrust-PPO-128"
        )
    ][
        [
            "seed",
            "mean_eval_reward",
        ]
    ]
    .copy()
)


baseline_B[
    "variant"
] = "Full QTrust"


B_plot = pd.concat(
    [
        baseline_B[
            [
                "seed",
                "mean_eval_reward",
                "variant",
            ]
        ],

        B_RESULTS[
            [
                "seed",
                "mean_eval_reward",
                "variant",
            ]
        ],
    ],
    ignore_index=True,
)


B_order = [
    "Full QTrust",
    "B_Delta_001",
    "B_Delta_010",
    "B_NoRollback",
]


B_labels = [
    "Full\nδ=.05",
    "δ=.01",
    "δ=.10",
    "No\nrollback",
]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        8.0,
        3.8,
    ),
)


for x, variant in enumerate(
    B_order
):

    values = (
        B_plot[
            B_plot[
                "variant"
            ]
            == variant
        ]
        .sort_values(
            "seed"
        )[
            "mean_eval_reward"
        ]
        .to_numpy(
            dtype=float
        )
    )


    axes[0].scatter(
        x
        + seed_offsets(
            len(values)
        ),
        values,
        s=18,
        alpha=0.7,
    )


    mean_ci_plot(
        axes[0],
        values,
        x,
    )


axes[0].set_xticks(
    range(
        len(B_order)
    )
)

axes[0].set_xticklabels(
    B_labels
)

axes[0].set_ylabel(
    "Final evaluation reward"
)

axes[0].set_title(
    "Reward"
)

axes[0].grid(
    axis="y",
    alpha=0.25,
)


B_resource = (
    B_RESULTS
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(
        MeanTrainingShots=(
            "training_quantum_shots",
            "mean",
        ),
        Rollbacks=(
            "rolled_back_updates",
            "sum",
        ),
    )
)


resource_values = []


for variant in [
    "B_Delta_001",
    "B_Delta_010",
    "B_NoRollback",
]:

    row = (
        B_resource[
            B_resource[
                "variant"
            ]
            == variant
        ]
        .iloc[0]
    )


    resource_values.append(
        float(
            row[
                "MeanTrainingShots"
            ]
        )
        / 1e6
    )


axes[1].bar(
    range(3),
    resource_values,
)


axes[1].set_xticks(
    range(3)
)

axes[1].set_xticklabels(
    [
        "δ=.01",
        "δ=.10",
        "No\nrollback",
    ]
)

axes[1].set_ylabel(
    "Mean training shots (millions)"
)

axes[1].set_title(
    "Quantum measurement cost"
)

axes[1].grid(
    axis="y",
    alpha=0.25,
)


fig.suptitle(
    "QTrust ablation study"
)

fig.tight_layout()


save_figure(
    fig,
    "Fig_J2_Ablation",
)



# FIGURE 3
# Experiment-C Noise Robustness


noise_order = [
    "C_Readout_002",
    "C_Depolarizing_0005",
]


noise_titles = {
    "C_Readout_002":
        "Readout error = 0.02",

    "C_Depolarizing_0005":
        "Depolarizing p = 0.005",
}


quantum_methods = [
    "Fixed-QPPO-128",
    "Fixed-QPPO-256",
    "QTrust-PPO-128",
]


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        8.2,
        3.8,
    ),
)


for ax, variant in zip(
    axes,
    noise_order,
):

    subset = (
        C_RESULTS[
            C_RESULTS[
                "variant"
            ]
            == variant
        ]
        .copy()
    )


    for x, method in enumerate(
        quantum_methods
    ):

        values = (
            subset[
                subset[
                    "method"
                ]
                == method
            ]
            .sort_values(
                "seed"
            )[
                "mean_eval_reward"
            ]
            .to_numpy(
                dtype=float
            )
        )


        ax.scatter(
            x
            + seed_offsets(
                len(values)
            ),
            values,
            s=18,
            alpha=0.7,
        )


        mean_ci_plot(
            ax,
            values,
            x,
        )


    ax.set_xticks(
        range(3)
    )

    ax.set_xticklabels(
        [
            "Fixed-128",
            "Fixed-256",
            "QTrust",
        ],
        rotation=15,
    )

    ax.set_title(
        noise_titles[
            variant
        ]
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )


axes[0].set_ylabel(
    "Final evaluation reward"
)


fig.suptitle(
    "Quantum-policy robustness under output noise"
)

fig.tight_layout()


save_figure(
    fig,
    "Fig_J3_Noise_Robustness",
)



# FIGURE 4
# Experiment-D Depth Scalability


depth_rows = []


for depth, variant in [
    (
        1,
        "D_Layers_1",
    ),
    (
        4,
        "D_Layers_4",
    ),
]:

    subset = (
        D_RESULTS[
            D_RESULTS[
                "variant"
            ]
            == variant
        ]
        .copy()
    )


    for _, row in (
        subset.iterrows()
    ):

        depth_rows.append(
            {
                "depth":
                    depth,

                "seed":
                    int(
                        row[
                            "seed"
                        ]
                    ),

                "reward":
                    float(
                        row[
                            "mean_eval_reward"
                        ]
                    ),
            }
        )


depth3_reference = (
    A_RESULTS[
        (
            A_RESULTS[
                "environment"
            ]
            == "CartPole-v1"
        )
        &
        (
            A_RESULTS[
                "method"
            ]
            == "QTrust-PPO-128"
        )
    ]
)


for _, row in (
    depth3_reference.iterrows()
):

    depth_rows.append(
        {
            "depth":
                3,

            "seed":
                int(
                    row[
                        "seed"
                    ]
                ),

            "reward":
                float(
                    row[
                        "mean_eval_reward"
                    ]
                ),
        }
    )


depth_df = pd.DataFrame(
    depth_rows
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        8.2,
        3.8,
    ),
)


for x, depth in enumerate(
    [
        1,
        3,
        4,
    ]
):

    values = (
        depth_df[
            depth_df[
                "depth"
            ]
            == depth
        ]
        .sort_values(
            "seed"
        )[
            "reward"
        ]
        .to_numpy(
            dtype=float
        )
    )


    axes[0].scatter(
        x
        + seed_offsets(
            len(values)
        ),
        values,
        s=18,
        alpha=0.7,
    )


    mean_ci_plot(
        axes[0],
        values,
        x,
    )


axes[0].set_xticks(
    range(3)
)

axes[0].set_xticklabels(
    [
        "1 layer\n8 params",
        "3 layers\n24 params",
        "4 layers\n32 params",
    ]
)

axes[0].set_ylabel(
    "Final evaluation reward"
)

axes[0].set_title(
    "Reward vs circuit depth"
)

axes[0].grid(
    axis="y",
    alpha=0.25,
)


D_diag = (
    D_RESULTS
    .groupby(
        "layers",
        as_index=False,
    )
    .agg(
        MeanShots=(
            "training_quantum_shots",
            "mean",
        ),
        Rollbacks=(
            "rolled_back_updates",
            "sum",
        ),
        MeanRuntime=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


xvals = np.arange(
    len(D_diag)
)


axes[1].bar(
    xvals,
    D_diag[
        "MeanShots"
    ]
    / 1e6,
)


axes[1].set_xticks(
    xvals
)

axes[1].set_xticklabels(
    [
        f"{int(x)} layer"
        if int(x) == 1
        else f"{int(x)} layers"
        for x in D_diag[
            "layers"
        ]
    ]
)

axes[1].set_ylabel(
    "Mean training shots (millions)"
)

axes[1].set_title(
    "Measurement cost"
)

axes[1].grid(
    axis="y",
    alpha=0.25,
)


fig.suptitle(
    "VQC depth scalability"
)

fig.tight_layout()


save_figure(
    fig,
    "Fig_J4_Depth_Scalability",
)



# FIGURE 5
# Experiment-E Confidence-Bound Benchmark


E_plot = (
    E_SUMMARY
    .copy()
)


method_display = [
    "Clopper-Pearson",
    "Wilson",
    "Hoeffding",
]


E_plot = (
    E_plot
    .set_index(
        "Method"
    )
    .loc[
        method_display
    ]
    .reset_index()
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        8.2,
        3.8,
    ),
)


axes[0].bar(
    range(3),
    100.0
    * E_plot[
        "Resolved Coverage"
    ],
)


axes[0].set_xticks(
    range(3)
)

axes[0].set_xticklabels(
    [
        "Exact CP",
        "Wilson",
        "Hoeffding",
    ]
)

axes[0].set_ylabel(
    "Resolved coverage (%)"
)

axes[0].set_title(
    "Decision coverage"
)

axes[0].grid(
    axis="y",
    alpha=0.25,
)


for i, value in enumerate(
    100.0
    * E_plot[
        "Resolved Coverage"
    ]
):

    axes[0].text(
        i,
        value,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
    )


axes[1].bar(
    range(3),
    E_plot[
        "Mean Total Probability Shots"
    ],
)


axes[1].set_xticks(
    range(3)
)

axes[1].set_xticklabels(
    [
        "Exact CP",
        "Wilson",
        "Hoeffding",
    ]
)

axes[1].set_ylabel(
    "Mean total probability shots"
)

axes[1].set_title(
    "Measurement cost"
)

axes[1].grid(
    axis="y",
    alpha=0.25,
)


fig.suptitle(
    "Finite-shot confidence-bound benchmark"
)

fig.tight_layout()


save_figure(
    fig,
    "Fig_J5_Confidence_Bounds",
)



# FIGURE 6
# Primary paired effects

# QTrust minus comparator.


forest_A = (
    A_INFERENCE
    .copy()
)


forest_A[
    "Label"
] = (
    forest_A[
        "Environment"
    ].map(
        ENV_LABELS
    )
    + " | "
    + forest_A[
        "Reference"
    ].map(
        METHOD_LABELS
    )
)


forest_A = (
    forest_A
    .reset_index(
        drop=True
    )
)


y = np.arange(
    len(
        forest_A
    )
)


means = (
    forest_A[
        "Mean Difference"
    ]
    .to_numpy(
        dtype=float
    )
)


low = (
    forest_A[
        "Bootstrap CI Low"
    ]
    .to_numpy(
        dtype=float
    )
)


high = (
    forest_A[
        "Bootstrap CI High"
    ]
    .to_numpy(
        dtype=float
    )
)


fig, ax = plt.subplots(
    figsize=(
        7.4,
        5.2,
    )
)


ax.errorbar(
    means,
    y,
    xerr=[
        means - low,
        high - means,
    ],
    fmt="o",
    capsize=3,
)


ax.axvline(
    0.0,
    linewidth=1.0,
    linestyle="--",
)


ax.set_yticks(
    y
)

ax.set_yticklabels(
    forest_A[
        "Label"
    ]
)

ax.invert_yaxis()

ax.set_xlabel(
    "Paired mean reward difference: QTrust − comparator"
)

ax.set_title(
    "Main paired effects with 95% bootstrap CIs"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


fig.tight_layout()


save_figure(
    fig,
    "Fig_J6_Main_Paired_Effects",
)



# FIGURE 7

# B + C + D


extension_rows = []


for _, row in (
    B_INFERENCE.iterrows()
):

    extension_rows.append(
        {
            "Label":
                f"B | {row['Target']}",

            "Mean":
                row[
                    "Mean Difference"
                ],

            "Low":
                row[
                    "Bootstrap CI Low"
                ],

            "High":
                row[
                    "Bootstrap CI High"
                ],
        }
    )


for _, row in (
    C_WITHIN_INFERENCE.iterrows()
):

    short_noise = (
        "Readout"
        if row[
            "Noise Variant"
        ]
        == "C_Readout_002"
        else "Depolarizing"
    )


    extension_rows.append(
        {
            "Label":
                (
                    f"C | {short_noise} | "
                    f"vs {METHOD_LABELS[row['Reference']]}"
                ),

            "Mean":
                row[
                    "Mean Difference"
                ],

            "Low":
                row[
                    "Bootstrap CI Low"
                ],

            "High":
                row[
                    "Bootstrap CI High"
                ],
        }
    )


for _, row in (
    D_INFERENCE.iterrows()
):

    extension_rows.append(
        {
            "Label":
                f"D | {int(row['Layers'])} layers vs 3",

            "Mean":
                row[
                    "Mean Difference"
                ],

            "Low":
                row[
                    "Bootstrap CI Low"
                ],

            "High":
                row[
                    "Bootstrap CI High"
                ],
        }
    )


extension_forest = (
    pd.DataFrame(
        extension_rows
    )
)


y = np.arange(
    len(
        extension_forest
    )
)


means = (
    extension_forest[
        "Mean"
    ]
    .to_numpy(
        dtype=float
    )
)


low = (
    extension_forest[
        "Low"
    ]
    .to_numpy(
        dtype=float
    )
)


high = (
    extension_forest[
        "High"
    ]
    .to_numpy(
        dtype=float
    )
)


fig, ax = plt.subplots(
    figsize=(
        7.5,
        5.4,
    )
)


ax.errorbar(
    means,
    y,
    xerr=[
        means - low,
        high - means,
    ],
    fmt="o",
    capsize=3,
)


ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)


ax.set_yticks(
    y
)

ax.set_yticklabels(
    extension_forest[
        "Label"
    ]
)

ax.invert_yaxis()

ax.set_xlabel(
    "Paired mean reward difference"
)

ax.set_title(
    "Journal-extension paired effects with 95% bootstrap CIs"
)

ax.grid(
    axis="x",
    alpha=0.25,
)


fig.tight_layout()


save_figure(
    fig,
    "Fig_J7_Extension_Paired_Effects",
)



# OPTIONAL FIGURE 8
# QTrust training dynamics

# We do not fabricate dynamics.


DYNAMICS_GENERATED = False
DYNAMICS_SOURCE_COUNT = 0


candidate_update_files = []


for root in [
    JOURNAL_ROOT
    / "experiment_A_frozen",
    JOURNAL_ROOT,
]:

    if root.exists():

        for path in root.rglob(
            "*.csv"
        ):

            name = (
                path.name.lower()
            )


            if (
                "update"
                in name
                or
                "trace"
                in name
            ):

                candidate_update_files.append(
                    path
                )


dynamics_frames = []


for path in candidate_update_files:

    try:

        frame = pd.read_csv(
            path
        )


        normalized_columns = {
            str(col).lower():
                col
            for col in frame.columns
        }


        required_columns = [
            "update",
            "unresolved_fraction",
            "mean_qtrust_shots",
        ]


        if not all(
            name
            in normalized_columns
            for name in required_columns
        ):

            continue


        temp = pd.DataFrame(
            {
                "update":
                    pd.to_numeric(
                        frame[
                            normalized_columns[
                                "update"
                            ]
                        ],
                        errors="coerce",
                    ),

                "unresolved_fraction":
                    pd.to_numeric(
                        frame[
                            normalized_columns[
                                "unresolved_fraction"
                            ]
                        ],
                        errors="coerce",
                    ),

                "mean_qtrust_shots":
                    pd.to_numeric(
                        frame[
                            normalized_columns[
                                "mean_qtrust_shots"
                            ]
                        ],
                        errors="coerce",
                    ),
            }
        )


        temp = (
            temp
            .dropna()
            .copy()
        )


        if len(
            temp
        ) == 0:

            continue


        temp[
            "source_file"
        ] = str(
            path
        )


        dynamics_frames.append(
            temp
        )


    except Exception:

        continue


if dynamics_frames:

    DYNAMICS_SOURCE_COUNT = (
        len(
            dynamics_frames
        )
    )


    dynamics = pd.concat(
        dynamics_frames,
        ignore_index=True,
    )


    dynamics_summary = (

        dynamics

        .groupby(
            "update",
            as_index=False,
        )

        .agg(
            unresolved_mean=(
                "unresolved_fraction",
                "mean",
            ),

            unresolved_sd=(
                "unresolved_fraction",
                "std",
            ),

            shots_mean=(
                "mean_qtrust_shots",
                "mean",
            ),

            shots_sd=(
                "mean_qtrust_shots",
                "std",
            ),

            n=(
                "source_file",
                "count",
            ),
        )
    )


    # Only generate if more than one trace contributed.
    if (
        dynamics_summary[
            "n"
        ].max()
        >= 2
    ):

        n = (
            dynamics_summary[
                "n"
            ]
            .to_numpy(
                dtype=float
            )
        )


        critical = np.array(
            [
                (
                    student_t.ppf(
                        0.975,
                        int(x) - 1,
                    )
                    if x > 1
                    else np.nan
                )
                for x in n
            ]
        )


        unresolved_half = (
            critical
            * dynamics_summary[
                "unresolved_sd"
            ].to_numpy(
                dtype=float
            )
            / np.sqrt(n)
        )


        shots_half = (
            critical
            * dynamics_summary[
                "shots_sd"
            ].to_numpy(
                dtype=float
            )
            / np.sqrt(n)
        )


        fig, axes = plt.subplots(
            2,
            1,
            figsize=(
                7.5,
                5.5,
            ),
            sharex=True,
        )


        x = (
            dynamics_summary[
                "update"
            ].to_numpy(
                dtype=float
            )
        )


        unresolved_mean = (
            dynamics_summary[
                "unresolved_mean"
            ].to_numpy(
                dtype=float
            )
        )


        shots_mean = (
            dynamics_summary[
                "shots_mean"
            ].to_numpy(
                dtype=float
            )
        )


        axes[0].plot(
            x,
            unresolved_mean,
        )


        axes[0].fill_between(
            x,
            unresolved_mean
            - unresolved_half,
            unresolved_mean
            + unresolved_half,
            alpha=0.2,
        )


        axes[0].set_ylabel(
            "Unresolved fraction"
        )

        axes[0].grid(
            alpha=0.25,
        )


        axes[1].plot(
            x,
            shots_mean,
        )


        axes[1].fill_between(
            x,
            shots_mean
            - shots_half,
            shots_mean
            + shots_half,
            alpha=0.2,
        )


        axes[1].set_ylabel(
            "Adaptive certification shots"
        )

        axes[1].set_xlabel(
            "PPO update"
        )

        axes[1].grid(
            alpha=0.25,
        )


        fig.suptitle(
            "QTrust certification dynamics during training"
        )


        fig.tight_layout()


        save_figure(
            fig,
            "Fig_J8_QTrust_Training_Dynamics",
        )


        DYNAMICS_GENERATED = True


if DYNAMICS_GENERATED:

    print(
        "Training-dynamics figure: GENERATED"
    )

else:

    print(
        "Training-dynamics figure: not regenerated "
        "because complete compatible A update traces "
        "were not found. No data were fabricated."
    )



# TABLE 1
# Primary performance


TABLE1 = (
    A_SEED_SUMMARY
    .copy()
)


TABLE1[
    "Environment"
] = (
    TABLE1[
        "environment"
    ].map(
        ENV_LABELS
    )
)


TABLE1[
    "Method"
] = (
    TABLE1[
        "method"
    ].map(
        METHOD_LABELS
    )
)


TABLE1 = TABLE1[
    [
        "Environment",
        "Method",
        "N Seeds",
        "Mean Reward",
        "SD Reward",
        "95% t-CI Low",
        "95% t-CI High",
    ]
]


TABLE1.to_csv(
    TABLES_DIR
    / "Table_J1_Main_Performance.csv",
    index=False,
)


safe_to_latex(
    TABLE1,
    TABLES_DIR
    / "Table_J1_Main_Performance.tex",
    "Final evaluation performance across training seeds.",
    "tab:main_performance",
)



# TABLE 2
# Primary paired statistics


TABLE2 = (
    A_INFERENCE[
        [
            "Environment",
            "Reference",
            "Mean Difference",
            "Bootstrap CI Low",
            "Bootstrap CI High",
            "Raw P",
            "Holm P",
            "Paired Rank-Biserial",
        ]
    ]
    .copy()
)


TABLE2[
    "Environment"
] = (
    TABLE2[
        "Environment"
    ].map(
        ENV_LABELS
    )
)


TABLE2[
    "Reference"
] = (
    TABLE2[
        "Reference"
    ].map(
        METHOD_LABELS
    )
)


TABLE2.to_csv(
    TABLES_DIR
    / "Table_J2_Main_Paired_Inference.csv",
    index=False,
)


safe_to_latex(
    TABLE2,
    TABLES_DIR
    / "Table_J2_Main_Paired_Inference.tex",
    "Paired QTrust comparisons. Differences are QTrust minus comparator.",
    "tab:main_paired",
)



# TABLE 3
# Experiment-B Ablations


B_SUMMARY = (

    B_RESULTS

    .groupby(
        "variant",
        as_index=False,
    )

    .agg(
        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Adaptive_Shots=(
            "mean_adaptive_shots",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),
    )
)


TABLE3 = (
    B_SUMMARY.merge(
        B_INFERENCE[
            [
                "Target",
                "Mean Difference",
                "Holm P",
            ]
        ],
        left_on="variant",
        right_on="Target",
        how="left",
    )
)


TABLE3 = TABLE3[
    [
        "variant",
        "Mean_Reward",
        "SD_Reward",
        "Mean_Unresolved",
        "Mean_Adaptive_Shots",
        "Mean_Training_Shots",
        "Accepted_Updates",
        "Rolled_Back_Updates",
        "Mean Difference",
        "Holm P",
    ]
]


TABLE3.to_csv(
    TABLES_DIR
    / "Table_J3_Ablations.csv",
    index=False,
)


safe_to_latex(
    TABLE3,
    TABLES_DIR
    / "Table_J3_Ablations.tex",
    "QTrust ablation results on CartPole-v1.",
    "tab:ablations",
)



# TABLE 4
# Experiment-C Noise


C_SUMMARY_TABLE = (

    C_RESULTS

    .groupby(
        [
            "variant",
            "method",
        ],
        as_index=False,
    )

    .agg(
        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),
    )
)


TABLE4 = (
    C_SUMMARY_TABLE
    .copy()
)


TABLE4.to_csv(
    TABLES_DIR
    / "Table_J4_Noise_Robustness.csv",
    index=False,
)


safe_to_latex(
    TABLE4,
    TABLES_DIR
    / "Table_J4_Noise_Robustness.tex",
    "Noise-robustness results for finite-shot quantum PPO methods.",
    "tab:noise",
)


C_WITHIN_INFERENCE.to_csv(
    TABLES_DIR
    / "Table_J4b_Noise_Paired_Inference.csv",
    index=False,
)


safe_to_latex(
    C_WITHIN_INFERENCE,
    TABLES_DIR
    / "Table_J4b_Noise_Paired_Inference.tex",
    "Paired QTrust comparisons under output-noise conditions.",
    "tab:noise_paired",
)



# TABLE 5
# Experiment-D Depth


D_SUMMARY_TABLE = (

    D_RESULTS

    .groupby(
        [
            "variant",
            "layers",
            "quantum_parameters",
        ],
        as_index=False,
    )

    .agg(
        Mean_Reward=(
            "mean_eval_reward",
            "mean",
        ),

        SD_Reward=(
            "mean_eval_reward",
            "std",
        ),

        Mean_Unresolved=(
            "mean_unresolved_fraction",
            "mean",
        ),

        Mean_Training_Shots=(
            "training_quantum_shots",
            "mean",
        ),

        Accepted_Updates=(
            "accepted_updates",
            "sum",
        ),

        Rolled_Back_Updates=(
            "rolled_back_updates",
            "sum",
        ),

        Mean_Runtime_s=(
            "wall_runtime_seconds",
            "mean",
        ),
    )
)


TABLE5 = (
    D_SUMMARY_TABLE
    .merge(
        D_INFERENCE[
            [
                "Target",
                "Mean Difference",
                "Holm P",
            ]
        ],
        left_on="variant",
        right_on="Target",
        how="left",
    )
)


TABLE5.to_csv(
    TABLES_DIR
    / "Table_J5_Depth_Scalability.csv",
    index=False,
)


safe_to_latex(
    TABLE5,
    TABLES_DIR
    / "Table_J5_Depth_Scalability.tex",
    "VQC depth-scaling results relative to the three-layer Experiment-A reference.",
    "tab:depth",
)



# TABLE 6
# Experiment-E Confidence Methods


TABLE6 = (
    E_SUMMARY
    .copy()
)


TABLE6.to_csv(
    TABLES_DIR
    / "Table_J6_Confidence_Bounds.csv",
    index=False,
)


safe_to_latex(
    TABLE6,
    TABLES_DIR
    / "Table_J6_Confidence_Bounds.tex",
    "Controlled finite-shot confidence-bound benchmark.",
    "tab:confidence_bounds",
)


E_PAIRED.to_csv(
    TABLES_DIR
    / "Table_J6b_Confidence_Paired.csv",
    index=False,
)


safe_to_latex(
    E_PAIRED,
    TABLES_DIR
    / "Table_J6b_Confidence_Paired.tex",
    "Paired comparison of Wilson and Hoeffding bounds against exact Clopper-Pearson bounds.",
    "tab:confidence_paired",
)



# CREATE RESULTS INTERPRETATION GUARDRAILS


GUARDRAILS = """
QTrust-PPO JOURNAL RESULT INTERPRETATION GUARDRAILS

1. PPO clipping is not a trust-region guarantee.
   QTrust certifies finite-shot clipped-versus-unclipped
   surrogate decisions under the stated confidence construction.

2. The exact finite-horizon theoretical guarantee is associated
   with the Clopper-Pearson implementation using Bonferroni
   allocation across stages and four one-sided tails per stage.

3. Wilson is an empirical comparator in Experiment E.
   Its slightly higher observed coverage and slightly lower shot
   cost do not transfer the exact Clopper-Pearson guarantee to Wilson.

4. Resolved classification accuracy must always be reported
   together with unresolved rate or resolved coverage.

5. CartPole Experiment A:
   Classical PPO obtains substantially higher final reward than
   QTrust-PPO. This negative result must remain visible.

6. Acrobot Experiment A:
   Quantum methods exhibit reward saturation around the environment
   floor. Do not claim quantum learning superiority.

7. LunarLander Experiment A:
   QTrust has a large numerical reward difference relative to
   Classical PPO, but the QTrust actor accepted zero updates in the
   frozen Experiment-A diagnostics. Therefore this result must not
   be described as evidence that QTrust learned a superior policy.

8. Experiment B:
   Reward differences among the tested QTrust ablations are not
   significant after Holm correction. The main value is the
   confidence/resource/rollback tradeoff.

9. Experiment C:
   QTrust exceeds Fixed-QPPO-256 under the depolarizing condition
   after Holm correction. Other tested noisy reward comparisons
   are not statistically significant after correction.

10. Experiment D:
    No tested depth comparison is significant after Holm correction.
    Deeper circuits nevertheless show substantially greater resource
    demand and more rollback events in the observed runs.

11. Experiment E:
    All three methods achieved perfect resolved accuracy in this
    controlled sample. This does not mean they provide identical
    theoretical guarantees.

12. Training seed is the inferential unit.
    The 30 evaluation episodes per seed are repeated measurements,
    not independent inferential replicates.
""".strip()


(
    TABLES_DIR
    / "RESULT_INTERPRETATION_GUARDRAILS.txt"
).write_text(
    GUARDRAILS,
    encoding="utf-8",
)



# FIGURE MANIFEST


figure_rows = [
    {
        "Figure":
            "Fig_J1_Main_Performance",

        "Recommended placement":
            "MAIN",

        "Purpose":
            "Primary three-environment performance result",
    },

    {
        "Figure":
            "Fig_J2_Ablation",

        "Recommended placement":
            "MAIN or APPENDIX",

        "Purpose":
            "Confidence/resource ablation",
    },

    {
        "Figure":
            "Fig_J3_Noise_Robustness",

        "Recommended placement":
            "MAIN",

        "Purpose":
            "Noise robustness",
    },

    {
        "Figure":
            "Fig_J4_Depth_Scalability",

        "Recommended placement":
            "APPENDIX or MAIN",

        "Purpose":
            "Circuit-depth scaling",
    },

    {
        "Figure":
            "Fig_J5_Confidence_Bounds",

        "Recommended placement":
            "MAIN",

        "Purpose":
            "Confidence-bound benchmark",
    },

    {
        "Figure":
            "Fig_J6_Main_Paired_Effects",

        "Recommended placement":
            "MAIN",

        "Purpose":
            "Primary paired statistical effects",
    },

    {
        "Figure":
            "Fig_J7_Extension_Paired_Effects",

        "Recommended placement":
            "APPENDIX",

        "Purpose":
            "B-D paired effect summary",
    },
]


if DYNAMICS_GENERATED:

    figure_rows.append(
        {
            "Figure":
                "Fig_J8_QTrust_Training_Dynamics",

            "Recommended placement":
                "MAIN",

            "Purpose":
                "Evolution of unresolved fraction and adaptive shots",
        }
    )


FIGURE_MANIFEST = (
    pd.DataFrame(
        figure_rows
    )
)


FIGURE_MANIFEST.to_csv(
    FIGURES_DIR
    / "figure_manifest.csv",
    index=False,
)



# TABLE MANIFEST


TABLE_MANIFEST = pd.DataFrame(
    [
        {
            "Table":
                "Table_J1_Main_Performance",

            "Placement":
                "MAIN",
        },

        {
            "Table":
                "Table_J2_Main_Paired_Inference",

            "Placement":
                "MAIN",
        },

        {
            "Table":
                "Table_J3_Ablations",

            "Placement":
                "APPENDIX",
        },

        {
            "Table":
                "Table_J4_Noise_Robustness",

            "Placement":
                "MAIN or APPENDIX",
        },

        {
            "Table":
                "Table_J4b_Noise_Paired_Inference",

            "Placement":
                "APPENDIX",
        },

        {
            "Table":
                "Table_J5_Depth_Scalability",

            "Placement":
                "APPENDIX",
        },

        {
            "Table":
                "Table_J6_Confidence_Bounds",

            "Placement":
                "MAIN",
        },

        {
            "Table":
                "Table_J6b_Confidence_Paired",

            "Placement":
                "APPENDIX",
        },
    ]
)


TABLE_MANIFEST.to_csv(
    TABLES_DIR
    / "table_manifest.csv",
    index=False,
)






MAIN_PAPER_PLAN = """
RECOMMENDED JOURNAL RESULTS STRUCTURE

MAIN TEXT

Figure J1
Main three-environment seed-level performance.

Figure J3
Noise robustness.

Figure J5
Clopper-Pearson, Wilson, and Hoeffding controlled benchmark.

Figure J6
Paired Experiment-A effect estimates.

Figure J8
QTrust training dynamics, if the complete frozen traces were
available and the figure was regenerated.

Table J1
Main seed-level performance with 95% t-confidence intervals.

Table J2
Primary paired inferential results.

Table J6
Controlled confidence-bound benchmark.

RESULTS NARRATIVE

A. Main benchmark
Report all positive and negative results. Do not frame QTrust as
universally reward-superior.

B. Reliability and resources
Emphasize resolved accuracy, abstention/unresolved behavior,
adaptive shots, and rollback mechanism.

C. Ablation
Use Experiment B to demonstrate the confidence-cost tradeoff,
rather than claiming reward improvement.

D. Noise
Report that QTrust versus Fixed-256 under depolarizing noise is
the only tested within-noise comparison significant after Holm
correction in Experiment C.

E. Circuit depth
Report the absence of a significant reward improvement together
with the observed increase in measurement/runtime burden.

F. Confidence-bound benchmark
Keep exact Clopper-Pearson as the theoretically certified method.
Discuss Wilson as a close empirical efficiency comparator and
Hoeffding as more conservative in this controlled benchmark.

APPENDIX / SUPPLEMENT

Figure J2
Ablation details.

Figure J4
Depth scalability.

Figure J7
Extension paired-effect forest.

Tables J3, J4b, J5, J6b
Detailed extension statistics.
""".strip()


(
    JOURNAL_ROOT
    / "FINAL_JOURNAL_RESULTS_PLAN.txt"
).write_text(
    MAIN_PAPER_PLAN,
    encoding="utf-8",
)



# CELL-9 PROVENANCE


generated_figure_pdfs = sorted(
    FIGURES_DIR.glob(
        "Fig_J*.pdf"
    )
)


generated_figure_pngs = sorted(
    FIGURES_DIR.glob(
        "Fig_J*.png"
    )
)


generated_tables = sorted(
    TABLES_DIR.glob(
        "Table_J*.csv"
    )
)


CELL9_MANIFEST = {

    "cell":
        9,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_hash":
        EXPECTED_PROTOCOL_HASH,

    "journal_protocol_hash":
        JOURNAL_PROTOCOL_HASH,

    "pdf_figures":
        len(
            generated_figure_pdfs
        ),

    "png_figures":
        len(
            generated_figure_pngs
        ),

    "csv_tables":
        len(
            generated_tables
        ),

    "figure_dpi":
        600,

    "vector_format":
        "PDF",

    "training_dynamics_regenerated":
        DYNAMICS_GENERATED,

    "training_dynamics_source_files":
        DYNAMICS_SOURCE_COUNT,
}


CELL9_HASH = (
    hashlib.sha256(
        json.dumps(
            CELL9_MANIFEST,
            sort_keys=True,
            separators=(",", ":"),
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()[:16]
)


CELL9_MANIFEST[
    "cell9_artifact_hash"
] = (
    CELL9_HASH
)


(
    LOGS_DIR
    / "cell09_publication_artifacts.json"
).write_text(
    json.dumps(
        CELL9_MANIFEST,
        indent=2,
    ),
    encoding="utf-8",
)






manifest_path = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)


manifest[
    "publication_artifacts"
] = {

    "status":
        "COMPLETE",

    "cell":
        9,

    "artifact_hash":
        CELL9_HASH,

    "pdf_figures":
        len(
            generated_figure_pdfs
        ),

    "png_figures":
        len(
            generated_figure_pngs
        ),

    "csv_tables":
        len(
            generated_tables
        ),

    "vector_figures":
        True,

    "png_dpi":
        600,

    "training_dynamics_regenerated":
        DYNAMICS_GENERATED,
}


manifest[
    "current_completed_cell"
] = 9


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)



# REPORT


print()
print("=" * 94)
print("PUBLICATION ARTIFACTS COMPLETE")
print("=" * 94)


print(
    "PDF figures :",
    len(
        generated_figure_pdfs
    )
)


print(
    "PNG figures :",
    len(
        generated_figure_pngs
    )
)


print(
    "CSV tables  :",
    len(
        generated_tables
    )
)


print(
    "Vector PDF  : YES"
)


print(
    "PNG DPI     : 600"
)


print(
    "Dynamics fig:",
    (
        "GENERATED"
        if DYNAMICS_GENERATED
        else "NOT REGENERATED"
    )
)


print(
    "Artifact hash:",
    CELL9_HASH
)


print()
print(
    "Figures:"
)


for path in generated_figure_pdfs:

    print(
        " ",
        path.name
    )


print()
print(
    "Tables:"
)


for path in generated_tables:

    print(
        " ",
        path.name
    )


print()
print(
    "Interpretation guardrails:",
    TABLES_DIR
    / "RESULT_INTERPRETATION_GUARDRAILS.txt"
)


print(
    "Journal results plan:",
    JOURNAL_ROOT
    / "FINAL_JOURNAL_RESULTS_PLAN.txt"
)


print("=" * 94)






CELL9_CHECKPOINT = (
    save_journal_checkpoint(
        9,
        "Publication_Artifacts_COMPLETE",
    )
)


print()
print("=" * 94)
print("CELL 9 COMPLETE")
print("=" * 94)

print(
    "Final journal figures and tables are frozen."
)

print(
    "Next: Cell 10 will perform final integrity checks "
    "and create the complete journal-extension ZIP."
)

In [ ]:
# No New Statistical Tests
# No Result Modification

# This cell:
# verifies Experiments A-E
# verifies protocol/provenance hashes
# verifies statistical outputs



# tests ZIP integrity



from pathlib import Path
from datetime import datetime, timezone

import os
import json
import shutil
import zipfile
import hashlib

import numpy as np
import pandas as pd






required = [
    "JOURNAL_ROOT",
    "EXP_B_DIR",
    "EXP_C_DIR",
    "EXP_D_DIR",
    "EXP_E_DIR",
    "STATS_DIR",
    "FIGURES_DIR",
    "TABLES_DIR",
    "LOGS_DIR",
    "JOURNAL_PROTOCOL_HASH",
    "EXPECTED_PROTOCOL_HASH",
    "save_journal_checkpoint",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing journal state: "
        + ", ".join(missing)
    )


print("=" * 96)
print("QTrust-PPO JOURNAL EXTENSION")
print("CELL 10 / 10")
print("FINAL INTEGRITY AUDIT + COMPLETE EXPORT")
print("=" * 96)






EXPECTED_PARENT_HASH = (
    "3cf4bcb18e1111b3"
)

EXPECTED_JOURNAL_HASH = (
    "4534e690e365694c"
)

EXPECTED_SEEDS = [
    11,
    23,
    37,
    53,
    71,
    89,
    107,
    131,
    157,
    181,
]


if (
    EXPECTED_PROTOCOL_HASH
    != EXPECTED_PARENT_HASH
):

    raise RuntimeError(
        "Parent hash changed."
    )


if (
    JOURNAL_PROTOCOL_HASH
    != EXPECTED_JOURNAL_HASH
):

    raise RuntimeError(
        "Journal hash changed."
    )



# VERIFY MASTER MANIFEST


MASTER_MANIFEST_PATH = (
    JOURNAL_ROOT
    / "journal_manifest.json"
)


if not MASTER_MANIFEST_PATH.exists():

    raise RuntimeError(
        "journal_manifest.json missing."
    )


MASTER_MANIFEST = json.loads(
    MASTER_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)



# VERIFY EXPERIMENT A


A_PATH = (
    JOURNAL_ROOT
    / "experiment_A_frozen"
    / "reference"
    / "all_completed_runs.csv"
)


if not A_PATH.exists():

    raise RuntimeError(
        "Authoritative Experiment-A "
        "all_completed_runs.csv is missing."
    )


A_FINAL = pd.read_csv(
    A_PATH
)


if len(A_FINAL) != 120:

    raise RuntimeError(
        f"Experiment A expected 120 runs; "
        f"found {len(A_FINAL)}."
    )


# Use already canonicalized Cell-8 version if present.
if "A_RESULTS" in globals():

    A_CHECK = A_RESULTS.copy()

    if len(A_CHECK) != 120:

        raise RuntimeError(
            "Canonical Experiment-A table != 120."
        )


    A_COUNTS = (
        A_CHECK
        .groupby(
            [
                "environment",
                "method",
            ]
        )
        .size()
    )


    if (
        len(A_COUNTS) != 12
        or
        not A_COUNTS.eq(10).all()
    ):

        raise RuntimeError(
            "Experiment-A group structure invalid."
        )


    for _, group in (
        A_CHECK.groupby(
            [
                "environment",
                "method",
            ]
        )
    ):

        seeds = sorted(
            group[
                "seed"
            ]
            .astype(int)
            .tolist()
        )


        if seeds != EXPECTED_SEEDS:

            raise RuntimeError(
                "Experiment-A seed validation failed."
            )


print(
    "Experiment A : 120 / 120 PASS"
)



# VERIFY EXPERIMENT B


B_PATH = (
    EXP_B_DIR
    / "experiment_B_all_runs.csv"
)


if not B_PATH.exists():

    raise RuntimeError(
        "Experiment-B master file missing."
    )


B_FINAL = pd.read_csv(
    B_PATH
)


if len(B_FINAL) != 30:

    raise RuntimeError(
        f"Experiment B expected 30 runs; "
        f"found {len(B_FINAL)}."
    )


B_COUNTS = (
    B_FINAL
    .groupby(
        "variant"
    )
    .size()
)


if (
    len(B_COUNTS) != 3
    or
    not B_COUNTS.eq(10).all()
):

    raise RuntimeError(
        "Experiment-B condition counts invalid."
    )


print(
    "Experiment B : 30 / 30 PASS"
)



# VERIFY EXPERIMENT C


C_PATH = (
    EXP_C_DIR
    / "experiment_C_all_runs.csv"
)


if not C_PATH.exists():

    raise RuntimeError(
        "Experiment-C master file missing."
    )


C_FINAL = pd.read_csv(
    C_PATH
)


if len(C_FINAL) != 60:

    raise RuntimeError(
        f"Experiment C expected 60 runs; "
        f"found {len(C_FINAL)}."
    )


C_COUNTS = (
    C_FINAL
    .groupby(
        [
            "variant",
            "method",
        ]
    )
    .size()
)


if (
    len(C_COUNTS) != 6
    or
    not C_COUNTS.eq(10).all()
):

    raise RuntimeError(
        "Experiment-C condition counts invalid."
    )


if not C_FINAL[
    "journal_protocol_hash"
].eq(
    EXPECTED_JOURNAL_HASH
).all():

    raise RuntimeError(
        "Experiment-C journal hash mismatch."
    )


print(
    "Experiment C : 60 / 60 PASS"
)



# VERIFY EXPERIMENT D


D_PATH = (
    EXP_D_DIR
    / "experiment_D_all_runs.csv"
)


if not D_PATH.exists():

    raise RuntimeError(
        "Experiment-D master file missing."
    )


D_FINAL = pd.read_csv(
    D_PATH
)


if len(D_FINAL) != 20:

    raise RuntimeError(
        f"Experiment D expected 20 runs; "
        f"found {len(D_FINAL)}."
    )


D_COUNTS = (
    D_FINAL
    .groupby(
        "variant"
    )
    .size()
)


if (
    len(D_COUNTS) != 2
    or
    not D_COUNTS.eq(10).all()
):

    raise RuntimeError(
        "Experiment-D condition counts invalid."
    )


expected_depth_params = {
    1: 8,
    4: 32,
}


for depth, params in (
    expected_depth_params.items()
):

    subset = (
        D_FINAL[
            D_FINAL[
                "layers"
            ]
            == depth
        ]
    )


    if len(subset) != 10:

        raise RuntimeError(
            f"Depth {depth} does not have 10 runs."
        )


    if not subset[
        "quantum_parameters"
    ].eq(
        params
    ).all():

        raise RuntimeError(
            f"Depth {depth} parameter count invalid."
        )


print(
    "Experiment D : 20 / 20 PASS"
)



# VERIFY EXPERIMENT E


E_PATH = (
    EXP_E_DIR
    / "experiment_E_all_trials.csv"
)


E_SUMMARY_PATH = (
    EXP_E_DIR
    / "experiment_E_summary.csv"
)


if not E_PATH.exists():

    raise RuntimeError(
        "Experiment-E trial file missing."
    )


if not E_SUMMARY_PATH.exists():

    raise RuntimeError(
        "Experiment-E summary missing."
    )


E_FINAL = pd.read_csv(
    E_PATH
)


E_FINAL_SUMMARY = pd.read_csv(
    E_SUMMARY_PATH
)


EXPECTED_E_ROWS = (
    22_800
    * 3
)


if len(E_FINAL) != EXPECTED_E_ROWS:

    raise RuntimeError(
        f"Experiment E expected "
        f"{EXPECTED_E_ROWS} method-level rows; "
        f"found {len(E_FINAL)}."
    )


E_METHOD_COUNTS = (
    E_FINAL[
        "Method"
    ]
    .value_counts()
)


for method in [
    "Clopper-Pearson",
    "Wilson",
    "Hoeffding",
]:

    if (
        E_METHOD_COUNTS.get(
            method,
            0,
        )
        != 22_800
    ):

        raise RuntimeError(
            f"Experiment-E count mismatch: {method}"
        )


if len(
    E_FINAL_SUMMARY
) != 3:

    raise RuntimeError(
        "Experiment-E summary should contain "
        "three confidence methods."
    )


print(
    "Experiment E : 22,800 paired paths × 3 methods PASS"
)



# VERIFY STATISTICAL OUTPUTS


EXPECTED_STAT_FILES = [

    "A_seed_level_summary.csv",

    "A_paired_inference.csv",

    "B_ablation_paired_inference.csv",

    "C_qtrust_vs_fixed_paired_inference.csv",

    "C_noise_vs_clean_paired_inference.csv",

    "D_depth_paired_inference.csv",

    "E_paired_confidence_method_analysis.csv",

    "statistical_analysis_protocol.json",
]


missing_stats = []


for filename in EXPECTED_STAT_FILES:

    path = (
        STATS_DIR
        / filename
    )


    if not path.exists():

        missing_stats.append(
            filename
        )


if missing_stats:

    raise RuntimeError(
        "Missing statistical outputs: "
        + ", ".join(
            missing_stats
        )
    )


A_INF = pd.read_csv(
    STATS_DIR
    / "A_paired_inference.csv"
)


B_INF = pd.read_csv(
    STATS_DIR
    / "B_ablation_paired_inference.csv"
)


C_INF = pd.read_csv(
    STATS_DIR
    / "C_qtrust_vs_fixed_paired_inference.csv"
)


D_INF = pd.read_csv(
    STATS_DIR
    / "D_depth_paired_inference.csv"
)


E_INF = pd.read_csv(
    STATS_DIR
    / "E_paired_confidence_method_analysis.csv"
)


if len(A_INF) != 9:
    raise RuntimeError(
        "Experiment-A inference should contain 9 comparisons."
    )


if len(B_INF) != 3:
    raise RuntimeError(
        "Experiment-B inference should contain 3 comparisons."
    )


if len(C_INF) != 4:
    raise RuntimeError(
        "Experiment-C within-noise inference "
        "should contain 4 comparisons."
    )


if len(D_INF) != 2:
    raise RuntimeError(
        "Experiment-D inference should contain 2 comparisons."
    )


if len(E_INF) != 2:
    raise RuntimeError(
        "Experiment-E paired inference "
        "should contain 2 comparator rows."
    )


print(
    "Statistical analysis : PASS"
)






EXPECTED_FIGURES = [

    "Fig_J1_Main_Performance",

    "Fig_J2_Ablation",

    "Fig_J3_Noise_Robustness",

    "Fig_J4_Depth_Scalability",

    "Fig_J5_Confidence_Bounds",

    "Fig_J6_Main_Paired_Effects",

    "Fig_J7_Extension_Paired_Effects",
]


for stem in EXPECTED_FIGURES:

    pdf_path = (
        FIGURES_DIR
        / f"{stem}.pdf"
    )


    png_path = (
        FIGURES_DIR
        / f"{stem}.png"
    )


    if not pdf_path.exists():

        raise RuntimeError(
            f"Missing figure PDF: {stem}"
        )


    if not png_path.exists():

        raise RuntimeError(
            f"Missing figure PNG: {stem}"
        )


    if pdf_path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty figure PDF: {stem}"
        )


    if png_path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty figure PNG: {stem}"
        )


PDF_FIGURES = sorted(
    FIGURES_DIR.glob(
        "Fig_J*.pdf"
    )
)


PNG_FIGURES = sorted(
    FIGURES_DIR.glob(
        "Fig_J*.png"
    )
)


if len(PDF_FIGURES) != 7:

    raise RuntimeError(
        f"Expected 7 final PDF figures; "
        f"found {len(PDF_FIGURES)}."
    )


if len(PNG_FIGURES) != 7:

    raise RuntimeError(
        f"Expected 7 final PNG figures; "
        f"found {len(PNG_FIGURES)}."
    )


print(
    "Publication figures : 7 PDF + 7 PNG PASS"
)






EXPECTED_TABLE_STEMS = [

    "Table_J1_Main_Performance",

    "Table_J2_Main_Paired_Inference",

    "Table_J3_Ablations",

    "Table_J4_Noise_Robustness",

    "Table_J4b_Noise_Paired_Inference",

    "Table_J5_Depth_Scalability",

    "Table_J6_Confidence_Bounds",

    "Table_J6b_Confidence_Paired",
]


for stem in EXPECTED_TABLE_STEMS:

    csv_path = (
        TABLES_DIR
        / f"{stem}.csv"
    )


    tex_path = (
        TABLES_DIR
        / f"{stem}.tex"
    )


    if not csv_path.exists():

        raise RuntimeError(
            f"Missing table CSV: {stem}"
        )


    if not tex_path.exists():

        raise RuntimeError(
            f"Missing table LaTeX: {stem}"
        )


if not (
    TABLES_DIR
    / "RESULT_INTERPRETATION_GUARDRAILS.txt"
).exists():

    raise RuntimeError(
        "Interpretation guardrails file missing."
    )


if not (
    JOURNAL_ROOT
    / "FINAL_JOURNAL_RESULTS_PLAN.txt"
).exists():

    raise RuntimeError(
        "Final journal results plan missing."
    )


print(
    "Publication tables : 8 CSV + 8 LaTeX PASS"
)



# VERIFY CRITICAL SOURCE FILES


CRITICAL_SOURCE_FILES = [

    JOURNAL_ROOT
    / "journal_engine.py",

    JOURNAL_ROOT
    / "journal_noise_engine.py",

    JOURNAL_ROOT
    / "journal_manifest.json",

    JOURNAL_ROOT
    / "journal_extension_protocol.json",

    JOURNAL_ROOT
    / "journal_run_manifest.csv",
]


for path in CRITICAL_SOURCE_FILES:

    if not path.exists():

        raise RuntimeError(
            f"Critical provenance file missing: {path}"
        )


print(
    "Source/provenance files : PASS"
)






def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as handle:

        while True:

            block = handle.read(
                chunk_size
            )


            if not block:

                break


            digest.update(
                block
            )


    return digest.hexdigest()


# Exclude old generated integrity files if this cell is rerun.
EXCLUDE_NAMES = {
    "FINAL_FILE_SHA256_MANIFEST.csv",
    "FINAL_INTEGRITY_REPORT.json",
    "FINAL_EXPORT_README.txt",
}


file_rows = []


for path in sorted(
    JOURNAL_ROOT.rglob("*")
):

    if not path.is_file():

        continue


    if path.name in EXCLUDE_NAMES:

        continue


    if path.suffix == ".tmp":

        continue


    relative = (
        path.relative_to(
            JOURNAL_ROOT
        )
    )


    file_rows.append(
        {
            "relative_path":
                str(relative),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


FILE_MANIFEST = pd.DataFrame(
    file_rows
)


if len(
    FILE_MANIFEST
) == 0:

    raise RuntimeError(
        "No files found for final manifest."
    )


FILE_MANIFEST_PATH = (
    JOURNAL_ROOT
    / "FINAL_FILE_SHA256_MANIFEST.csv"
)


FILE_MANIFEST.to_csv(
    FILE_MANIFEST_PATH,
    index=False,
)






STAT_PROTOCOL_PATH = (
    STATS_DIR
    / "statistical_analysis_protocol.json"
)


STAT_PROTOCOL = json.loads(
    STAT_PROTOCOL_PATH.read_text(
        encoding="utf-8"
    )
)


CELL9_LOG_PATH = (
    LOGS_DIR
    / "cell09_publication_artifacts.json"
)


CELL9_LOG = json.loads(
    CELL9_LOG_PATH.read_text(
        encoding="utf-8"
    )
)


FINAL_REPORT = {

    "project":
        "QTrust-PPO Journal Extension",

    "status":
        "COMPLETE",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_experiment_A_protocol_hash":
        EXPECTED_PARENT_HASH,

    "journal_extension_protocol_hash":
        EXPECTED_JOURNAL_HASH,

    "statistical_protocol_hash":
        STAT_PROTOCOL.get(
            "statistical_protocol_hash"
        ),

    "publication_artifact_hash":
        CELL9_LOG.get(
            "cell9_artifact_hash"
        ),

    "experiment_A": {

        "status":
            "FROZEN",

        "runs":
            120,

        "environments":
            3,

        "methods":
            4,

        "seeds_per_condition":
            10,
    },

    "experiment_B": {

        "status":
            "FROZEN",

        "runs":
            30,

        "conditions":
            3,

        "seeds_per_condition":
            10,
    },

    "experiment_C": {

        "status":
            "FROZEN",

        "runs":
            60,

        "conditions":
            6,

        "seeds_per_condition":
            10,
    },

    "experiment_D": {

        "status":
            "FROZEN",

        "runs":
            20,

        "new_depths":
            [
                1,
                4,
            ],

        "reference_depth":
            3,

        "seeds_per_new_depth":
            10,
    },

    "experiment_E": {

        "status":
            "FROZEN",

        "paired_trial_paths":
            22_800,

        "confidence_methods":
            3,

        "method_level_rows":
            68_400,
    },

    "new_gpu_runs_B_to_D":
        110,

    "publication_artifacts": {

        "vector_pdf_figures":
            7,

        "png_600dpi_figures":
            7,

        "csv_tables":
            8,

        "latex_tables":
            8,

        "training_dynamics_regenerated":
            False,

        "reason":
            (
                "Complete compatible frozen Experiment-A "
                "update traces were not available in the "
                "journal workspace. No dynamics were fabricated."
            ),
    },

    "inferential_unit":
        "training seed",

    "evaluation_episodes_per_training_seed":
        30,

    "final_file_count_before_report":
        int(
            len(
                FILE_MANIFEST
            )
        ),

    "integrity":
        "PASS",
}


FINAL_REPORT_PATH = (
    JOURNAL_ROOT
    / "FINAL_INTEGRITY_REPORT.json"
)


FINAL_REPORT_PATH.write_text(
    json.dumps(
        FINAL_REPORT,
        indent=2,
    ),
    encoding="utf-8",
)






FINAL_README = f"""
QTrust-PPO JOURNAL EXTENSION
FINAL EXPORT

STATUS
COMPLETE

Authoritative parent Experiment A
Protocol hash: {EXPECTED_PARENT_HASH}
Runs: 120

Journal extension
Protocol hash: {EXPECTED_JOURNAL_HASH}

Experiment B
30 ablation runs

Experiment C
60 noise-robustness runs

Experiment D
20 depth-scalability runs

Total new GPU runs
110

Experiment E
22,800 paired controlled sample paths
3 confidence methods
68,400 method-level records

Statistics
Training seed is the inferential unit.
10 training seeds per condition.
30 evaluation episodes are repeated measurements within seed.
Paired Wilcoxon signed-rank tests.
Paired rank-biserial effect sizes.
Paired bootstrap 95% mean-difference intervals.
Holm correction within predefined comparison families.

Figures
7 vector PDF figures
7 PNG figures at 600 DPI

Tables
8 CSV publication tables
8 LaTeX publication tables

Important dynamics note
The original Experiment-A QTrust training-dynamics figure was not
reconstructed in Cell 9 because complete compatible frozen update
traces were not present in this journal workspace. No data were
fabricated. Use the previously verified Experiment-A publication
figure when assembling the final manuscript.

Interpretation
Read:
tables/RESULT_INTERPRETATION_GUARDRAILS.txt

Recommended paper structure
Read:
FINAL_JOURNAL_RESULTS_PLAN.txt

File checksums
Read:
FINAL_FILE_SHA256_MANIFEST.csv

Final integrity report
Read:
FINAL_INTEGRITY_REPORT.json
""".strip()


FINAL_README_PATH = (
    JOURNAL_ROOT
    / "FINAL_EXPORT_README.txt"
)


FINAL_README_PATH.write_text(
    FINAL_README,
    encoding="utf-8",
)






MASTER_MANIFEST = json.loads(
    MASTER_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


MASTER_MANIFEST[
    "final_export"
] = {

    "status":
        "COMPLETE",

    "cell":
        10,

    "parent_protocol_hash":
        EXPECTED_PARENT_HASH,

    "journal_protocol_hash":
        EXPECTED_JOURNAL_HASH,

    "experiment_A_runs":
        120,

    "experiment_B_runs":
        30,

    "experiment_C_runs":
        60,

    "experiment_D_runs":
        20,

    "experiment_E_paired_paths":
        22_800,

    "publication_pdf_figures":
        7,

    "publication_png_figures":
        7,

    "publication_csv_tables":
        8,

    "publication_latex_tables":
        8,

    "integrity":
        "PASS",
}


MASTER_MANIFEST[
    "current_completed_cell"
] = 10


MASTER_MANIFEST[
    "journal_extension_status"
] = (
    "COMPLETE"
)


MASTER_MANIFEST_PATH.write_text(
    json.dumps(
        MASTER_MANIFEST,
        indent=2,
    ),
    encoding="utf-8",
)



# REFRESH FILE HASH MANIFEST


# Do not hash the checksum file itself.


file_rows = []


for path in sorted(
    JOURNAL_ROOT.rglob("*")
):

    if not path.is_file():

        continue


    if path.name == (
        "FINAL_FILE_SHA256_MANIFEST.csv"
    ):

        continue


    if path.suffix == ".tmp":

        continue


    relative = (
        path.relative_to(
            JOURNAL_ROOT
        )
    )


    file_rows.append(
        {
            "relative_path":
                str(relative),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


FILE_MANIFEST = pd.DataFrame(
    file_rows
)


FILE_MANIFEST.to_csv(
    FILE_MANIFEST_PATH,
    index=False,
)






FINAL_ZIP = Path(
    "/kaggle/working/"
    "QTrust_PPO_Journal_Extension_FINAL_COMPLETE.zip"
)


TEMP_ZIP = Path(
    str(FINAL_ZIP)
    + ".tmp"
)


if TEMP_ZIP.exists():

    TEMP_ZIP.unlink()


if FINAL_ZIP.exists():

    FINAL_ZIP.unlink()


with zipfile.ZipFile(

    TEMP_ZIP,

    "w",

    compression=
        zipfile.ZIP_DEFLATED,

    compresslevel=6,

) as zf:


    for path in sorted(
        JOURNAL_ROOT.rglob("*")
    ):

        if not path.is_file():

            continue


        if path.suffix == ".tmp":

            continue


        relative = (
            path.relative_to(
                JOURNAL_ROOT.parent
            )
        )


        zf.write(
            path,
            arcname=str(
                relative
            ),
        )



# VERIFY ZIP CRC + REQUIRED FILES


with zipfile.ZipFile(
    TEMP_ZIP,
    "r",
) as zf:

    bad_file = (
        zf.testzip()
    )


    if bad_file is not None:

        raise RuntimeError(
            "Final ZIP CRC verification failed: "
            f"{bad_file}"
        )


    members = set(
        zf.namelist()
    )


    REQUIRED_ARCHIVE_MEMBERS = [

        "QTrust_PPO_Journal/"
        "journal_manifest.json",

        "QTrust_PPO_Journal/"
        "journal_engine.py",

        "QTrust_PPO_Journal/"
        "journal_noise_engine.py",

        "QTrust_PPO_Journal/"
        "FINAL_FILE_SHA256_MANIFEST.csv",

        "QTrust_PPO_Journal/"
        "FINAL_INTEGRITY_REPORT.json",

        "QTrust_PPO_Journal/"
        "FINAL_EXPORT_README.txt",

        "QTrust_PPO_Journal/"
        "experiment_B_ablations/"
        "experiment_B_all_runs.csv",

        "QTrust_PPO_Journal/"
        "experiment_C_noise/"
        "experiment_C_all_runs.csv",

        "QTrust_PPO_Journal/"
        "experiment_D_scalability/"
        "experiment_D_all_runs.csv",

        "QTrust_PPO_Journal/"
        "experiment_E_confidence/"
        "experiment_E_all_trials.csv",

        "QTrust_PPO_Journal/"
        "statistics/"
        "A_paired_inference.csv",

        "QTrust_PPO_Journal/"
        "figures/"
        "Fig_J1_Main_Performance.pdf",

        "QTrust_PPO_Journal/"
        "tables/"
        "Table_J1_Main_Performance.csv",
    ]


    missing_members = [

        member

        for member in REQUIRED_ARCHIVE_MEMBERS

        if member not in members
    ]


    if missing_members:

        raise RuntimeError(
            "Final ZIP missing required files: "
            + ", ".join(
                missing_members
            )
        )


os.replace(
    TEMP_ZIP,
    FINAL_ZIP,
)






FINAL_ZIP_SHA256 = (
    sha256_file(
        FINAL_ZIP
    )
)


FINAL_ZIP_SIZE_MB = (
    FINAL_ZIP.stat().st_size
    / (1024 ** 2)
)



# SAVE EXTERNAL ZIP CHECKSUM


FINAL_ZIP_CHECKSUM_PATH = Path(
    "/kaggle/working/"
    "QTrust_PPO_Journal_Extension_FINAL_COMPLETE_SHA256.txt"
)


FINAL_ZIP_CHECKSUM_PATH.write_text(
    (
        f"{FINAL_ZIP_SHA256}  "
        f"{FINAL_ZIP.name}\n"
    ),
    encoding="utf-8",
)



# CELL-10 PROVENANCE


CELL10_RECORD = {

    "cell":
        10,

    "status":
        "PASS",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_protocol_hash":
        EXPECTED_PARENT_HASH,

    "journal_protocol_hash":
        EXPECTED_JOURNAL_HASH,

    "experiment_A_runs":
        120,

    "experiment_B_runs":
        30,

    "experiment_C_runs":
        60,

    "experiment_D_runs":
        20,

    "experiment_E_trial_paths":
        22_800,

    "experiment_E_method_rows":
        68_400,

    "final_zip":
        str(
            FINAL_ZIP
        ),

    "final_zip_size_mb":
        FINAL_ZIP_SIZE_MB,

    "final_zip_sha256":
        FINAL_ZIP_SHA256,

    "integrity":
        "PASS",
}


(
    LOGS_DIR
    / "cell10_final_export.json"
).write_text(
    json.dumps(
        CELL10_RECORD,
        indent=2,
    ),
    encoding="utf-8",
)






CELL10_CHECKPOINT = (
    save_journal_checkpoint(
        10,
        "FINAL_COMPLETE",
    )
)






print()
print("=" * 96)
print("FINAL JOURNAL EXTENSION INTEGRITY REPORT")
print("=" * 96)

print(
    "Experiment A       : 120 / 120 PASS"
)

print(
    "Experiment B       : 30 / 30 PASS"
)

print(
    "Experiment C       : 60 / 60 PASS"
)

print(
    "Experiment D       : 20 / 20 PASS"
)

print(
    "Experiment E       : "
    "22,800 paired paths × 3 methods PASS"
)

print(
    "New GPU runs B-D   : 110"
)

print(
    "Statistics         : PASS"
)

print(
    "PDF figures        : 7"
)

print(
    "600-DPI PNG figures: 7"
)

print(
    "CSV tables         : 8"
)

print(
    "LaTeX tables       : 8"
)

print(
    "File hash manifest : PASS"
)

print(
    "Final ZIP CRC      : PASS"
)

print(
    "Parent hash        :",
    EXPECTED_PARENT_HASH
)

print(
    "Journal hash       :",
    EXPECTED_JOURNAL_HASH
)

print()
print(
    "FINAL ZIP:"
)

print(
    FINAL_ZIP
)

print(
    "ZIP size:",
    f"{FINAL_ZIP_SIZE_MB:.2f} MB"
)

print(
    "ZIP SHA256:",
    FINAL_ZIP_SHA256
)

print()
print(
    "Checksum file:"
)

print(
    FINAL_ZIP_CHECKSUM_PATH
)

print()
print(
    "FINAL INTEGRITY STATUS: PASS"
)

print(
    "QTRUST-PPO JOURNAL EXTENSION: COMPLETE"
)

print("=" * 96)